# Healthcare Challenge 2 - Baseline Submission

This notebook provides a simple baseline for **Healthcare Challenge 2: ED Cost Prediction**.

**Goal**: Predict `ed_cost_next3y_usd` for each patient
**Metric**: Mean Absolute Error (MAE) - Lower is better

## Instructions:
1. **Replace API credentials** in the first cell with your team's API key and name
2. **Run all cells** to generate and submit baseline predictions
3. **Check the output** for your submission score

This baseline uses only tabular ED cost data with a simple Random Forest regressor.


In [ ]:
# 1. Initialize Client and Load Data

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from agentds import BenchmarkClient

# 🔑 REPLACE WITH YOUR CREDENTIALS
client = BenchmarkClient(
    api_key="your_api_key",        # Get from your team dashboard
    team_name="your_team_name"     # Your exact team name
)

# Iteration 1 
- 483.0910

In [14]:
# ============================================================
# AgentDS Challenge 2 — FINAL GPU-First Pipeline
#   - Windows + CUDA: XGBoost(GPU) + CatBoost(GPU)
#   - PyMuPDF PDF parsing cached to disk (no OCR)
#   - 5-fold stratified CV (NO repeats)
#   - Logging with timestamps + stage/fold timings + GPU verification
#   - Outputs: submission_ICHI_V1.csv with exact test order
# ============================================================

import os
import re
import json
import time
import math
import random
import logging
import platform
import subprocess
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, List, Tuple, Dict, Any

import numpy as np
import pandas as pd

# --- PDF ---
import fitz  # PyMuPDF

# --- ML ---
import sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import OneHotEncoder

from scipy import sparse

import xgboost as xgb
from catboost import CatBoostRegressor, Pool

# Optional GPU visibility info (you said torch CUDA is available)
import torch


# -----------------------------
# CONFIG (EDIT PATHS HERE)
# -----------------------------
DATA_DIR = Path(r"D:\AgentDs\agent_ds_healthcare")
TRAIN_CSV = DATA_DIR / "ed_cost_train.csv"
TEST_CSV  = DATA_DIR / "ed_cost_test.csv"
PATIENTS_CSV = DATA_DIR / "patients.csv"  # optional but recommended (exists in dataset spec)

PDF_DIR = DATA_DIR / "receipts_pdf"
CACHE_DIR = DATA_DIR / "cache"
MODEL_DIR = DATA_DIR / "models_gpu_final"
SUBMISSION_PATH = DATA_DIR / "submission_ICHI_V1.csv"

# Runtime control
N_FOLDS = 5
SEED = 2026
N_JOBS_PDF = 6  # PDF extraction parallelism (threads)
FORCE_PDF_REEXTRACT = False

# Text features (fold-safe)
TFIDF_MAX_FEATURES = 6000
SVD_COMPONENTS = 64

# Model caps + early stopping (keep sane; GPU should be fast)
XGB_N_ESTIMATORS = 8000
XGB_EARLY_STOPPING = 400

CAT_ITERATIONS = 12000
CAT_OD_WAIT = 200

# Sanity
ASSERT_GPU_REQUIRED = True   # if True, we error if torch can't see GPU


# -----------------------------
# Logging + timing utilities
# -----------------------------
def setup_logger(log_path: Optional[Path] = None, level=logging.INFO) -> logging.Logger:
    logger = logging.getLogger("ed_cost")
    logger.setLevel(level)
    logger.handlers.clear()
    logger.propagate = False

    fmt = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    logger.addHandler(sh)

    if log_path is not None:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        fh = logging.FileHandler(log_path, encoding="utf-8")
        fh.setFormatter(fmt)
        logger.addHandler(fh)

    return logger


class StageTimer:
    def __init__(self, logger: logging.Logger, name: str):
        self.logger = logger
        self.name = name
        self.t0 = None

    def __enter__(self):
        self.t0 = time.perf_counter()
        self.logger.info(f"[{self.name}] START")
        return self

    def __exit__(self, exc_type, exc, tb):
        dt = time.perf_counter() - self.t0
        if exc is None:
            self.logger.info(f"[{self.name}] END ({dt:.2f}s)")
        else:
            self.logger.exception(f"[{self.name}] FAILED after {dt:.2f}s: {exc}")


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


def try_run_cmd(cmd: List[str]) -> str:
    try:
        out = subprocess.check_output(cmd, stderr=subprocess.STDOUT, text=True)
        return out.strip()
    except Exception as e:
        return f"<command failed: {e}>"


def pymupdf_version() -> str:
    try:
        doc = fitz.__doc__ or ""
        m = re.search(r"PyMuPDF\s+(\d+\.\d+(?:\.\d+)?)", doc)
        return m.group(1) if m else "unknown"
    except Exception:
        return "unknown"


# -----------------------------
# PDF parsing (fast, no OCR)
# -----------------------------
_MONEY_RE = re.compile(r"^\$?[\d,]+(?:\.\d{2})?$")
_CODE_RE  = re.compile(r"^[A-Z0-9]{5}$")
_LINE_PAT = re.compile(r"^([A-Z0-9]{5})\s+(.+?)\s+(\d+)\s+([\d,]+\.\d{2})\s+([\d,]+\.\d{2})$")

def _parse_money(s: str) -> float:
    s = str(s).strip().replace("$", "").replace(",", "")
    try:
        return float(s)
    except Exception:
        return float("nan")


@dataclass(frozen=True)
class PdfParsed:
    patient_id_pdf: Optional[int]
    zip3_pdf: Optional[str]
    insurance_pdf: Optional[str]
    total_pdf: float
    items: List[Tuple[str, str, int, float, float]]  # code, desc, qty, unit, line_total
    raw_text: str


def parse_receipt_pdf(pdf_path: Path) -> Optional[PdfParsed]:
    if not pdf_path.exists():
        return None

    try:
        doc = fitz.open(pdf_path)
        raw_text = "".join([p.get_text("text", sort=True) for p in doc])
        doc.close()
    except Exception:
        return None

    lines = [ln.strip() for ln in raw_text.splitlines() if ln.strip()]

    pid = None
    zip3 = None
    ins = None

    for ln in lines[:12]:
        m = re.search(r"Patient ID:\s*(\d+)", ln)
        if m:
            pid = int(m.group(1))
        m = re.search(r"ZIP3:\s*(\d{3})", ln)
        if m:
            zip3 = m.group(1)
        m = re.search(r"Insurance:\s*([A-Za-z_]+)", ln)
        if m:
            ins = m.group(1)

    # TOTAL on same line or next line
    total = float("nan")
    for i, ln in enumerate(lines):
        if ln.startswith("TOTAL"):
            parts = ln.split()
            if len(parts) >= 2 and _MONEY_RE.match(parts[1]):
                total = _parse_money(parts[1])
            elif i + 1 < len(lines) and _MONEY_RE.match(lines[i + 1]):
                total = _parse_money(lines[i + 1])
            break

    items: List[Tuple[str, str, int, float, float]] = []

    # Layout A: row-per-line
    for ln in lines:
        m = _LINE_PAT.match(ln)
        if m:
            code, desc, qty, unit, lt = m.groups()
            items.append((code, desc, int(qty), _parse_money(unit), _parse_money(lt)))

    # Layout B: columnar / cell-per-line
    if not items:
        start = 0
        for i, ln in enumerate(lines):
            if ln.lower().startswith("line total"):
                start = i + 1
                break

        i = start
        while i < len(lines):
            ln = lines[i]
            if ln.startswith("TOTAL"):
                break

            if _CODE_RE.match(ln):
                code = ln
                j = i + 1

                desc_parts = []
                while j < len(lines) and not re.fullmatch(r"\d+", lines[j]):
                    if lines[j].startswith("TOTAL"):
                        break
                    desc_parts.append(lines[j])
                    j += 1

                if j >= len(lines) or lines[j].startswith("TOTAL"):
                    break

                qty = int(lines[j]); j += 1
                if j + 1 >= len(lines):
                    break

                unit = _parse_money(lines[j])
                lt = _parse_money(lines[j + 1])
                desc = " ".join(desc_parts).strip()

                items.append((code, desc, qty, unit, lt))
                i = j + 2
            else:
                i += 1

    return PdfParsed(
        patient_id_pdf=pid,
        zip3_pdf=zip3,
        insurance_pdf=ins,
        total_pdf=total,
        items=items,
        raw_text=raw_text,
    )


def compute_pdf_features(parsed: Optional[PdfParsed]) -> Dict[str, Any]:
    if parsed is None:
        return {
            "zip3_pdf": "UNK",
            "insurance_pdf": "UNK",
            "pdf_total": float("nan"),
            "pdf_n_items": 0,
            "pdf_n_unique_codes": 0,
            "pdf_sum_qty": 0.0,
            "pdf_mean_line_total": 0.0,
            "pdf_max_line_total": 0.0,
            "pdf_std_line_total": 0.0,
            "pdf_share_top1": 0.0,
            "pdf_cost_eval_mgmt": 0.0,
            "pdf_cost_lab": 0.0,
            "pdf_cost_radiology": 0.0,
            "pdf_cost_surgery": 0.0,
            "pdf_cost_other_numeric": 0.0,
            "pdf_cost_hcpcs_alpha": 0.0,
            "pdf_em_level_max": 0,
            "pdf_em_count": 0,
            "pdf_critical_count": 0,
            "pdf_text": "",
        }

    items = parsed.items
    total = parsed.total_pdf

    codes = [it[0] for it in items]
    descs = [it[1] for it in items]

    qtys = np.array([it[2] for it in items], dtype=np.float32) if items else np.array([], dtype=np.float32)
    lts  = np.array([it[4] for it in items], dtype=np.float32) if items else np.array([], dtype=np.float32)

    n = len(items)
    n_unique = len(set(codes)) if n else 0

    mean_lt = float(lts.mean()) if n else 0.0
    max_lt  = float(lts.max()) if n else 0.0
    std_lt  = float(lts.std()) if n else 0.0
    share_top1 = float(max_lt / total) if n and (not math.isnan(total)) and total > 0 else 0.0

    # Coarse “claims mix” buckets + ED severity proxies
    eval_mgmt = lab = radiology = surgery = other_numeric = hcpcs_alpha = 0.0
    em_level_max = 0
    em_count = 0
    critical_count = 0

    for code, _desc, _qty, _unit, lt in items:
        if code and code[0].isdigit():
            ci = int(code)
            if 90000 <= ci <= 99999:
                eval_mgmt += lt
            elif 80000 <= ci <= 89999:
                lab += lt
            elif 70000 <= ci <= 79999:
                radiology += lt
            elif 10000 <= ci <= 69999:
                surgery += lt
            else:
                other_numeric += lt

            if 99281 <= ci <= 99285:
                em_count += 1
                em_level_max = max(em_level_max, ci - 99280)

            if ci in (99291, 99292):
                critical_count += 1
        else:
            hcpcs_alpha += lt

    pdf_text = " ".join([f"{c} {d}" for c, d in zip(codes, descs)])

    return {
        "zip3_pdf": parsed.zip3_pdf or "UNK",
        "insurance_pdf": parsed.insurance_pdf or "UNK",
        "pdf_total": total,
        "pdf_n_items": int(n),
        "pdf_n_unique_codes": int(n_unique),
        "pdf_sum_qty": float(qtys.sum()) if n else 0.0,
        "pdf_mean_line_total": mean_lt,
        "pdf_max_line_total": max_lt,
        "pdf_std_line_total": std_lt,
        "pdf_share_top1": share_top1,
        "pdf_cost_eval_mgmt": float(eval_mgmt),
        "pdf_cost_lab": float(lab),
        "pdf_cost_radiology": float(radiology),
        "pdf_cost_surgery": float(surgery),
        "pdf_cost_other_numeric": float(other_numeric),
        "pdf_cost_hcpcs_alpha": float(hcpcs_alpha),
        "pdf_em_level_max": int(em_level_max),
        "pdf_em_count": int(em_count),
        "pdf_critical_count": int(critical_count),
        "pdf_text": pdf_text,
    }

class XGBProgressCallback(xgb.callback.TrainingCallback):
    """Log eval metric every `period` rounds."""
    def __init__(self, logger, period=200):
        self.logger = logger
        self.period = period

    def after_iteration(self, model, epoch, evals_log):
        if (epoch + 1) % self.period == 0:
            msg_parts = []
            for data, metrics in evals_log.items():
                for metric, values in metrics.items():
                    msg_parts.append(f"{data}-{metric}={values[-1]:.4f}")
            self.logger.info(f"  [XGB iter {epoch+1}] {' | '.join(msg_parts)}")
        return False  # False = don't stop


def fit_xgb_with_fallback(params, fold_seed, callbacks, X_tr, y_tr, X_va, y_va, logger):
    """
    Default path: early_stopping_rounds + callbacks in constructor (your fixed pattern).
    Fallback: if TypeError, try moving callbacks / early stopping into fit().
    """
    # 1) default: fixed pattern
    try:
        model = xgb.XGBRegressor(
            **{**params, "random_state": fold_seed},
            callbacks=callbacks,
        )
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        return model
    except TypeError as e:
        logger.warning(f"[XGB fallback] constructor pattern TypeError: {e}")

    # 2) fallback A: no callbacks in constructor, pass to fit
    try:
        model = xgb.XGBRegressor(**{**params, "random_state": fold_seed})
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False, callbacks=callbacks)
        return model
    except TypeError as e:
        logger.warning(f"[XGB fallback] fit(callbacks=...) TypeError: {e}")

    # 3) fallback B: if early_stopping_rounds not accepted in constructor, move to fit
    esr = params.get("early_stopping_rounds", None)
    params2 = dict(params)
    params2.pop("early_stopping_rounds", None)
    model = xgb.XGBRegressor(**{**params2, "random_state": fold_seed})
    try:
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False,
                  early_stopping_rounds=esr, callbacks=callbacks)
        return model
    except TypeError as e:
        logger.error(f"[XGB fallback] unable to apply early stopping: {e}")
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        return model


# -----------------------------
# PDF feature cache
# -----------------------------
def parquet_available() -> bool:
    try:
        import pyarrow  # noqa
        return True
    except Exception:
        try:
            import fastparquet  # noqa
            return True
        except Exception:
            return False


def load_or_extract_pdf_features(logger: logging.Logger) -> pd.DataFrame:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    use_parquet = parquet_available()
    PDF_CACHE_VERSION = "v2"  # bump when schema changes
    cache_path = CACHE_DIR / (f"pdf_features_all_{PDF_CACHE_VERSION}.parquet" if use_parquet else "pdf_features_all.pkl")

    # Read train/test IDs
    train = pd.read_csv(TRAIN_CSV, usecols=["patient_id"])
    test = pd.read_csv(TEST_CSV,  usecols=["patient_id"])
    all_ids = pd.concat([train["patient_id"], test["patient_id"]]).astype(int).tolist()
    expected_n = len(all_ids)

    # Count PDFs found/missing
    found = 0
    missing_list = []
    for pid in all_ids:
        if (PDF_DIR / f"receipt_{pid}.pdf").exists():
            found += 1
        else:
            missing_list.append(pid)
    logger.info(f"[PDF] Expected PDFs: {expected_n} | Found: {found} | Missing: {expected_n - found}")
    if missing_list:
        logger.warning(f"[PDF] Missing example patient_ids: {missing_list[:10]}")

    if cache_path.exists() and not FORCE_PDF_REEXTRACT:
        with StageTimer(logger, "pdf_cache_load"):
            df = pd.read_parquet(cache_path) if cache_path.suffix == ".parquet" else pd.read_pickle(cache_path)
        if "patient_id" in df.columns and len(df) == expected_n:
            logger.info(f"[PDF] Using cached features: {cache_path}")
            return df
        else:
            logger.warning("[PDF] Cache shape mismatch; re-extracting...")

    # Extract
    logger.info(f"[PDF] Extracting with PyMuPDF (version={pymupdf_version()}) | cache={cache_path.name}")
    from joblib import Parallel, delayed

    def extract_one(pid: int) -> Dict[str, Any]:
        parsed = parse_receipt_pdf(PDF_DIR / f"receipt_{pid}.pdf")
        feats = compute_pdf_features(parsed)
        feats["patient_id"] = int(pid)
        return feats

    with StageTimer(logger, "pdf_extract_total"):
        rows = Parallel(n_jobs=N_JOBS_PDF, prefer="threads")(
            delayed(extract_one)(pid) for pid in all_ids
        )

    df = pd.DataFrame(rows)
    # Preserve test order safety: do NOT sort here; merges will preserve left order.
    # But cache should be stable; we store sorted by patient_id for reproducibility.
    df = df.sort_values("patient_id").reset_index(drop=True)

    with StageTimer(logger, "pdf_cache_write"):
        if cache_path.suffix == ".parquet":
            df.to_parquet(cache_path, index=False)
        else:
            df.to_pickle(cache_path)
    logger.info(f"[PDF] Cached features written: {cache_path}")

    return df


def add_top_code_features(train_df, test_df, top_k=200, min_total_count=8, logger=None):
    """
    Adds numeric features for top codes:
      - code_cnt_<CODE>
      - code_cost_share_<CODE>  (sum line_total for code / (prior_cost + 1))
    Uses train to select top_k codes by total count.
    """
    def parse_dict_col(s):
        if not isinstance(s, str) or not s:
            return {}
        try:
            return json.loads(s)
        except Exception:
            return {}

    # 1) select top codes from train
    freq = {}
    for s in train_df["pdf_code_counts_json"].fillna("{}").values:
        d = parse_dict_col(s)
        for k, v in d.items():
            freq[k] = freq.get(k, 0) + int(v)
    # filter
    freq_items = [(k, v) for k, v in freq.items() if v >= min_total_count]
    freq_items.sort(key=lambda x: x[1], reverse=True)
    top_codes = [k for k, _ in freq_items[:top_k]]
    code_to_idx = {c:i for i,c in enumerate(top_codes)}
    if logger:
        logger.info(f"[TopCodes] selected={len(top_codes)} (top_k={top_k}, min_total_count={min_total_count})")
        logger.info(f"[TopCodes] examples={top_codes[:20]}")

    def build_matrix(df):
        n = len(df)
        K = len(top_codes)
        cnt = np.zeros((n, K), dtype=np.float32)
        share = np.zeros((n, K), dtype=np.float32)
        denom = df["prior_ed_cost_5y_usd"].astype(np.float32).values + 1.0

        for i, s in enumerate(df["pdf_code_counts_json"].fillna("{}").values):
            d = parse_dict_col(s)
            for code, v in d.items():
                j = code_to_idx.get(code)
                if j is not None:
                    cnt[i, j] = float(v)

        for i, s in enumerate(df["pdf_code_costs_json"].fillna("{}").values):
            d = parse_dict_col(s)
            for code, v in d.items():
                j = code_to_idx.get(code)
                if j is not None:
                    share[i, j] = float(v) / denom[i]

        cols_cnt = [f"code_cnt_{c}" for c in top_codes]
        cols_shr = [f"code_cost_share_{c}" for c in top_codes]
        return pd.DataFrame(np.hstack([cnt, share]), columns=cols_cnt + cols_shr)

    tr_mat = build_matrix(train_df)
    te_mat = build_matrix(test_df)
    train_df = pd.concat([train_df.reset_index(drop=True), tr_mat], axis=1)
    test_df  = pd.concat([test_df.reset_index(drop=True),  te_mat], axis=1)
    
    return train_df, test_df, top_codes

# -----------------------------
# Feature engineering
# -----------------------------
def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # categorical clean
    for c in ["primary_chronic", "sex", "insurance", "zip3", "zip3_pdf", "insurance_pdf"]:
        if c in out.columns:
            out[c] = out[c].fillna("UNK").astype(str)

    # numeric fill
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].fillna(0.0)

    # utilization transforms
    out["prior_cost_log1p"] = np.log1p(out["prior_ed_cost_5y_usd"].astype(float))
    out["prior_visits_log1p"] = np.log1p(out["prior_ed_visits_5y"].astype(float))
    out["prior_cost_per_visit"] = out["prior_ed_cost_5y_usd"] / (out["prior_ed_visits_5y"] + 1.0)
    out["prior_cost_per_visit_log1p"] = np.log1p(out["prior_cost_per_visit"].astype(float))
    out["prior_cost_x_visits"] = out["prior_ed_cost_5y_usd"] * out["prior_ed_visits_5y"]

    # composition shares (normalize by prior cost, NOT pdf_total)
    denom = out["prior_ed_cost_5y_usd"].astype(float) + 1.0
    for c in [
        "pdf_cost_eval_mgmt",
        "pdf_cost_lab",
        "pdf_cost_radiology",
        "pdf_cost_surgery",
        "pdf_cost_other_numeric",
        "pdf_cost_hcpcs_alpha",
    ]:
        if c in out.columns:
            out[f"{c}_share"] = out[c].astype(float) / denom

    out["pdf_items_per_visit"] = out["pdf_n_items"].astype(float) / (out["prior_ed_visits_5y"] + 1.0)

    # final numeric fill
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].fillna(0.0)

    return out


# -----------------------------
# Fold-safe text: TF-IDF -> SVD
# -----------------------------
def fit_text_models(train_text: np.ndarray, seed: int) -> Tuple[Optional[TfidfVectorizer], Optional[TruncatedSVD], int]:
    # min_df fallback if corpus is tiny
    for min_df in (2, 1):
        vec = TfidfVectorizer(
            lowercase=True,
            token_pattern=r"(?u)\b[\w']+\b",
            ngram_range=(1, 2),
            max_features=TFIDF_MAX_FEATURES,
            min_df=min_df,
            sublinear_tf=True,
        )
        X = vec.fit_transform(train_text)
        if X.shape[1] < 2:
            continue

        n_comp = min(SVD_COMPONENTS, X.shape[1] - 1)
        svd = TruncatedSVD(n_components=n_comp, random_state=seed)
        svd.fit(X)
        return vec, svd, n_comp

    return None, None, 0


def transform_text(vec: Optional[TfidfVectorizer], svd: Optional[TruncatedSVD], text: np.ndarray) -> np.ndarray:
    if vec is None or svd is None:
        return np.zeros((len(text), 0), dtype=np.float32)
    X = vec.transform(text)
    if X.shape[1] == 0:
        return np.zeros((len(text), 0), dtype=np.float32)
    return svd.transform(X).astype(np.float32)


# -----------------------------
# Modeling matrices
# -----------------------------
def build_base_feature_lists(df: pd.DataFrame, target_col: str, text_col: str, drop_cols: List[str]) -> Tuple[List[str], List[str]]:
    # cat features used by CatBoost
    cat_cols = ["primary_chronic"]
    for c in ["sex", "insurance", "zip3", "zip3_pdf", "insurance_pdf"]:
        if c in df.columns:
            cat_cols.append(c)

    # remove duplicates while preserving order
    seen = set()
    cat_cols = [c for c in cat_cols if not (c in seen or seen.add(c))]

    excluded = set(drop_cols + [target_col, text_col] + cat_cols)
    num_cols = [c for c in df.columns if c not in excluded and pd.api.types.is_numeric_dtype(df[c])]
    num_cols = sorted(num_cols)
    return num_cols, cat_cols


def make_ohe():
    # sklearn compat across versions
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True, dtype=np.float32)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True, dtype=np.float32)


# -----------------------------
# Training + submission
# -----------------------------
def run_train_and_submit(logger: logging.Logger) -> None:
    seed_everything(SEED)

    # ---- Env / GPU logs ----
    with StageTimer(logger, "env_report"):
        logger.info(f"OS: {platform.platform()}")
        logger.info(f"Python: {platform.python_version()}")
        logger.info(f"numpy={np.__version__} pandas={pd.__version__} sklearn={sklearn.__version__}")
        logger.info(f"xgboost={xgb.__version__} catboost={CatBoostRegressor.__module__.split('.')[0]}")
        logger.info(f"PyMuPDF={pymupdf_version()}")
        logger.info(f"torch={torch.__version__} cuda_available={torch.cuda.is_available()}")

        if ASSERT_GPU_REQUIRED and (not torch.cuda.is_available()):
            raise RuntimeError("CUDA not available in torch — GPU required by your constraints.")

        if torch.cuda.is_available():
            logger.info(f"torch.cuda.device_count={torch.cuda.device_count()}")
            logger.info(f"GPU[0]={torch.cuda.get_device_name(0)}")
            logger.info(f"GPU[0] capability={torch.cuda.get_device_capability(0)}")
            logger.info("nvidia-smi -L:\n" + try_run_cmd(["nvidia-smi", "-L"]))

        # XGBoost build info if available
        if hasattr(xgb, "build_info"):
            try:
                logger.info(f"xgboost.build_info(): {xgb.build_info()}")
            except Exception:
                pass

    # ---- Load data ----
    with StageTimer(logger, "load_data"):
        train = pd.read_csv(TRAIN_CSV)
        test  = pd.read_csv(TEST_CSV)
        logger.info(f"train shape={train.shape} | test shape={test.shape}")
        logger.info(f"train cols={list(train.columns)}")

    # ---- Optional patients.csv join ----
    if PATIENTS_CSV.exists():
        with StageTimer(logger, "join_patients_csv"):
            patients = pd.read_csv(PATIENTS_CSV, dtype={"zip3": str})
            keep = [c for c in ["patient_id", "age", "sex", "insurance", "zip3"] if c in patients.columns]
            patients = patients[keep]
            train = train.merge(patients, on="patient_id", how="left", sort=False)
            test  = test.merge(patients, on="patient_id", how="left", sort=False)
            logger.info(f"Joined patients.csv | keep_cols={keep} | train shape now={train.shape}")
    else:
        logger.warning("patients.csv not found; continuing without it.")

    # ---- PDF feature cache ----
    pdf_feats = load_or_extract_pdf_features(logger)

    with StageTimer(logger, "merge_pdf_features"):
        train = train.merge(pdf_feats, on="patient_id", how="left", sort=False)
        test  = test.merge(pdf_feats, on="patient_id", how="left", sort=False)

    # Sanity: PDF TOTAL matches prior_ed_cost_5y_usd (dataset spec) — use as CHECK only
    # We'll log abs diff stats where pdf_total exists.
    if "pdf_total" in train.columns:
        ok = train["pdf_total"].notna() & (train["pdf_total"] > 0)
        if ok.any():
            abs_diff = (train.loc[ok, "prior_ed_cost_5y_usd"] - train.loc[ok, "pdf_total"]).abs()
            logger.info("[PDF sanity] |prior_ed_cost_5y_usd - pdf_total| describe:\n" + str(abs_diff.describe()))

    # ---- Feature engineering ----
    with StageTimer(logger, "feature_engineering"):
        # train = add_derived_features(train)
        # test  = add_derived_features(test)
        train, test, top_codes = add_top_code_features(train, test, top_k=200, min_total_count=8, logger=logger)
        (MODEL_DIR / "top_codes.json").write_text(json.dumps(top_codes, indent=2))
        logger.info(f"[TopCodes] train shape now={train.shape} | test shape now={test.shape}")

    target_col = "ed_cost_next3y_usd"
    text_col = "pdf_text"

    # IMPORTANT: drop pdf_total to avoid redundancy (TOTAL matches prior cost by dataset design)
    drop_cols = ["patient_id", "pdf_total"]  # keep pdf_text for fold-safe vectorization
    num_cols, cat_cols = build_base_feature_lists(train, target_col, text_col, drop_cols=drop_cols)

    logger.info(f"Numeric base cols: {len(num_cols)} | Cat cols (CatBoost): {len(cat_cols)} -> {cat_cols}")
    logger.info(f"Text col: {text_col}")

    # ---- CV setup (exactly 5 folds, no repeats) ----
    assert N_FOLDS == 5, "Hard requirement: CV must be max 5 folds."
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    logger.info(f"Total folds: {N_FOLDS} (single stratified pass; no repeats)")

    y = train[target_col].astype(np.float32).values
    oof_xgb = np.zeros(len(train), dtype=np.float32)
    oof_xgb_slog = np.zeros(len(train), dtype=np.float32)
    oof_cat = np.zeros(len(train), dtype=np.float32)
    test_pred_xgb = np.zeros(len(test), dtype=np.float32)
    test_pred_xgb_slog = np.zeros(len(test), dtype=np.float32)
    test_pred_cat = np.zeros(len(test), dtype=np.float32)

    # Create output dirs
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    (MODEL_DIR / "fold_artifacts").mkdir(parents=True, exist_ok=True)

    # Log model params (explicit GPU)
    xgb_params = dict(
        n_estimators=XGB_N_ESTIMATORS,
        learning_rate=0.03,
        max_depth=7,
        min_child_weight=20.0,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=2.0,
        objective="reg:absoluteerror",   # official objective option
        eval_metric="mae",              # official metric option
        tree_method="hist",
        device="cuda:0",                # explicit GPU enable (per docs)
        random_state=SEED,
        n_jobs=0,  # GPU-focused; keep CPU threads minimal
        early_stopping_rounds=XGB_EARLY_STOPPING,  # <-- add here
    )
    logger.info(f"[XGB] params={xgb_params}")

    cat_params = dict(
        loss_function="MAE",
        eval_metric="MAE",
        iterations=CAT_ITERATIONS,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=3.0,
        random_seed=SEED,
        task_type="GPU",  # explicit GPU enable (per docs)
        devices="0",
        od_type="Iter",
        od_wait=CAT_OD_WAIT,
        allow_writing_files=False,
        verbose=1000, # log every 500 iterations (overrides verbose=False for logging purposes only
    )
    logger.info(f"[CAT] params={cat_params}")

    # ---- Fold loop ----
    for fold, (tr_idx, va_idx) in enumerate(skf.split(train, train["primary_chronic"].astype(str))):
        with StageTimer(logger, f"fold_{fold}"):
            tr = train.iloc[tr_idx].reset_index(drop=True)
            va = train.iloc[va_idx].reset_index(drop=True)

            y_tr = tr[target_col].astype(np.float32).values
            y_va = va[target_col].astype(np.float32).values

            # Fold-safe text models
            with StageTimer(logger, f"fold_{fold}/text_fit"):
                vec, svd, svd_dim = fit_text_models(tr[text_col].fillna("").values, seed=SEED + fold)
                logger.info(f"[fold {fold}] TFIDF fitted. SVD dim={svd_dim}")

            with StageTimer(logger, f"fold_{fold}/text_transform"):
                tr_svd = transform_text(vec, svd, tr[text_col].fillna("").values)
                va_svd = transform_text(vec, svd, va[text_col].fillna("").values)
                te_svd = transform_text(vec, svd, test[text_col].fillna("").values)

            # Build CatBoost matrices (dense + cat)
            with StageTimer(logger, f"fold_{fold}/build_cat_matrices"):
                X_tr_cat = tr[num_cols].copy()
                X_va_cat = va[num_cols].copy()
                X_te_cat = test[num_cols].copy()

                for j in range(tr_svd.shape[1]):
                    col = f"svd_{j}"
                    X_tr_cat[col] = tr_svd[:, j]
                    X_va_cat[col] = va_svd[:, j]
                    X_te_cat[col] = te_svd[:, j]

                # Add categorical columns
                for c in cat_cols:
                    X_tr_cat[c] = tr[c].fillna("UNK").astype(str).values
                    X_va_cat[c] = va[c].fillna("UNK").astype(str).values
                    X_te_cat[c] = test[c].fillna("UNK").astype(str).values

                cat_feat_idx = [X_tr_cat.columns.get_loc(c) for c in cat_cols]
                logger.info(f"[fold {fold}] CatBoost X shapes: tr={X_tr_cat.shape} va={X_va_cat.shape} te={X_te_cat.shape}")

            # Build XGBoost matrices (numeric+svd + one-hot cats) as sparse
            with StageTimer(logger, f"fold_{fold}/build_xgb_matrices"):
                ohe = make_ohe()
                ohe.fit(tr[cat_cols].fillna("UNK").astype(str))

                tr_cat_oh = ohe.transform(tr[cat_cols].fillna("UNK").astype(str))
                va_cat_oh = ohe.transform(va[cat_cols].fillna("UNK").astype(str))
                te_cat_oh = ohe.transform(test[cat_cols].fillna("UNK").astype(str))

                tr_num = tr[num_cols].astype(np.float32).values
                va_num = va[num_cols].astype(np.float32).values
                te_num = test[num_cols].astype(np.float32).values

                # combine numeric + svd to sparse
                tr_dense = np.hstack([tr_num, tr_svd]).astype(np.float32)
                va_dense = np.hstack([va_num, va_svd]).astype(np.float32)
                te_dense = np.hstack([te_num, te_svd]).astype(np.float32)

                X_tr_xgb = sparse.hstack([sparse.csr_matrix(tr_dense), tr_cat_oh], format="csr")
                X_va_xgb = sparse.hstack([sparse.csr_matrix(va_dense), va_cat_oh], format="csr")
                X_te_xgb = sparse.hstack([sparse.csr_matrix(te_dense), te_cat_oh], format="csr")

                logger.info(f"[fold {fold}] XGB X shapes: tr={X_tr_xgb.shape} va={X_va_xgb.shape} te={X_te_xgb.shape}")

            # ---- Train XGBoost GPU ----
            with StageTimer(logger, f"fold_{fold}/train_xgb_gpu"):
                xgb_model = xgb.XGBRegressor(
                    **{**xgb_params, "random_state": SEED + fold},
                    callbacks=[XGBProgressCallback(logger, period=200)],
                )
                xgb_model.fit(
                    X_tr_xgb, y_tr,
                    eval_set=[(X_va_xgb, y_va)],
                    verbose=False,
                    # callbacks=[XGBProgressCallback(logger, period=200)],
                    # early_stopping_rounds=XGB_EARLY_STOPPING,
                )
                best_iter = getattr(xgb_model, "best_iteration", None)
                logger.info(f"[fold {fold}] XGB best_iteration={best_iter}")

                # GPU verification: check booster config includes 'cuda'
                booster_cfg = xgb_model.get_booster().save_config()
                has_cuda = ("cuda" in booster_cfg.lower())
                logger.info(f"[fold {fold}] XGB booster config contains 'cuda': {has_cuda}")
                if fold == 0:
                    # write a snippet for inspection
                    (MODEL_DIR / "xgb_booster_config_fold0.json").write_text(booster_cfg)

                # Save model
                xgb_model.save_model(str(MODEL_DIR / "fold_artifacts" / f"xgb_fold{fold}.json"))

            pred_va_xgb = xgb_model.predict(X_va_xgb)
            oof_xgb[va_idx] = pred_va_xgb.astype(np.float32)
            test_pred_xgb += xgb_model.predict(X_te_xgb).astype(np.float32) / N_FOLDS
            fold_mae_xgb = mean_absolute_error(y_va, pred_va_xgb)
            logger.info(f"[fold {fold}] XGB fold MAE={fold_mae_xgb:.6f}")

            # ---- Train CatBoost GPU ----
            with StageTimer(logger, f"fold_{fold}/train_cat_gpu"):
                cat_model = CatBoostRegressor(**{**cat_params, "random_seed": SEED + fold})
                tr_pool = Pool(X_tr_cat, y_tr, cat_features=cat_feat_idx)
                va_pool = Pool(X_va_cat, y_va, cat_features=cat_feat_idx)
                te_pool = Pool(X_te_cat, cat_features=cat_feat_idx)

                cat_model.fit(tr_pool, eval_set=va_pool, use_best_model=True)
                best_iter_cat = cat_model.get_best_iteration()
                logger.info(f"[fold {fold}] CAT best_iteration={best_iter_cat}")

                # GPU verification: confirm params reflect GPU
                allp = cat_model.get_all_params()
                logger.info(f"[fold {fold}] CAT task_type={allp.get('task_type', '<missing>')} devices={allp.get('devices', '<missing>')}")

                cat_model.save_model(str(MODEL_DIR / "fold_artifacts" / f"cat_fold{fold}.cbm"))

            pred_va_cat = cat_model.predict(va_pool)
            oof_cat[va_idx] = pred_va_cat.astype(np.float32)
            test_pred_cat += cat_model.predict(te_pool).astype(np.float32) / N_FOLDS
            fold_mae_cat = mean_absolute_error(y_va, pred_va_cat)
            logger.info(f"[fold {fold}] CAT fold MAE={fold_mae_cat:.6f}")

            # Save fold text models + encoder for reproducibility/audits (optional)
            import joblib
            joblib.dump(vec, MODEL_DIR / "fold_artifacts" / f"tfidf_fold{fold}.joblib")
            joblib.dump(svd, MODEL_DIR / "fold_artifacts" / f"svd_fold{fold}.joblib")
            joblib.dump(ohe, MODEL_DIR / "fold_artifacts" / f"ohe_fold{fold}.joblib")

    # ---- OOF summary + blend ----
    with StageTimer(logger, "blend_oof_search"):
        oof_mae_xgb = mean_absolute_error(y, oof_xgb)
        oof_mae_cat = mean_absolute_error(y, oof_cat)

        best = {"w_xgb": None, "bias": None, "mae": float("inf")}
        for w in np.linspace(0.0, 1.0, 101):
            pred = np.clip(w * oof_xgb + (1 - w) * oof_cat, 0.0, None)
            bias = float(np.median(pred - y))  # MAE-optimal constant shift
            pred2 = np.clip(pred - bias, 0.0, None)
            mae = float(mean_absolute_error(y, pred2))
            if mae < best["mae"]:
                best = {"w_xgb": float(w), "bias": float(bias), "mae": float(mae)}

        logger.info(f"OOF MAE XGB={oof_mae_xgb:.6f} | CAT={oof_mae_cat:.6f} | BLEND={best['mae']:.6f} (w_xgb={best['w_xgb']:.2f}, bias={best['bias']:.4f})")
        (MODEL_DIR / "blend.json").write_text(json.dumps(best, indent=2))

    # ---- Final test prediction + submission ----
    with StageTimer(logger, "submission_write"):
        pred_test = np.clip(best["w_xgb"] * test_pred_xgb + (1 - best["w_xgb"]) * test_pred_cat - best["bias"], 0.0, None)

        # Guarantee exact test order: we never reorder test after loading; merges preserve left order (sort=False).
        sub = pd.DataFrame({
            "patient_id": test["patient_id"].values,
            "ed_cost_next3y_usd": pred_test.astype(float),
        })
        sub.to_csv(SUBMISSION_PATH, index=False)
        logger.info(f"Submission written: {SUBMISSION_PATH} | rows={len(sub)}")
        logger.info("Submission head:\n" + str(sub.head()))

    logger.info("DONE ✅")


# -----------------------------
# RUN
# -----------------------------
LOG_PATH = MODEL_DIR / "train_log.txt"
logger = setup_logger(LOG_PATH, level=logging.INFO)

RUN = True  # <-- set True to execute

if RUN:
    run_train_and_submit(logger)
else:
    logger.info("Set RUN=True to execute the full pipeline.")


2026-02-13 04:45:56 | INFO | ed_cost | [env_report] START
2026-02-13 04:45:56 | INFO | ed_cost | OS: Windows-11-10.0.26100-SP0
2026-02-13 04:45:56 | INFO | ed_cost | Python: 3.12.0
2026-02-13 04:45:56 | INFO | ed_cost | numpy=2.3.4 pandas=2.3.3 sklearn=1.7.2
2026-02-13 04:45:56 | INFO | ed_cost | xgboost=3.0.0 catboost=catboost
2026-02-13 04:45:56 | INFO | ed_cost | PyMuPDF=1.26.5
2026-02-13 04:45:56 | INFO | ed_cost | torch=2.9.0+cu126 cuda_available=True
2026-02-13 04:45:56 | INFO | ed_cost | torch.cuda.device_count=1
2026-02-13 04:45:56 | INFO | ed_cost | GPU[0]=NVIDIA GeForce RTX 4060 Laptop GPU
2026-02-13 04:45:56 | INFO | ed_cost | GPU[0] capability=(8, 9)
2026-02-13 04:45:56 | INFO | ed_cost | nvidia-smi -L:
GPU 0: NVIDIA GeForce RTX 4060 Laptop GPU (UUID: GPU-9515bdbf-1472-4fa2-9f73-28e1d27eedbf)
2026-02-13 04:45:56 | INFO | ed_cost | xgboost.build_info(): {'BUILTIN_PREFETCH_PRESENT': False, 'CUDA_VERSION': [12, 5], 'DEBUG': False, 'MM_PREFETCH_PRESENT': True, 'THRUST_VERSION':

0:	learn: 1438.3384375	test: 1414.0309375	best: 1414.0309375 (0)	total: 43.2ms	remaining: 8m 38s
1000:	learn: 1428.2825000	test: 1404.4190625	best: 1404.4190625 (1000)	total: 41.6s	remaining: 7m 37s
2000:	learn: 1418.1915625	test: 1394.8184375	best: 1394.8184375 (2000)	total: 1m 24s	remaining: 7m 2s
3000:	learn: 1408.1318750	test: 1385.2875000	best: 1385.2875000 (3000)	total: 2m 9s	remaining: 6m 28s
4000:	learn: 1398.1771875	test: 1375.8559375	best: 1375.8559375 (4000)	total: 2m 54s	remaining: 5m 48s
5000:	learn: 1388.2937500	test: 1366.4290625	best: 1366.4290625 (5000)	total: 3m 39s	remaining: 5m 7s
6000:	learn: 1378.4425000	test: 1357.0093750	best: 1357.0093750 (6000)	total: 4m 25s	remaining: 4m 25s
7000:	learn: 1368.6115625	test: 1347.6010938	best: 1347.6010938 (7000)	total: 5m 13s	remaining: 3m 43s
8000:	learn: 1358.8085938	test: 1338.2046875	best: 1338.2046875 (8000)	total: 6m	remaining: 3m
9000:	learn: 1349.1076562	test: 1328.8931250	best: 1328.8931250 (9000)	total: 6m 48s	remain

2026-02-13 04:55:08 | INFO | ed_cost | [fold 0] CAT best_iteration=11999
2026-02-13 04:55:08 | INFO | ed_cost | [fold 0] CAT task_type=GPU devices=0
2026-02-13 04:55:08 | INFO | ed_cost | [fold_0/train_cat_gpu] END (548.59s)
2026-02-13 04:55:08 | INFO | ed_cost | [fold 0] CAT fold MAE=1301.421890
2026-02-13 04:55:08 | INFO | ed_cost | [fold_0] END (552.22s)
2026-02-13 04:55:08 | INFO | ed_cost | [fold_1] START
2026-02-13 04:55:08 | INFO | ed_cost | [fold_1/text_fit] START
2026-02-13 04:55:08 | INFO | ed_cost | [fold 1] TFIDF fitted. SVD dim=64
2026-02-13 04:55:08 | INFO | ed_cost | [fold_1/text_fit] END (0.13s)
2026-02-13 04:55:08 | INFO | ed_cost | [fold_1/text_transform] START
2026-02-13 04:55:08 | INFO | ed_cost | [fold_1/text_transform] END (0.12s)
2026-02-13 04:55:08 | INFO | ed_cost | [fold_1/build_cat_matrices] START
2026-02-13 04:55:08 | INFO | ed_cost | [fold 1] CatBoost X shapes: tr=(1600, 103) va=(400, 103) te=(2000, 103)
2026-02-13 04:55:08 | INFO | ed_cost | [fold_1/build_

0:	learn: 1425.2678125	test: 1465.2098437	best: 1465.2098437 (0)	total: 52.1ms	remaining: 10m 24s
1000:	learn: 1415.2487500	test: 1455.1546875	best: 1455.1546875 (1000)	total: 49.9s	remaining: 9m 8s
2000:	learn: 1405.2451563	test: 1445.1734375	best: 1445.1734375 (2000)	total: 1m 41s	remaining: 8m 28s
3000:	learn: 1395.2831250	test: 1435.2625000	best: 1435.2625000 (3000)	total: 2m 34s	remaining: 7m 43s
4000:	learn: 1385.4029687	test: 1425.4143750	best: 1425.4143750 (4000)	total: 3m 24s	remaining: 6m 47s
5000:	learn: 1375.5896875	test: 1415.6446875	best: 1415.6446875 (5000)	total: 4m 15s	remaining: 5m 58s
6000:	learn: 1365.7959375	test: 1405.8787500	best: 1405.8787500 (6000)	total: 5m 5s	remaining: 5m 5s
7000:	learn: 1356.0782812	test: 1396.1487500	best: 1396.1487500 (7000)	total: 5m 55s	remaining: 4m 13s
8000:	learn: 1346.4046875	test: 1386.4679687	best: 1386.4679687 (8000)	total: 6m 46s	remaining: 3m 23s
9000:	learn: 1336.8250000	test: 1376.8540625	best: 1376.8540625 (9000)	total: 7m 3

2026-02-13 05:05:19 | INFO | ed_cost | [fold 1] CAT best_iteration=11999
2026-02-13 05:05:19 | INFO | ed_cost | [fold 1] CAT task_type=GPU devices=0
2026-02-13 05:05:19 | INFO | ed_cost | [fold_1/train_cat_gpu] END (606.60s)
2026-02-13 05:05:19 | INFO | ed_cost | [fold 1] CAT fold MAE=1348.118971
2026-02-13 05:05:19 | INFO | ed_cost | [fold_1] END (610.81s)
2026-02-13 05:05:19 | INFO | ed_cost | [fold_2] START
2026-02-13 05:05:19 | INFO | ed_cost | [fold_2/text_fit] START
2026-02-13 05:05:19 | INFO | ed_cost | [fold 2] TFIDF fitted. SVD dim=64
2026-02-13 05:05:19 | INFO | ed_cost | [fold_2/text_fit] END (0.12s)
2026-02-13 05:05:19 | INFO | ed_cost | [fold_2/text_transform] START
2026-02-13 05:05:19 | INFO | ed_cost | [fold_2/text_transform] END (0.12s)
2026-02-13 05:05:19 | INFO | ed_cost | [fold_2/build_cat_matrices] START
2026-02-13 05:05:19 | INFO | ed_cost | [fold 2] CatBoost X shapes: tr=(1600, 103) va=(400, 103) te=(2000, 103)
2026-02-13 05:05:19 | INFO | ed_cost | [fold_2/build_

...

2026-02-13 05:15:10 | INFO | ed_cost | [fold 2] CAT best_iteration=11999
2026-02-13 05:15:10 | INFO | ed_cost | [fold 2] CAT task_type=GPU devices=0
2026-02-13 05:15:10 | INFO | ed_cost | [fold_2/train_cat_gpu] END (584.45s)
2026-02-13 05:15:10 | INFO | ed_cost | [fold 2] CAT fold MAE=1328.361483
2026-02-13 05:15:10 | INFO | ed_cost | [fold_2] END (591.23s)
2026-02-13 05:15:10 | INFO | ed_cost | [fold_3] START
2026-02-13 05:15:10 | INFO | ed_cost | [fold_3/text_fit] START
2026-02-13 05:15:10 | INFO | ed_cost | [fold 3] TFIDF fitted. SVD dim=64
2026-02-13 05:15:10 | INFO | ed_cost | [fold_3/text_fit] END (0.12s)
2026-02-13 05:15:10 | INFO | ed_cost | [fold_3/text_transform] START
2026-02-13 05:15:10 | INFO | ed_cost | [fold_3/text_transform] END (0.14s)
2026-02-13 05:15:10 | INFO | ed_cost | [fold_3/build_cat_matrices] START
2026-02-13 05:15:10 | INFO | ed_cost | [fold 3] CatBoost X shapes: tr=(1600, 103) va=(400, 103) te=(2000, 103)
2026-02-13 05:15:10 | INFO | ed_cost | [fold_3/build_

0:	learn: 1413.9860938	test: 1510.3362500	best: 1510.3362500 (0)	total: 117ms	remaining: 23m 28s
1000:	learn: 1403.8800000	test: 1500.3078125	best: 1500.3078125 (1000)	total: 49s	remaining: 8m 58s
2000:	learn: 1393.7443750	test: 1490.3532813	best: 1490.3532813 (2000)	total: 1m 37s	remaining: 8m 8s
3000:	learn: 1383.6840625	test: 1480.5075000	best: 1480.5075000 (3000)	total: 2m 29s	remaining: 7m 29s
4000:	learn: 1373.6878125	test: 1470.7120313	best: 1470.7120313 (4000)	total: 3m 17s	remaining: 6m 34s
5000:	learn: 1363.7971875	test: 1460.9421875	best: 1460.9421875 (5000)	total: 4m 5s	remaining: 5m 43s
6000:	learn: 1353.9165625	test: 1451.1759375	best: 1451.1759375 (6000)	total: 4m 52s	remaining: 4m 52s
7000:	learn: 1344.0948438	test: 1441.4346875	best: 1441.4346875 (7000)	total: 5m 41s	remaining: 4m 3s
8000:	learn: 1334.3189063	test: 1431.7159375	best: 1431.7159375 (8000)	total: 6m 30s	remaining: 3m 14s
9000:	learn: 1324.6471875	test: 1422.0784375	best: 1422.0784375 (9000)	total: 7m 18s	

2026-02-13 05:25:00 | INFO | ed_cost | [fold 3] CAT best_iteration=11999
2026-02-13 05:25:00 | INFO | ed_cost | [fold 3] CAT task_type=GPU devices=0
2026-02-13 05:25:00 | INFO | ed_cost | [fold_3/train_cat_gpu] END (581.92s)
2026-02-13 05:25:00 | INFO | ed_cost | [fold 3] CAT fold MAE=1393.485758
2026-02-13 05:25:00 | INFO | ed_cost | [fold_3] END (590.08s)
2026-02-13 05:25:00 | INFO | ed_cost | [fold_4] START
2026-02-13 05:25:00 | INFO | ed_cost | [fold_4/text_fit] START
2026-02-13 05:25:00 | INFO | ed_cost | [fold 4] TFIDF fitted. SVD dim=64
2026-02-13 05:25:00 | INFO | ed_cost | [fold_4/text_fit] END (0.16s)
2026-02-13 05:25:00 | INFO | ed_cost | [fold_4/text_transform] START
2026-02-13 05:25:00 | INFO | ed_cost | [fold_4/text_transform] END (0.14s)
2026-02-13 05:25:00 | INFO | ed_cost | [fold_4/build_cat_matrices] START
2026-02-13 05:25:01 | INFO | ed_cost | [fold 4] CatBoost X shapes: tr=(1600, 103) va=(400, 103) te=(2000, 103)
2026-02-13 05:25:01 | INFO | ed_cost | [fold_4/build_

0:	learn: 1456.0321875	test: 1342.0618750	best: 1342.0618750 (0)	total: 52.6ms	remaining: 10m 31s
1000:	learn: 1445.7978125	test: 1332.3592188	best: 1332.3592188 (1000)	total: 49s	remaining: 8m 58s
2000:	learn: 1435.6004687	test: 1322.7317188	best: 1322.7317188 (2000)	total: 1m 39s	remaining: 8m 16s
3000:	learn: 1425.4695313	test: 1313.1654687	best: 1313.1654687 (3000)	total: 2m 29s	remaining: 7m 29s
4000:	learn: 1415.3850000	test: 1303.6043750	best: 1303.6043750 (4000)	total: 3m 19s	remaining: 6m 39s
5000:	learn: 1405.3956250	test: 1294.0937500	best: 1294.0937500 (5000)	total: 4m 9s	remaining: 5m 49s
6000:	learn: 1395.4846875	test: 1284.6541406	best: 1284.6541406 (6000)	total: 4m 59s	remaining: 4m 59s
7000:	learn: 1385.6387500	test: 1275.2500000	best: 1275.2500000 (7000)	total: 5m 47s	remaining: 4m 8s
8000:	learn: 1375.8400000	test: 1265.8581250	best: 1265.8581250 (8000)	total: 6m 37s	remaining: 3m 18s
9000:	learn: 1366.1431250	test: 1256.5300000	best: 1256.5300000 (9000)	total: 7m 25

2026-02-13 05:34:58 | INFO | ed_cost | [fold 4] CAT best_iteration=11999
2026-02-13 05:34:58 | INFO | ed_cost | [fold 4] CAT task_type=GPU devices=0
2026-02-13 05:34:58 | INFO | ed_cost | [fold_4/train_cat_gpu] END (594.19s)
2026-02-13 05:34:58 | INFO | ed_cost | [fold 4] CAT fold MAE=1228.795787
2026-02-13 05:34:58 | INFO | ed_cost | [fold_4] END (598.14s)
2026-02-13 05:34:58 | INFO | ed_cost | [blend_oof_search] START
2026-02-13 05:34:58 | INFO | ed_cost | OOF MAE XGB=474.474945 | CAT=1320.036743 | BLEND=474.474810 (w_xgb=1.00, bias=-0.6155)
2026-02-13 05:34:58 | INFO | ed_cost | [blend_oof_search] END (0.04s)
2026-02-13 05:34:58 | INFO | ed_cost | [submission_write] START
2026-02-13 05:34:58 | INFO | ed_cost | Submission written: D:\AgentDs\agent_ds_healthcare\submission_ICHI_V1.csv | rows=2000
2026-02-13 05:34:58 | INFO | ed_cost | Submission head:
   patient_id  ed_cost_next3y_usd
0        3870         2566.093018
1        3622         4293.876953
2        1967         2075.880859

# Iteration 2
- 483.9550

In [18]:
# ============================================================
# AgentDS Challenge 2 — ITERATION CELL (GPU XGB + GPU CatBoost)
# Experiments implemented:
#   (1) Top-K CPT/HCPCS code count + cost-share features from PDFs
#   (2) Add XGB tail model: objective="reg:squaredlogerror"
#       + 3-way blend (XGB_MAE + XGB_SLOG + CAT) + group median calibration
# Also:
#   - Forensic OOF analysis (MAE by decile + segments)
#   - Strong logging + stage/fold timings + GPU verification
#   - PDF parsing via PyMuPDF with disk cache (never re-parse unless forced)
#
# HARD REQUIREMENTS PRESERVED:
#   - XGB early stopping rounds in CONSTRUCTOR (not in fit)
#   - XGB callbacks in CONSTRUCTOR (progress callback)
#   - CatBoost verbose=500
#   - Max CV folds = 5
# ============================================================

# If you need installs (commented):
# !pip install -U pymupdf xgboost catboost scikit-learn scipy joblib pyarrow

import os
import re
import json
import math
import time
import random
import logging
import platform
import subprocess
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, List, Tuple, Dict, Any
from collections import defaultdict

import numpy as np
import pandas as pd

import fitz  # PyMuPDF

from joblib import Parallel, delayed
from scipy import sparse

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import OneHotEncoder

import xgboost as xgb
from catboost import CatBoostRegressor, Pool

import torch

# -----------------------------
# USER CONFIG (paths + toggles)
# -----------------------------
DATA_DIR = Path(r"D:\AgentDs\agent_ds_healthcare")
TRAIN_CSV = DATA_DIR / "ed_cost_train.csv"
TEST_CSV  = DATA_DIR / "ed_cost_test.csv"
PATIENTS_CSV = DATA_DIR / "patients.csv"          # optional but recommended
PDF_DIR = DATA_DIR / "receipts_pdf"
SUBMISSION_PATH = DATA_DIR / "submission_ICHI_V1.csv"

CACHE_DIR = DATA_DIR / "cache"
MODEL_DIR = DATA_DIR / "models_gpu_iter_v2"

# Cache version: bump this string to force a new cache file (without FORCE_PDF_REEXTRACT)
PDF_CACHE_VERSION = "v2"  # contains code dicts + entropy/gini
FORCE_PDF_REEXTRACT = False  # must be True to re-parse if cache exists but is wrong/outdated

# CV control
N_FOLDS = 5
SEED = 2026

# PDF extraction parallelism
PDF_N_JOBS = 8  # threads

# Text embedding (fold-safe)
TFIDF_MAX_FEATURES = 6000
SVD_COMPONENTS = 256

# Top code features (Experiment 1)
TOP_CODE_K = 200
TOP_CODE_MIN_TOTAL_COUNT = 8

# XGBoost / CatBoost caps
XGB_N_ESTIMATORS = 8000
XGB_EARLY_STOPPING = 400

CAT_ITERATIONS = 12000
CAT_OD_WAIT = 400

# Optional "smoke test" (keeps pipeline identical, just cheaper)
FAST_DEV_RUN = False
if FAST_DEV_RUN:
    N_FOLDS = 2
    XGB_N_ESTIMATORS = 800
    XGB_EARLY_STOPPING = 80
    CAT_ITERATIONS = 1200
    CAT_OD_WAIT = 80
    SVD_COMPONENTS = 64
    TFIDF_MAX_FEATURES = 2000
    TOP_CODE_K = 60
    PDF_N_JOBS = 4

# GPU requirement
ASSERT_GPU_REQUIRED = True  # set False if you want CPU fallback (not recommended for your env)

# -----------------------------
# Logging utils
# -----------------------------
def setup_logger(log_path: Path, level=logging.INFO) -> logging.Logger:
    logger = logging.getLogger("ed_cost_iter")
    logger.setLevel(level)
    logger.handlers.clear()
    logger.propagate = False

    fmt = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    logger.addHandler(sh)

    log_path.parent.mkdir(parents=True, exist_ok=True)
    fh = logging.FileHandler(log_path, encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    return logger


class StageTimer:
    def __init__(self, logger: logging.Logger, name: str):
        self.logger = logger
        self.name = name
        self.t0 = None

    def __enter__(self):
        self.t0 = time.perf_counter()
        self.logger.info(f"[{self.name}] START")
        return self

    def __exit__(self, exc_type, exc, tb):
        dt = time.perf_counter() - self.t0
        if exc is None:
            self.logger.info(f"[{self.name}] END ({dt:.2f}s)")
        else:
            self.logger.exception(f"[{self.name}] FAILED after {dt:.2f}s: {exc}")


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


def try_run_cmd(cmd: List[str]) -> str:
    try:
        out = subprocess.check_output(cmd, stderr=subprocess.STDOUT, text=True)
        return out.strip()
    except Exception as e:
        return f"<command failed: {e}>"


def parquet_available() -> bool:
    try:
        import pyarrow  # noqa: F401
        return True
    except Exception:
        try:
            import fastparquet  # noqa: F401
            return True
        except Exception:
            return False


# -----------------------------
# XGB progress callback (logs via python logging)
# -----------------------------
try:
    from xgboost.callback import TrainingCallback
except Exception:
    TrainingCallback = object  # fallback

class XGBProgressCallback(TrainingCallback):
    def __init__(self, logger: logging.Logger, period: int = 200, prefix: str = "XGB"):
        self.logger = logger
        self.period = int(period)
        self.prefix = prefix

    def after_iteration(self, model, epoch: int, evals_log: Dict[str, Any]) -> bool:
        # called each boosting iteration
        if (epoch + 1) % self.period == 0:
            try:
                parts = []
                for ds, mets in (evals_log or {}).items():
                    for mname, hist in mets.items():
                        if hist:
                            parts.append(f"{ds}-{mname}={hist[-1]:.6f}")
                msg = " ".join(parts) if parts else "(no eval log)"
                self.logger.info(f"[{self.prefix}] iter={epoch+1} {msg}")
            except Exception:
                self.logger.info(f"[{self.prefix}] iter={epoch+1} (log parse fail)")
        return False


def fit_xgb_with_guard(
    params: Dict[str, Any],
    fold_seed: int,
    callbacks: List[Any],
    X_tr,
    y_tr,
    X_va,
    y_va,
    logger: logging.Logger,
) -> xgb.XGBRegressor:
    """
    Default path (MUST KEEP):
      - early_stopping_rounds in constructor params
      - callbacks in constructor
      - fit() clean: eval_set only, verbose=False
    Fallback path triggers only on TypeError.
    """
    # 1) Default (your fixed pattern)
    try:
        model = xgb.XGBRegressor(
            **{**params, "random_state": fold_seed},
            callbacks=callbacks,
        )
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        return model
    except TypeError as e:
        logger.warning(f"[XGB guard] Default constructor pattern TypeError: {e}")

    # 2) Fallback A: callbacks in fit()
    try:
        model = xgb.XGBRegressor(**{**params, "random_state": fold_seed})
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False, callbacks=callbacks)
        return model
    except TypeError as e:
        logger.warning(f"[XGB guard] fit(callbacks=...) TypeError: {e}")

    # 3) Fallback B: move early_stopping_rounds to fit() ONLY if constructor rejects it
    esr = params.get("early_stopping_rounds", None)
    params2 = dict(params)
    params2.pop("early_stopping_rounds", None)
    model = xgb.XGBRegressor(**{**params2, "random_state": fold_seed})
    try:
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False,
            early_stopping_rounds=esr,
            callbacks=callbacks,
        )
        return model
    except TypeError as e:
        logger.error(f"[XGB guard] Could not apply early stopping in fallback: {e}")
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        return model


# -----------------------------
# PDF parsing (PyMuPDF, no OCR)
# -----------------------------
_MONEY_RE = re.compile(r"^\$?[\d,]+(?:\.\d{2})?$")
_CODE_RE  = re.compile(r"^[A-Z0-9]{5}$")
_LINE_PAT = re.compile(r"^([A-Z0-9]{5})\s+(.+?)\s+(\d+)\s+([\d,]+\.\d{2})\s+([\d,]+\.\d{2})$")

def _parse_money(s: str) -> float:
    s = str(s).strip().replace("$", "").replace(",", "")
    try:
        return float(s)
    except Exception:
        return float("nan")


@dataclass(frozen=True)
class PdfParsed:
    patient_id_pdf: Optional[int]
    zip3_pdf: Optional[str]
    insurance_pdf: Optional[str]
    total_pdf: float
    items: List[Tuple[str, str, int, float, float]]  # code, desc, qty, unit, line_total
    raw_text: str


def parse_receipt_pdf(pdf_path: Path) -> Optional[PdfParsed]:
    if not pdf_path.exists():
        return None

    try:
        doc = fitz.open(pdf_path)
        raw_text = "".join([p.get_text("text", sort=True) for p in doc])
        doc.close()
    except Exception:
        return None

    lines = [ln.strip() for ln in raw_text.splitlines() if ln.strip()]

    pid = None
    zip3 = None
    ins = None

    for ln in lines[:12]:
        m = re.search(r"Patient ID:\s*(\d+)", ln)
        if m:
            pid = int(m.group(1))
        m = re.search(r"ZIP3:\s*(\d{3})", ln)
        if m:
            zip3 = m.group(1)
        m = re.search(r"Insurance:\s*([A-Za-z_]+)", ln)
        if m:
            ins = m.group(1)

    # TOTAL
    total = float("nan")
    for i, ln in enumerate(lines):
        if ln.startswith("TOTAL"):
            parts = ln.split()
            if len(parts) >= 2 and _MONEY_RE.match(parts[1]):
                total = _parse_money(parts[1])
            elif i + 1 < len(lines) and _MONEY_RE.match(lines[i + 1]):
                total = _parse_money(lines[i + 1])
            break

    items: List[Tuple[str, str, int, float, float]] = []

    # Layout A: row-per-line
    for ln in lines:
        m = _LINE_PAT.match(ln)
        if m:
            code, desc, qty, unit, lt = m.groups()
            items.append((code, desc, int(qty), _parse_money(unit), _parse_money(lt)))

    # Layout B: columnar / cell-per-line
    if not items:
        start = 0
        for i, ln in enumerate(lines):
            if ln.lower().startswith("line total"):
                start = i + 1
                break

        i = start
        while i < len(lines):
            ln = lines[i]
            if ln.startswith("TOTAL"):
                break

            if _CODE_RE.match(ln):
                code = ln
                j = i + 1

                desc_parts = []
                while j < len(lines) and not re.fullmatch(r"\d+", lines[j]):
                    if lines[j].startswith("TOTAL"):
                        break
                    desc_parts.append(lines[j])
                    j += 1

                if j >= len(lines) or lines[j].startswith("TOTAL"):
                    break

                qty = int(lines[j]); j += 1
                if j + 1 >= len(lines):
                    break

                unit = _parse_money(lines[j])
                lt = _parse_money(lines[j + 1])
                desc = " ".join(desc_parts).strip()

                items.append((code, desc, qty, unit, lt))
                i = j + 2
            else:
                i += 1

    return PdfParsed(
        patient_id_pdf=pid,
        zip3_pdf=zip3,
        insurance_pdf=ins,
        total_pdf=total,
        items=items,
        raw_text=raw_text,
    )


def _gini(x: np.ndarray) -> float:
    # Gini for non-negative values
    if x.size == 0:
        return 0.0
    s = np.sort(np.maximum(x, 0.0))
    total = s.sum()
    if total <= 0:
        return 0.0
    n = s.size
    return float((2*np.arange(1, n+1) - n - 1).dot(s) / (n * total + 1e-12))


def compute_pdf_features(parsed: Optional[PdfParsed]) -> Dict[str, Any]:
    if parsed is None:
        return {
            "zip3_pdf": "UNK",
            "insurance_pdf": "UNK",
            "pdf_total": float("nan"),
            "pdf_n_items": 0,
            "pdf_n_unique_codes": 0,
            "pdf_sum_qty": 0.0,
            "pdf_mean_line_total": 0.0,
            "pdf_max_line_total": 0.0,
            "pdf_std_line_total": 0.0,
            "pdf_share_top1": 0.0,
            "pdf_cost_eval_mgmt": 0.0,
            "pdf_cost_lab": 0.0,
            "pdf_cost_radiology": 0.0,
            "pdf_cost_surgery": 0.0,
            "pdf_cost_other_numeric": 0.0,
            "pdf_cost_hcpcs_alpha": 0.0,
            "pdf_em_level_max": 0,
            "pdf_em_count": 0,
            "pdf_critical_count": 0,
            "pdf_code_counts_json": "{}",
            "pdf_code_costs_json": "{}",
            "pdf_cost_entropy": 0.0,
            "pdf_cost_gini": 0.0,
            "pdf_text": "",
            "pdf_missing": 1,
            "pdf_empty_items": 1,
        }

    items = parsed.items
    total = parsed.total_pdf

    codes = [it[0] for it in items]
    descs = [it[1] for it in items]

    qtys = np.array([it[2] for it in items], dtype=np.float32) if items else np.array([], dtype=np.float32)
    lts  = np.array([it[4] for it in items], dtype=np.float32) if items else np.array([], dtype=np.float32)

    n = len(items)
    n_unique = len(set(codes)) if n else 0

    mean_lt = float(lts.mean()) if n else 0.0
    max_lt  = float(lts.max()) if n else 0.0
    std_lt  = float(lts.std()) if n else 0.0
    share_top1 = float(max_lt / total) if n and (not math.isnan(total)) and total > 0 else 0.0

    eval_mgmt = lab = radiology = surgery = other_numeric = hcpcs_alpha = 0.0
    em_level_max = 0
    em_count = 0
    critical_count = 0

    code_counts = defaultdict(int)
    code_costs  = defaultdict(float)

    for code, _desc, qty, _unit, lt in items:
        if code:
            code_counts[code] += int(qty)
            code_costs[code]  += float(lt)

        if code and code[0].isdigit():
            ci = int(code)
            if 90000 <= ci <= 99999:
                eval_mgmt += lt
            elif 80000 <= ci <= 89999:
                lab += lt
            elif 70000 <= ci <= 79999:
                radiology += lt
            elif 10000 <= ci <= 69999:
                surgery += lt
            else:
                other_numeric += lt

            if 99281 <= ci <= 99285:
                em_count += 1
                em_level_max = max(em_level_max, ci - 99280)

            if ci in (99291, 99292):
                critical_count += 1
        else:
            hcpcs_alpha += lt

    cost_vals = np.array(list(code_costs.values()), dtype=np.float64)
    if cost_vals.size > 0 and cost_vals.sum() > 0:
        p = cost_vals / cost_vals.sum()
        cost_entropy = float(-(p * np.log(p + 1e-12)).sum())
        cost_gini = _gini(cost_vals)
    else:
        cost_entropy = 0.0
        cost_gini = 0.0

    pdf_text = " ".join([f"{c} {d}" for c, d in zip(codes, descs)])

    return {
        "zip3_pdf": parsed.zip3_pdf or "UNK",
        "insurance_pdf": parsed.insurance_pdf or "UNK",
        "pdf_total": total,
        "pdf_n_items": int(n),
        "pdf_n_unique_codes": int(n_unique),
        "pdf_sum_qty": float(qtys.sum()) if n else 0.0,
        "pdf_mean_line_total": mean_lt,
        "pdf_max_line_total": max_lt,
        "pdf_std_line_total": std_lt,
        "pdf_share_top1": share_top1,
        "pdf_cost_eval_mgmt": float(eval_mgmt),
        "pdf_cost_lab": float(lab),
        "pdf_cost_radiology": float(radiology),
        "pdf_cost_surgery": float(surgery),
        "pdf_cost_other_numeric": float(other_numeric),
        "pdf_cost_hcpcs_alpha": float(hcpcs_alpha),
        "pdf_em_level_max": int(em_level_max),
        "pdf_em_count": int(em_count),
        "pdf_critical_count": int(critical_count),
        "pdf_code_counts_json": json.dumps(code_counts),
        "pdf_code_costs_json": json.dumps(code_costs),
        "pdf_cost_entropy": float(cost_entropy),
        "pdf_cost_gini": float(cost_gini),
        "pdf_text": pdf_text,
        "pdf_missing": 0,
        "pdf_empty_items": 1 if n == 0 else 0,
    }


def load_or_extract_pdf_features(
    logger: logging.Logger,
    all_patient_ids: List[int],
    pdf_dir: Path,
    cache_dir: Path,
    cache_version: str,
    force: bool,
    n_jobs: int,
) -> pd.DataFrame:
    cache_dir.mkdir(parents=True, exist_ok=True)
    use_parquet = parquet_available()
    cache_path = cache_dir / (f"pdf_features_all_{cache_version}.parquet" if use_parquet else f"pdf_features_all_{cache_version}.pkl")

    required_cols = {
        "patient_id",
        "zip3_pdf",
        "insurance_pdf",
        "pdf_total",
        "pdf_n_items",
        "pdf_code_counts_json",
        "pdf_code_costs_json",
        "pdf_cost_entropy",
        "pdf_cost_gini",
        "pdf_text",
        "pdf_missing",
        "pdf_empty_items",
    }

    # file existence audit
    found = 0
    for pid in all_patient_ids:
        if (pdf_dir / f"receipt_{pid}.pdf").exists():
            found += 1
    logger.info(f"[PDF] Expected PDFs={len(all_patient_ids)} | Found={found} | Missing={len(all_patient_ids)-found}")

    if cache_path.exists() and not force:
        with StageTimer(logger, "pdf_cache_load"):
            df = pd.read_parquet(cache_path) if cache_path.suffix == ".parquet" else pd.read_pickle(cache_path)

        missing_cols = required_cols - set(df.columns)
        if missing_cols:
            raise RuntimeError(
                f"PDF cache exists but missing columns: {sorted(missing_cols)}\n"
                f"Cache path: {cache_path}\n"
                f"Set FORCE_PDF_REEXTRACT=True OR bump PDF_CACHE_VERSION."
            )
        logger.info(f"[PDF] Using cached features: {cache_path}")
        return df

    logger.info(f"[PDF] Extracting via PyMuPDF -> cache={cache_path.name}")

    def extract_one(pid: int) -> Dict[str, Any]:
        pdf_path = pdf_dir / f"receipt_{pid}.pdf"
        parsed = parse_receipt_pdf(pdf_path)
        feats = compute_pdf_features(parsed)
        feats["patient_id"] = int(pid)
        return feats

    with StageTimer(logger, "pdf_extract_total"):
        rows = Parallel(n_jobs=n_jobs, prefer="threads")(
            delayed(extract_one)(pid) for pid in all_patient_ids
        )

    df = pd.DataFrame(rows).sort_values("patient_id").reset_index(drop=True)

    # parse quality stats
    n_missing = int(df["pdf_missing"].sum()) if "pdf_missing" in df.columns else -1
    n_empty = int(df["pdf_empty_items"].sum()) if "pdf_empty_items" in df.columns else -1
    logger.info(f"[PDF] parsed rows={len(df)} | pdf_missing={n_missing} | pdf_empty_items={n_empty}")

    with StageTimer(logger, "pdf_cache_write"):
        if cache_path.suffix == ".parquet":
            df.to_parquet(cache_path, index=False)
        else:
            df.to_pickle(cache_path)
    logger.info(f"[PDF] Cache written: {cache_path}")

    return df


# -----------------------------
# Feature engineering
# -----------------------------
def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # categorical normalization
    for c in ["primary_chronic", "sex", "insurance", "zip3", "zip3_pdf", "insurance_pdf"]:
        if c in out.columns:
            out[c] = out[c].fillna("UNK").astype(str)

    # numeric fill
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].fillna(0.0)

    # utilization transforms
    if "prior_ed_cost_5y_usd" in out.columns:
        out["prior_cost_log1p"] = np.log1p(out["prior_ed_cost_5y_usd"].astype(float))
    if "prior_ed_visits_5y" in out.columns:
        out["prior_visits_log1p"] = np.log1p(out["prior_ed_visits_5y"].astype(float))
    if "prior_ed_cost_5y_usd" in out.columns and "prior_ed_visits_5y" in out.columns:
        out["prior_cost_per_visit"] = out["prior_ed_cost_5y_usd"] / (out["prior_ed_visits_5y"] + 1.0)
        out["prior_cost_per_visit_log1p"] = np.log1p(out["prior_cost_per_visit"].astype(float))
        out["prior_cost_x_visits"] = out["prior_ed_cost_5y_usd"] * out["prior_ed_visits_5y"]

    # normalize PDF cost buckets by prior cost (avoid using pdf_total directly)
    denom = out["prior_ed_cost_5y_usd"].astype(float) + 1.0
    for c in [
        "pdf_cost_eval_mgmt",
        "pdf_cost_lab",
        "pdf_cost_radiology",
        "pdf_cost_surgery",
        "pdf_cost_other_numeric",
        "pdf_cost_hcpcs_alpha",
    ]:
        if c in out.columns:
            out[f"{c}_share"] = out[c].astype(float) / denom

    if "pdf_n_items" in out.columns and "prior_ed_visits_5y" in out.columns:
        out["pdf_items_per_visit"] = out["pdf_n_items"].astype(float) / (out["prior_ed_visits_5y"] + 1.0)

    # final numeric fill
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].fillna(0.0)

    return out


def add_top_code_features(
    logger: logging.Logger,
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    top_k: int,
    min_total_count: int,
) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    """
    Experiment 1:
      - Choose top codes by total count in TRAIN (unsupervised; no label leakage)
      - Add:
         code_cnt_<CODE>
         code_cost_share_<CODE> = code_cost / (prior_cost + 1)
    """
    def parse_dict(s: Any) -> Dict[str, Any]:
        if not isinstance(s, str) or not s:
            return {}
        try:
            return json.loads(s)
        except Exception:
            return {}

    # select codes from train counts
    freq: Dict[str, int] = {}
    for s in train_df["pdf_code_counts_json"].fillna("{}").values:
        d = parse_dict(s)
        for k, v in d.items():
            freq[k] = freq.get(k, 0) + int(v)

    items = [(k, v) for k, v in freq.items() if v >= min_total_count]
    items.sort(key=lambda x: x[1], reverse=True)
    top_codes = [k for k, _ in items[:top_k]]
    code_to_idx = {c: i for i, c in enumerate(top_codes)}

    logger.info(f"[TopCodes] selected={len(top_codes)} (top_k={top_k}, min_total_count={min_total_count})")
    logger.info(f"[TopCodes] examples={top_codes[:20]}")

    def build(df: pd.DataFrame) -> pd.DataFrame:
        n = len(df)
        K = len(top_codes)
        cnt = np.zeros((n, K), dtype=np.float32)
        shr = np.zeros((n, K), dtype=np.float32)
        denom = df["prior_ed_cost_5y_usd"].astype(np.float32).values + 1.0

        for i, s in enumerate(df["pdf_code_counts_json"].fillna("{}").values):
            d = parse_dict(s)
            for code, v in d.items():
                j = code_to_idx.get(code)
                if j is not None:
                    cnt[i, j] = float(v)

        for i, s in enumerate(df["pdf_code_costs_json"].fillna("{}").values):
            d = parse_dict(s)
            for code, v in d.items():
                j = code_to_idx.get(code)
                if j is not None:
                    shr[i, j] = float(v) / denom[i]

        cols_cnt = [f"code_cnt_{c}" for c in top_codes]
        cols_shr = [f"code_cost_share_{c}" for c in top_codes]
        mat = np.hstack([cnt, shr])
        return pd.DataFrame(mat, columns=cols_cnt + cols_shr)

    with StageTimer(logger, "topcodes_build"):
        tr_mat = build(train_df)
        te_mat = build(test_df)

    train_df2 = pd.concat([train_df.reset_index(drop=True), tr_mat], axis=1)
    test_df2  = pd.concat([test_df.reset_index(drop=True),  te_mat], axis=1)
    logger.info(f"[TopCodes] train shape {train_df.shape} -> {train_df2.shape} | test {test_df.shape} -> {test_df2.shape}")

    return train_df2, test_df2, top_codes


# -----------------------------
# Fold-safe text: TFIDF -> SVD
# -----------------------------
def fit_text_models(train_text: np.ndarray, seed: int) -> Tuple[Optional[TfidfVectorizer], Optional[TruncatedSVD], int]:
    for min_df in (2, 1):
        vec = TfidfVectorizer(
            lowercase=True,
            token_pattern=r"(?u)\b[\w']+\b",
            ngram_range=(1, 2),
            max_features=TFIDF_MAX_FEATURES,
            min_df=min_df,
            sublinear_tf=True,
        )
        X = vec.fit_transform(train_text)
        if X.shape[1] < 2:
            continue
        n_comp = min(SVD_COMPONENTS, X.shape[1] - 1)
        svd = TruncatedSVD(n_components=n_comp, random_state=seed)
        svd.fit(X)
        return vec, svd, n_comp
    return None, None, 0


def transform_text(vec: Optional[TfidfVectorizer], svd: Optional[TruncatedSVD], text: np.ndarray) -> np.ndarray:
    if vec is None or svd is None:
        return np.zeros((len(text), 0), dtype=np.float32)
    X = vec.transform(text)
    if X.shape[1] == 0:
        return np.zeros((len(text), 0), dtype=np.float32)
    return svd.transform(X).astype(np.float32)


def make_ohe() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True, dtype=np.float32)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True, dtype=np.float32)


# -----------------------------
# Blender + calibration + OOF diagnostics
# -----------------------------
def find_best_3way_blend(oof_a: np.ndarray, oof_b: np.ndarray, oof_c: np.ndarray, y: np.ndarray, step: float = 0.02) -> Dict[str, float]:
    best = {"w_a": 0.0, "w_b": 0.0, "w_c": 1.0, "bias": 0.0, "mae": float("inf")}
    grid = np.arange(0.0, 1.0 + 1e-9, step)
    for w_a in grid:
        for w_b in grid:
            w_c = 1.0 - w_a - w_b
            if w_c < -1e-9:
                continue
            if w_c < 0:
                w_c = 0.0
            pred = w_a * oof_a + w_b * oof_b + w_c * oof_c
            bias = float(np.median(pred - y))  # MAE-optimal constant shift (median residual)
            pred2 = np.clip(pred - bias, 0.0, None)
            mae = float(mean_absolute_error(y, pred2))
            if mae < best["mae"]:
                best = {"w_a": float(w_a), "w_b": float(w_b), "w_c": float(w_c), "bias": float(bias), "mae": float(mae)}
    return best


def fit_group_median_calibration(oof_pred: np.ndarray, y: np.ndarray, meta: pd.DataFrame, logger: logging.Logger) -> Dict[str, Any]:
    ins_col = "insurance" if "insurance" in meta.columns else ("insurance_pdf" if "insurance_pdf" in meta.columns else None)
    if ins_col is None:
        logger.warning("[Calib] No insurance column found; calibration will be primary_chronic only.")
        ins_col = "__none__"
        g1 = meta["primary_chronic"].astype(str).values
    else:
        g1 = (meta["primary_chronic"].astype(str) + "|" + meta[ins_col].astype(str)).values

    g2 = meta["primary_chronic"].astype(str).values
    resid = (oof_pred - y).astype(np.float64)

    df = pd.DataFrame({"g1": g1, "g2": g2, "resid": resid})
    med1 = df.groupby("g1")["resid"].median()
    med2 = df.groupby("g2")["resid"].median()

    cal = {"ins_col": ins_col, "level1": med1.to_dict(), "level2": med2.to_dict()}
    logger.info(f"[Calib] ins_col={ins_col} | level1_groups={len(cal['level1'])} | level2_groups={len(cal['level2'])}")
    return cal


def apply_group_median_calibration(pred: np.ndarray, meta: pd.DataFrame, cal: Dict[str, Any]) -> np.ndarray:
    ins_col = cal["ins_col"]
    if ins_col == "__none__":
        g1 = meta["primary_chronic"].astype(str).values
    else:
        g1 = (meta["primary_chronic"].astype(str) + "|" + meta[ins_col].astype(str)).values

    g2 = meta["primary_chronic"].astype(str).values
    lvl1 = cal["level1"]
    lvl2 = cal["level2"]

    adj = np.zeros(len(pred), dtype=np.float64)
    for i in range(len(pred)):
        k1 = g1[i]
        if k1 in lvl1:
            adj[i] = lvl1[k1]
        else:
            adj[i] = lvl2.get(g2[i], 0.0)

    return np.clip(pred - adj, 0.0, None)


def log_oof_error_analysis(logger: logging.Logger, meta: pd.DataFrame, y_true: np.ndarray, y_pred: np.ndarray) -> None:
    tmp = meta.copy()
    tmp["_y"] = y_true
    tmp["_pred"] = y_pred
    tmp["_ae"] = np.abs(tmp["_y"] - tmp["_pred"])

    logger.info(f"[OOF] MAE={tmp['_ae'].mean():.6f} | median_AE={tmp['_ae'].median():.6f} | p90_AE={tmp['_ae'].quantile(0.90):.2f}")

    # MAE by target decile + share of total absolute error (who drives MAE)
    tmp["_decile"] = pd.qcut(tmp["_y"], 10, labels=False, duplicates="drop")
    dec = tmp.groupby("_decile").agg(
        count=("_ae","size"),
        mae=("_ae","mean"),
        sum_ae=("_ae","sum"),
        y_mean=("_y","mean"),
        y_p90=("_y", lambda s: float(np.quantile(s, 0.9))),
    ).reset_index()
    dec["share_ae"] = dec["sum_ae"] / (dec["sum_ae"].sum() + 1e-12)
    logger.info("[OOF] MAE by target decile:\n" + dec.to_string(index=False))

    # segments
    seg_cols = ["primary_chronic"]
    for c in ["insurance", "insurance_pdf", "zip3", "zip3_pdf"]:
        if c in tmp.columns:
            seg_cols.append(c)

    for c in seg_cols:
        g = tmp.groupby(c).agg(count=("_ae","size"), mae=("_ae","mean"), sum_ae=("_ae","sum")).sort_values("count", ascending=False)
        logger.info(f"[OOF] by {c} (top 15 by count):\n" + g.head(15).to_string())

    # interaction chronic x insurance
    ins_col = "insurance" if "insurance" in tmp.columns else ("insurance_pdf" if "insurance_pdf" in tmp.columns else None)
    if ins_col is not None:
        key = tmp["primary_chronic"].astype(str) + "|" + tmp[ins_col].astype(str)
        g = tmp.groupby(key).agg(count=("_ae","size"), mae=("_ae","mean"), sum_ae=("_ae","sum")).sort_values("count", ascending=False)
        logger.info(f"[OOF] by primary_chronic x {ins_col}:\n" + g.to_string())


# -----------------------------
# MAIN TRAIN + SUBMIT
# -----------------------------
def run_pipeline():
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

    log_path = MODEL_DIR / "train_log.txt"
    logger = setup_logger(log_path)
    seed_everything(SEED)

    # ---- env report ----
    with StageTimer(logger, "env_report"):
        logger.info(f"OS: {platform.platform()}")
        logger.info(f"Python: {platform.python_version()}")
        logger.info(f"numpy={np.__version__} pandas={pd.__version__}")
        import sklearn
        logger.info(f"sklearn={sklearn.__version__}")
        logger.info(f"xgboost={xgb.__version__}")
        import catboost
        logger.info(f"catboost={catboost.__version__}")
        logger.info(f"torch={torch.__version__} cuda_available={torch.cuda.is_available()}")

        if ASSERT_GPU_REQUIRED and not torch.cuda.is_available():
            raise RuntimeError("CUDA not available in torch. Set ASSERT_GPU_REQUIRED=False to proceed (not recommended).")

        if torch.cuda.is_available():
            logger.info(f"torch.cuda.device_count={torch.cuda.device_count()}")
            logger.info(f"GPU[0]={torch.cuda.get_device_name(0)}")
            logger.info(f"GPU[0] capability={torch.cuda.get_device_capability(0)}")
            logger.info("nvidia-smi -L:\n" + try_run_cmd(["nvidia-smi", "-L"]))

    # ---- load train/test ----
    with StageTimer(logger, "load_data"):
        train = pd.read_csv(TRAIN_CSV)
        test = pd.read_csv(TEST_CSV)
        logger.info(f"train shape={train.shape} | test shape={test.shape}")
        logger.info(f"train cols={list(train.columns)}")

        # preserve test order strictly
        test_order_ids = test["patient_id"].astype(int).values.copy()

    # ---- optional patients join ----
    if PATIENTS_CSV.exists():
        with StageTimer(logger, "join_patients"):
            patients = pd.read_csv(PATIENTS_CSV, dtype={"zip3": str})
            keep = [c for c in ["patient_id", "age", "sex", "insurance", "zip3"] if c in patients.columns]
            patients = patients[keep]
            train = train.merge(patients, on="patient_id", how="left", sort=False)
            test  = test.merge(patients, on="patient_id", how="left", sort=False)
            logger.info(f"patients.csv joined | keep={keep} | train shape now={train.shape}")
    else:
        logger.warning("patients.csv not found. Continuing without it.")

    # ---- PDF cache ----
    with StageTimer(logger, "pdf_features"):
        all_ids = pd.concat([train["patient_id"], test["patient_id"]]).astype(int).tolist()
        pdf_feats = load_or_extract_pdf_features(
            logger=logger,
            all_patient_ids=all_ids,
            pdf_dir=PDF_DIR,
            cache_dir=CACHE_DIR,
            cache_version=PDF_CACHE_VERSION,
            force=FORCE_PDF_REEXTRACT,
            n_jobs=PDF_N_JOBS,
        )

    # ---- merge PDF feats ----
    with StageTimer(logger, "merge_pdf_features"):
        train = train.merge(pdf_feats, on="patient_id", how="left", sort=False)
        test  = test.merge(pdf_feats, on="patient_id", how="left", sort=False)

        # re-assert test order EXACTLY
        test = test.set_index("patient_id").loc[test_order_ids].reset_index()

    # ---- sanity: pdf_total vs prior_ed_cost_5y_usd (dataset says they match) ----
    if "pdf_total" in train.columns:
        ok = train["pdf_total"].notna() & (train["pdf_total"] > 0)
        if ok.any():
            abs_diff = (train.loc[ok, "prior_ed_cost_5y_usd"] - train.loc[ok, "pdf_total"]).abs()
            logger.info("[PDF sanity] |prior_ed_cost_5y_usd - pdf_total| describe:\n" + str(abs_diff.describe()))

    # ---- derived features ----
    with StageTimer(logger, "feature_engineering"):
        train = add_derived_features(train)
        test  = add_derived_features(test)

    # ---- EXP 1: top code features ----
    if ("pdf_code_counts_json" in train.columns) and ("pdf_code_costs_json" in train.columns):
        train, test, top_codes = add_top_code_features(
            logger=logger,
            train_df=train,
            test_df=test,
            top_k=TOP_CODE_K,
            min_total_count=TOP_CODE_MIN_TOTAL_COUNT,
        )
        (MODEL_DIR / "top_codes.json").write_text(json.dumps(top_codes, indent=2))
    else:
        logger.warning("pdf_code_*_json columns missing; skipping top code features.")

    # ---- build feature lists ----
    target_col = "ed_cost_next3y_usd"
    text_col = "pdf_text"

    # NOTE: pdf_total matches prior cost by dataset design; do NOT use it as feature (use only for sanity check)
    drop_cols = {
        "patient_id",
        target_col,
        text_col,
        "pdf_total",
        "pdf_code_counts_json",
        "pdf_code_costs_json",
    }

    # categorical columns
    cat_cols = ["primary_chronic"]
    for c in ["sex", "insurance", "zip3", "zip3_pdf", "insurance_pdf"]:
        if c in train.columns:
            cat_cols.append(c)
    # unique preserve
    seen = set()
    cat_cols = [c for c in cat_cols if not (c in seen or seen.add(c))]

    # numeric columns
    num_cols = []
    for c in train.columns:
        if c in drop_cols or c in cat_cols:
            continue
        if pd.api.types.is_numeric_dtype(train[c]):
            num_cols.append(c)
    num_cols = sorted(num_cols)

    logger.info(f"[Features] num_cols={len(num_cols)} | cat_cols={len(cat_cols)} -> {cat_cols}")
    logger.info(f"[Features] train shape={train.shape} | test shape={test.shape}")

    # ---- CV (max 5 folds; no repeats) ----
    assert N_FOLDS <= 5 and N_FOLDS == 5, "Hard requirement: CV must be exactly 5 folds."
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    logger.info(f"[CV] folds={N_FOLDS} stratify=primary_chronic (single pass)")

    y = train[target_col].astype(np.float32).values

    # predictions
    oof_xgb = np.zeros(len(train), dtype=np.float32)
    oof_xgb_slog = np.zeros(len(train), dtype=np.float32)
    oof_cat = np.zeros(len(train), dtype=np.float32)

    test_pred_xgb = np.zeros(len(test), dtype=np.float32)
    test_pred_xgb_slog = np.zeros(len(test), dtype=np.float32)
    test_pred_cat = np.zeros(len(test), dtype=np.float32)

    # ---- XGB params (KEEP YOUR FIXED PATTERN) ----
    xgb_params = dict(
        n_estimators=XGB_N_ESTIMATORS,
        learning_rate=0.03,
        max_depth=7,
        min_child_weight=20.0,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=2.0,
        objective="reg:absoluteerror",
        eval_metric="mae",
        tree_method="hist",
        device="cuda:0",
        random_state=SEED,
        n_jobs=0,
        early_stopping_rounds=XGB_EARLY_STOPPING,
    )
    # EXP 2: squaredlogerror model (tail-friendly); XGBoost docs list it and note labels must be > -1
    # (our costs are positive, so OK)
    xgb_params_slog = {**xgb_params, "objective": "reg:squaredlogerror", "eval_metric": "mae"}

    logger.info(f"[XGB] params(MAE)={xgb_params}")
    logger.info(f"[XGB] params(SLOG)={xgb_params_slog}")

    # ---- CatBoost params (GPU + verbose=500) ----
    cat_params = dict(
        loss_function="MAE",
        eval_metric="MAE",
        iterations=CAT_ITERATIONS,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=3.0,
        random_seed=SEED,
        task_type="GPU",
        devices="0",
        od_type="Iter",
        od_wait=CAT_OD_WAIT,
        verbose=500,  # MUST KEEP (prints every 500 iters)
        allow_writing_files=False,
    )
    logger.info(f"[CAT] params={cat_params}")

    # artifact dirs
    (MODEL_DIR / "fold_artifacts").mkdir(parents=True, exist_ok=True)

    # ---- fold loop ----
    for fold, (tr_idx, va_idx) in enumerate(skf.split(train, train["primary_chronic"].astype(str))):
        with StageTimer(logger, f"fold_{fold}"):
            tr = train.iloc[tr_idx].reset_index(drop=True)
            va = train.iloc[va_idx].reset_index(drop=True)

            y_tr = tr[target_col].astype(np.float32).values
            y_va = va[target_col].astype(np.float32).values

            # fold-safe text model
            with StageTimer(logger, f"fold_{fold}/text_fit"):
                vec, svd, svd_dim = fit_text_models(tr[text_col].fillna("").values, seed=SEED + fold)
                logger.info(f"[fold {fold}] text svd_dim={svd_dim}")

            with StageTimer(logger, f"fold_{fold}/text_transform"):
                tr_svd = transform_text(vec, svd, tr[text_col].fillna("").values)
                va_svd = transform_text(vec, svd, va[text_col].fillna("").values)
                te_svd = transform_text(vec, svd, test[text_col].fillna("").values)

            # --- CatBoost matrices ---
            with StageTimer(logger, f"fold_{fold}/build_cat_matrices"):
                X_tr_cb = tr[num_cols].astype(np.float32).copy()
                X_va_cb = va[num_cols].astype(np.float32).copy()
                X_te_cb = test[num_cols].astype(np.float32).copy()

                # add svd dims
                for j in range(tr_svd.shape[1]):
                    col = f"svd_{j}"
                    X_tr_cb[col] = tr_svd[:, j]
                    X_va_cb[col] = va_svd[:, j]
                    X_te_cb[col] = te_svd[:, j]

                # add categorical cols
                for c in cat_cols:
                    X_tr_cb[c] = tr[c].fillna("UNK").astype(str).values
                    X_va_cb[c] = va[c].fillna("UNK").astype(str).values
                    X_te_cb[c] = test[c].fillna("UNK").astype(str).values

                cat_feat_idx = [X_tr_cb.columns.get_loc(c) for c in cat_cols]
                logger.info(f"[fold {fold}] CatBoost X shapes tr={X_tr_cb.shape} va={X_va_cb.shape} te={X_te_cb.shape}")

                tr_pool = Pool(X_tr_cb, y_tr, cat_features=cat_feat_idx)
                va_pool = Pool(X_va_cb, y_va, cat_features=cat_feat_idx)
                te_pool = Pool(X_te_cb, cat_features=cat_feat_idx)

            # --- XGBoost matrices (sparse) ---
            with StageTimer(logger, f"fold_{fold}/build_xgb_matrices"):
                ohe = make_ohe()
                ohe.fit(tr[cat_cols].fillna("UNK").astype(str))

                tr_cat = ohe.transform(tr[cat_cols].fillna("UNK").astype(str))
                va_cat = ohe.transform(va[cat_cols].fillna("UNK").astype(str))
                te_cat = ohe.transform(test[cat_cols].fillna("UNK").astype(str))

                tr_num = tr[num_cols].astype(np.float32).values
                va_num = va[num_cols].astype(np.float32).values
                te_num = test[num_cols].astype(np.float32).values

                tr_dense = np.hstack([tr_num, tr_svd]).astype(np.float32)
                va_dense = np.hstack([va_num, va_svd]).astype(np.float32)
                te_dense = np.hstack([te_num, te_svd]).astype(np.float32)

                X_tr_xgb = sparse.hstack([sparse.csr_matrix(tr_dense), tr_cat], format="csr")
                X_va_xgb = sparse.hstack([sparse.csr_matrix(va_dense), va_cat], format="csr")
                X_te_xgb = sparse.hstack([sparse.csr_matrix(te_dense), te_cat], format="csr")

                logger.info(f"[fold {fold}] XGB X shapes tr={X_tr_xgb.shape} va={X_va_xgb.shape} te={X_te_xgb.shape}")

            # ---- Train XGB MAE (GPU) with REQUIRED pattern ----
            callbacks = [XGBProgressCallback(logger, period=200, prefix=f"XGB_MAE_f{fold}")]
            with StageTimer(logger, f"fold_{fold}/train_xgb_mae"):
                xgb_model = fit_xgb_with_guard(
                    params=xgb_params,
                    fold_seed=SEED + fold,
                    callbacks=callbacks,
                    X_tr=X_tr_xgb, y_tr=y_tr,
                    X_va=X_va_xgb, y_va=y_va,
                    logger=logger,
                )
                logger.info(f"[fold {fold}] XGB(MAE) get_params device={xgb_model.get_params().get('device')} tree_method={xgb_model.get_params().get('tree_method')}")
                best_iter = getattr(xgb_model, "best_iteration", None)
                logger.info(f"[fold {fold}] XGB(MAE) best_iteration={best_iter}")

                # verify GPU via booster config contains "cuda"
                try:
                    cfg = xgb_model.get_booster().save_config().lower()
                    logger.info(f"[fold {fold}] XGB(MAE) booster config contains 'cuda': {'cuda' in cfg}")
                    if fold == 0:
                        (MODEL_DIR / "xgb_booster_config_fold0_mae.json").write_text(xgb_model.get_booster().save_config())
                except Exception:
                    pass

                xgb_model.save_model(str(MODEL_DIR / "fold_artifacts" / f"xgb_mae_fold{fold}.json"))

            pred_va_xgb = xgb_model.predict(X_va_xgb)
            oof_xgb[va_idx] = pred_va_xgb.astype(np.float32)
            test_pred_xgb += xgb_model.predict(X_te_xgb).astype(np.float32) / N_FOLDS
            mae_fold_xgb = mean_absolute_error(y_va, pred_va_xgb)
            logger.info(f"[fold {fold}] XGB(MAE) fold MAE={mae_fold_xgb:.6f}")

            # ---- Train XGB SLOG (GPU) with same required pattern ----
            callbacks2 = [XGBProgressCallback(logger, period=200, prefix=f"XGB_SLOG_f{fold}")]
            with StageTimer(logger, f"fold_{fold}/train_xgb_slog"):
                xgb_model_slog = fit_xgb_with_guard(
                    params=xgb_params_slog,
                    fold_seed=SEED + fold,
                    callbacks=callbacks2,
                    X_tr=X_tr_xgb, y_tr=y_tr,
                    X_va=X_va_xgb, y_va=y_va,
                    logger=logger,
                )
                logger.info(f"[fold {fold}] XGB(SLOG) get_params device={xgb_model_slog.get_params().get('device')} objective={xgb_model_slog.get_params().get('objective')}")
                best_iter2 = getattr(xgb_model_slog, "best_iteration", None)
                logger.info(f"[fold {fold}] XGB(SLOG) best_iteration={best_iter2}")

                try:
                    cfg2 = xgb_model_slog.get_booster().save_config().lower()
                    logger.info(f"[fold {fold}] XGB(SLOG) booster config contains 'cuda': {'cuda' in cfg2}")
                    if fold == 0:
                        (MODEL_DIR / "xgb_booster_config_fold0_slog.json").write_text(xgb_model_slog.get_booster().save_config())
                except Exception:
                    pass

                xgb_model_slog.save_model(str(MODEL_DIR / "fold_artifacts" / f"xgb_slog_fold{fold}.json"))

            pred_va_xgb_slog = xgb_model_slog.predict(X_va_xgb)
            oof_xgb_slog[va_idx] = pred_va_xgb_slog.astype(np.float32)
            test_pred_xgb_slog += xgb_model_slog.predict(X_te_xgb).astype(np.float32) / N_FOLDS
            mae_fold_xgb_slog = mean_absolute_error(y_va, pred_va_xgb_slog)
            logger.info(f"[fold {fold}] XGB(SLOG) fold MAE={mae_fold_xgb_slog:.6f}")

            # ---- Train CatBoost (GPU, verbose=500) ----
            with StageTimer(logger, f"fold_{fold}/train_cat"):
                cat_model = CatBoostRegressor(**{**cat_params, "random_seed": SEED + fold})
                cat_model.fit(tr_pool, eval_set=va_pool, use_best_model=True)
                best_it_cat = cat_model.get_best_iteration()
                logger.info(f"[fold {fold}] CAT best_iteration={best_it_cat}")

                # GPU verification
                try:
                    allp = cat_model.get_all_params()
                    logger.info(f"[fold {fold}] CAT task_type={allp.get('task_type')} devices={allp.get('devices')}")
                except Exception:
                    pass

                cat_model.save_model(str(MODEL_DIR / "fold_artifacts" / f"cat_fold{fold}.cbm"))

            pred_va_cat = cat_model.predict(va_pool)
            oof_cat[va_idx] = np.array(pred_va_cat, dtype=np.float32)
            test_pred_cat += np.array(cat_model.predict(te_pool), dtype=np.float32) / N_FOLDS
            mae_fold_cat = mean_absolute_error(y_va, pred_va_cat)
            logger.info(f"[fold {fold}] CAT fold MAE={mae_fold_cat:.6f}")

            # save fold text/ohe for reproducibility
            import joblib
            joblib.dump(vec, MODEL_DIR / "fold_artifacts" / f"tfidf_fold{fold}.joblib")
            joblib.dump(svd, MODEL_DIR / "fold_artifacts" / f"svd_fold{fold}.joblib")
            joblib.dump(ohe, MODEL_DIR / "fold_artifacts" / f"ohe_fold{fold}.joblib")

    # ---- OOF summary + blend ----
    with StageTimer(logger, "oof_blend"):
        mae_oof_xgb = mean_absolute_error(y, oof_xgb)
        mae_oof_xgb_slog = mean_absolute_error(y, oof_xgb_slog)
        mae_oof_cat = mean_absolute_error(y, oof_cat)
        logger.info(f"[OOF] base MAE: XGB(MAE)={mae_oof_xgb:.6f} | XGB(SLOG)={mae_oof_xgb_slog:.6f} | CAT={mae_oof_cat:.6f}")

        best = find_best_3way_blend(oof_xgb, oof_xgb_slog, oof_cat, y, step=0.02)
        logger.info(f"[Blend] best={best}")

        oof_raw = best["w_a"] * oof_xgb + best["w_b"] * oof_xgb_slog + best["w_c"] * oof_cat
        oof_pred = np.clip(oof_raw - best["bias"], 0.0, None)
        logger.info(f"[Blend] OOF MAE (pre-cal)={mean_absolute_error(y, oof_pred):.6f}")

        # group median calibration (cheap win)
        cal = fit_group_median_calibration(oof_pred, y, train, logger=logger)
        (MODEL_DIR / "group_calibration.json").write_text(json.dumps(cal))
        oof_pred_cal = apply_group_median_calibration(oof_pred, train, cal)
        logger.info(f"[Blend+Cal] OOF MAE (post-cal)={mean_absolute_error(y, oof_pred_cal):.6f}")

        # forensic analysis (required)
        log_oof_error_analysis(logger, train, y, oof_pred_cal)

        # save OOF predictions
        oof_out = pd.DataFrame({
            "patient_id": train["patient_id"].astype(int).values,
            "y_true": y.astype(float),
            "oof_xgb_mae": oof_xgb.astype(float),
            "oof_xgb_slog": oof_xgb_slog.astype(float),
            "oof_cat": oof_cat.astype(float),
            "oof_blend_cal": oof_pred_cal.astype(float),
        })
        oof_out.to_csv(MODEL_DIR / "oof_predictions.csv", index=False)

        # apply blend+cal to test
        test_raw = best["w_a"] * test_pred_xgb + best["w_b"] * test_pred_xgb_slog + best["w_c"] * test_pred_cat
        test_pred = np.clip(test_raw - best["bias"], 0.0, None)
        test_pred = apply_group_median_calibration(test_pred, test, cal)

    # ---- write submission (strict test order) ----
    with StageTimer(logger, "write_submission"):
        # test already reindexed to test_order_ids after merge
        sub = pd.DataFrame({
            "patient_id": test["patient_id"].astype(int).values,
            "ed_cost_next3y_usd": test_pred.astype(float),
        })
        # ensure order matches original test file order
        assert np.array_equal(sub["patient_id"].values, test_order_ids), "Test order mismatch — submission would be invalid."
        sub.to_csv(SUBMISSION_PATH, index=False)
        logger.info(f"Submission written: {SUBMISSION_PATH} | rows={len(sub)}")
        logger.info("Submission head:\n" + str(sub.head()))

    # ---- save run config ----
    config = {
        "PDF_CACHE_VERSION": PDF_CACHE_VERSION,
        "FORCE_PDF_REEXTRACT": FORCE_PDF_REEXTRACT,
        "N_FOLDS": N_FOLDS,
        "SEED": SEED,
        "TOP_CODE_K": TOP_CODE_K,
        "TOP_CODE_MIN_TOTAL_COUNT": TOP_CODE_MIN_TOTAL_COUNT,
        "TFIDF_MAX_FEATURES": TFIDF_MAX_FEATURES,
        "SVD_COMPONENTS": SVD_COMPONENTS,
        "XGB_N_ESTIMATORS": XGB_N_ESTIMATORS,
        "XGB_EARLY_STOPPING": XGB_EARLY_STOPPING,
        "CAT_ITERATIONS": CAT_ITERATIONS,
        "CAT_OD_WAIT": CAT_OD_WAIT,
        "xgb_params": xgb_params,
        "xgb_params_slog": xgb_params_slog,
        "cat_params": cat_params,
    }
    (MODEL_DIR / "run_config.json").write_text(json.dumps(config, indent=2))
    print(f"DONE ✅ Logs: {MODEL_DIR / 'train_log.txt'}")
    print(f"Submission: {SUBMISSION_PATH}")

# -----------------------------
# RUN
# -----------------------------
RUN_ALL = True  # set False to just define functions

if RUN_ALL:
    run_pipeline()


2026-02-13 12:22:32 | INFO | ed_cost_iter | [env_report] START
2026-02-13 12:22:32 | INFO | ed_cost_iter | OS: Windows-11-10.0.26100-SP0
2026-02-13 12:22:32 | INFO | ed_cost_iter | Python: 3.12.0
2026-02-13 12:22:32 | INFO | ed_cost_iter | numpy=2.3.4 pandas=2.3.3
2026-02-13 12:22:32 | INFO | ed_cost_iter | sklearn=1.7.2
2026-02-13 12:22:32 | INFO | ed_cost_iter | xgboost=3.0.0
2026-02-13 12:22:32 | INFO | ed_cost_iter | catboost=1.2.8
2026-02-13 12:22:32 | INFO | ed_cost_iter | torch=2.9.0+cu126 cuda_available=True
2026-02-13 12:22:32 | INFO | ed_cost_iter | torch.cuda.device_count=1
2026-02-13 12:22:32 | INFO | ed_cost_iter | GPU[0]=NVIDIA GeForce RTX 4060 Laptop GPU
2026-02-13 12:22:32 | INFO | ed_cost_iter | GPU[0] capability=(8, 9)
2026-02-13 12:22:32 | INFO | ed_cost_iter | nvidia-smi -L:
GPU 0: NVIDIA GeForce RTX 4060 Laptop GPU (UUID: GPU-9515bdbf-1472-4fa2-9f73-28e1d27eedbf)
2026-02-13 12:22:32 | INFO | ed_cost_iter | [env_report] END (0.06s)
2026-02-13 12:22:32 | INFO | ed_co

0:	learn: 1438.3384375	test: 1414.0312500	best: 1414.0312500 (0)	total: 75.9ms	remaining: 15m 10s
500:	learn: 1433.3215625	test: 1409.2484375	best: 1409.2484375 (500)	total: 40.2s	remaining: 15m 21s
1000:	learn: 1428.2428125	test: 1404.4262500	best: 1404.4262500 (1000)	total: 1m 24s	remaining: 15m 29s
1500:	learn: 1423.1718750	test: 1399.6021875	best: 1399.6021875 (1500)	total: 2m 8s	remaining: 14m 59s
2000:	learn: 1418.1078125	test: 1394.7887500	best: 1394.7887500 (2000)	total: 2m 50s	remaining: 14m 11s
2500:	learn: 1413.0503125	test: 1389.9990625	best: 1389.9990625 (2500)	total: 3m 34s	remaining: 13m 33s
3000:	learn: 1408.0020313	test: 1385.2325000	best: 1385.2325000 (3000)	total: 4m 18s	remaining: 12m 54s
3500:	learn: 1402.9887500	test: 1380.4906250	best: 1380.4906250 (3500)	total: 5m 1s	remaining: 12m 12s
4000:	learn: 1397.9970312	test: 1375.7642188	best: 1375.7642188 (4000)	total: 5m 44s	remaining: 11m 27s
4500:	learn: 1393.0203125	test: 1371.0296875	best: 1371.0296875 (4500)	tota

2026-02-13 12:39:42 | INFO | ed_cost_iter | [fold 0] CAT best_iteration=11999
2026-02-13 12:39:42 | INFO | ed_cost_iter | [fold 0] CAT task_type=GPU devices=0
2026-02-13 12:39:42 | INFO | ed_cost_iter | [fold_0/train_cat] END (1021.20s)
2026-02-13 12:39:42 | INFO | ed_cost_iter | [fold 0] CAT fold MAE=1301.063653
2026-02-13 12:39:42 | INFO | ed_cost_iter | [fold_0] END (1029.68s)
2026-02-13 12:39:42 | INFO | ed_cost_iter | [fold_1] START
2026-02-13 12:39:42 | INFO | ed_cost_iter | [fold_1/text_fit] START
2026-02-13 12:39:43 | INFO | ed_cost_iter | [fold 1] text svd_dim=256
2026-02-13 12:39:43 | INFO | ed_cost_iter | [fold_1/text_fit] END (0.81s)
2026-02-13 12:39:43 | INFO | ed_cost_iter | [fold_1/text_transform] START
2026-02-13 12:39:43 | INFO | ed_cost_iter | [fold_1/text_transform] END (0.15s)
2026-02-13 12:39:43 | INFO | ed_cost_iter | [fold_1/build_cat_matrices] START
2026-02-13 12:39:43 | INFO | ed_cost_iter | [fold 1] CatBoost X shapes tr=(1600, 333) va=(400, 333) te=(2000, 333)

0:	learn: 1425.2675000	test: 1465.2100000	best: 1465.2100000 (0)	total: 373ms	remaining: 1h 14m 32s
500:	learn: 1420.2948437	test: 1460.2714062	best: 1460.2714062 (500)	total: 42.3s	remaining: 16m 11s
1000:	learn: 1415.2515625	test: 1455.3325000	best: 1455.3325000 (1000)	total: 1m 24s	remaining: 15m 31s
1500:	learn: 1410.2425000	test: 1450.4181250	best: 1450.4181250 (1500)	total: 2m 8s	remaining: 14m 56s
2000:	learn: 1405.2396875	test: 1445.5125000	best: 1445.5125000 (2000)	total: 2m 50s	remaining: 14m 12s
2500:	learn: 1400.2462500	test: 1440.6117187	best: 1440.6117187 (2500)	total: 3m 32s	remaining: 13m 27s
3000:	learn: 1395.2712500	test: 1435.7431250	best: 1435.7431250 (3000)	total: 4m 19s	remaining: 12m 57s
3500:	learn: 1390.3148437	test: 1430.8781250	best: 1430.8781250 (3500)	total: 5m 2s	remaining: 12m 14s
4000:	learn: 1385.3790625	test: 1426.0156250	best: 1426.0156250 (4000)	total: 5m 46s	remaining: 11m 31s
4500:	learn: 1380.4675000	test: 1421.1957812	best: 1421.1957812 (4500)	to

2026-02-13 12:57:16 | INFO | ed_cost_iter | [fold 1] CAT best_iteration=11999
2026-02-13 12:57:16 | INFO | ed_cost_iter | [fold 1] CAT task_type=GPU devices=0
2026-02-13 12:57:16 | INFO | ed_cost_iter | [fold_1/train_cat] END (1029.51s)
2026-02-13 12:57:16 | INFO | ed_cost_iter | [fold 1] CAT fold MAE=1350.129929
2026-02-13 12:57:16 | INFO | ed_cost_iter | [fold_1] END (1053.59s)
2026-02-13 12:57:16 | INFO | ed_cost_iter | [fold_2] START
2026-02-13 12:57:16 | INFO | ed_cost_iter | [fold_2/text_fit] START
2026-02-13 12:57:17 | INFO | ed_cost_iter | [fold 2] text svd_dim=256
2026-02-13 12:57:17 | INFO | ed_cost_iter | [fold_2/text_fit] END (0.82s)
2026-02-13 12:57:17 | INFO | ed_cost_iter | [fold_2/text_transform] START
2026-02-13 12:57:17 | INFO | ed_cost_iter | [fold_2/text_transform] END (0.16s)
2026-02-13 12:57:17 | INFO | ed_cost_iter | [fold_2/build_cat_matrices] START
2026-02-13 12:57:17 | INFO | ed_cost_iter | [fold 2] CatBoost X shapes tr=(1600, 333) va=(400, 333) te=(2000, 333)

...

0:	learn: 1413.9862500	test: 1510.3365625	best: 1510.3365625 (0)	total: 92ms	remaining: 18m 23s
500:	learn: 1408.9743750	test: 1505.4109375	best: 1505.4109375 (500)	total: 42.9s	remaining: 16m 24s
1000:	learn: 1403.8739062	test: 1500.4565625	best: 1500.4565625 (1000)	total: 1m 25s	remaining: 15m 42s
1500:	learn: 1398.7948437	test: 1495.5357812	best: 1495.5357812 (1500)	total: 2m 8s	remaining: 15m 1s
2000:	learn: 1393.7212500	test: 1490.6371875	best: 1490.6371875 (2000)	total: 2m 51s	remaining: 14m 17s
2500:	learn: 1388.6803125	test: 1485.7646875	best: 1485.7646875 (2500)	total: 3m 34s	remaining: 13m 35s
3000:	learn: 1383.6575000	test: 1480.9231250	best: 1480.9231250 (3000)	total: 4m 17s	remaining: 12m 52s
3500:	learn: 1378.6518750	test: 1476.1000000	best: 1476.1000000 (3500)	total: 4m 59s	remaining: 12m 7s
4000:	learn: 1373.6685937	test: 1471.2804687	best: 1471.2804687 (4000)	total: 5m 41s	remaining: 11m 22s
4500:	learn: 1368.7185938	test: 1466.4732812	best: 1466.4732812 (4500)	total: 

2026-02-13 13:31:23 | INFO | ed_cost_iter | [fold 3] CAT best_iteration=11999
2026-02-13 13:31:23 | INFO | ed_cost_iter | [fold 3] CAT task_type=GPU devices=0
2026-02-13 13:31:23 | INFO | ed_cost_iter | [fold_3/train_cat] END (1007.70s)
2026-02-13 13:31:24 | INFO | ed_cost_iter | [fold 3] CAT fold MAE=1395.728286
2026-02-13 13:31:24 | INFO | ed_cost_iter | [fold_3] END (1018.06s)
2026-02-13 13:31:24 | INFO | ed_cost_iter | [fold_4] START
2026-02-13 13:31:24 | INFO | ed_cost_iter | [fold_4/text_fit] START
2026-02-13 13:31:24 | INFO | ed_cost_iter | [fold 4] text svd_dim=256
2026-02-13 13:31:24 | INFO | ed_cost_iter | [fold_4/text_fit] END (0.71s)
2026-02-13 13:31:24 | INFO | ed_cost_iter | [fold_4/text_transform] START
2026-02-13 13:31:24 | INFO | ed_cost_iter | [fold_4/text_transform] END (0.16s)
2026-02-13 13:31:24 | INFO | ed_cost_iter | [fold_4/build_cat_matrices] START
2026-02-13 13:31:25 | INFO | ed_cost_iter | [fold 4] CatBoost X shapes tr=(1600, 333) va=(400, 333) te=(2000, 333)

0:	learn: 1456.0318750	test: 1342.0618750	best: 1342.0618750 (0)	total: 81.1ms	remaining: 16m 13s
500:	learn: 1450.9296875	test: 1337.3107812	best: 1337.3107812 (500)	total: 42.7s	remaining: 16m 19s
1000:	learn: 1445.7871875	test: 1332.5162500	best: 1332.5162500 (1000)	total: 1m 25s	remaining: 15m 36s
1500:	learn: 1440.6746875	test: 1327.7825000	best: 1327.7825000 (1500)	total: 2m 7s	remaining: 14m 53s
2000:	learn: 1435.5782812	test: 1323.0625000	best: 1323.0625000 (2000)	total: 2m 49s	remaining: 14m 7s
2500:	learn: 1430.5025000	test: 1318.3635938	best: 1318.3635938 (2500)	total: 3m 31s	remaining: 13m 24s
3000:	learn: 1425.4367187	test: 1313.6543750	best: 1313.6543750 (3000)	total: 4m 13s	remaining: 12m 41s
3500:	learn: 1420.3784375	test: 1308.9342187	best: 1308.9342187 (3500)	total: 4m 56s	remaining: 11m 59s
4000:	learn: 1415.3371875	test: 1304.2319531	best: 1304.2319531 (4000)	total: 5m 38s	remaining: 11m 17s
4500:	learn: 1410.3289063	test: 1299.5501563	best: 1299.5501563 (4500)	tota

2026-02-13 13:48:05 | INFO | ed_cost_iter | [fold 4] CAT best_iteration=11999
2026-02-13 13:48:05 | INFO | ed_cost_iter | [fold 4] CAT task_type=GPU devices=0
2026-02-13 13:48:05 | INFO | ed_cost_iter | [fold_4/train_cat] END (993.50s)
2026-02-13 13:48:05 | INFO | ed_cost_iter | [fold 4] CAT fold MAE=1231.010907
2026-02-13 13:48:05 | INFO | ed_cost_iter | [fold_4] END (1001.58s)
2026-02-13 13:48:05 | INFO | ed_cost_iter | [oof_blend] START
2026-02-13 13:48:05 | INFO | ed_cost_iter | [OOF] base MAE: XGB(MAE)=479.118378 | XGB(SLOG)=1457.528809 | CAT=1321.686157
2026-02-13 13:48:06 | INFO | ed_cost_iter | [Blend] best={'w_a': 1.0, 'w_b': 0.0, 'w_c': 0.0, 'bias': -1.429931640625, 'mae': 479.11796842956545}
2026-02-13 13:48:06 | INFO | ed_cost_iter | [Blend] OOF MAE (pre-cal)=479.118011
2026-02-13 13:48:06 | INFO | ed_cost_iter | [Calib] ins_col=insurance | level1_groups=9 | level2_groups=3
2026-02-13 13:48:06 | INFO | ed_cost_iter | [Blend+Cal] OOF MAE (post-cal)=473.228972
2026-02-13 13:4

DONE ✅ Logs: D:\AgentDs\agent_ds_healthcare\models_gpu_iter_v2\train_log.txt
Submission: D:\AgentDs\agent_ds_healthcare\submission_ICHI_V1.csv


# Iteration 3
- 492.1075

In [20]:
# ============================================================
# ONE-CELL PIPELINE: Postmortem + Ablations + Sanity Holdout
#   - GPU XGBoost (device=cuda:0 + tree_method=hist)
#   - PDF PyMuPDF cache (no OCR) with code dicts
#   - 5-fold CV (no repeats) + Zip3-group holdout
#   - Cross-fitted + shrinkage calibration (optional)
#   - 6 ablation toggles supported
#
# IMPORTANT: CatBoost + XGB(SLOG) default OFF due to train_log evidence.
# ============================================================

import os
import re
import json
import math
import time
import random
import logging
import platform
import subprocess
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional, List, Tuple, Dict, Any
from collections import defaultdict

import numpy as np
import pandas as pd

# PDF
import fitz  # PyMuPDF

# ML
from joblib import Parallel, delayed
from scipy import sparse

from sklearn.model_selection import StratifiedKFold, GroupShuffleSplit, StratifiedShuffleSplit
from sklearn.metrics import mean_absolute_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import OneHotEncoder

import xgboost as xgb

# Optional (kept but default off)
try:
    from catboost import CatBoostRegressor, Pool
    _HAS_CATBOOST = True
except Exception:
    _HAS_CATBOOST = False

# GPU visibility (for logging)
try:
    import torch
    _HAS_TORCH = True
except Exception:
    _HAS_TORCH = False


# -----------------------------
# USER PATHS (Windows)
# -----------------------------
DATA_DIR = Path(r"D:\AgentDs\agent_ds_healthcare")
TRAIN_CSV = DATA_DIR / "ed_cost_train.csv"
TEST_CSV  = DATA_DIR / "ed_cost_test.csv"
PATIENTS_CSV = DATA_DIR / "patients.csv"   # join for age/sex/insurance/zip3

PDF_DIR = DATA_DIR / "receipts_pdf"
CACHE_DIR = DATA_DIR / "cache"
RUN_DIR = DATA_DIR / "runs_postmortem_onecell"   # logs/configs/OOF dumps
SUBMISSION_PATH = DATA_DIR / "submission_ICHI_V1.csv"


# -----------------------------
# GLOBAL RUNTIME CONTROL
# -----------------------------
SEED = 2026
N_FOLDS = 5   # hard requirement: max 5, no repeats
ASSERT_GPU_REQUIRED = True   # set False if you want to allow CPU fallback (not recommended)

PDF_CACHE_VERSION = "v2"
FORCE_PDF_REEXTRACT = False
PDF_N_JOBS = 8  # threads

# Text features
TFIDF_MAX_FEATURES = 6000
SVD_COMPONENTS = 256

# Top-code pivots
TOP_CODE_K = 200
TOP_CODE_MIN_TOTAL_COUNT = 8

# XGB params (keep your pattern)
XGB_N_ESTIMATORS = 8000
XGB_EARLY_STOPPING = 400

# CatBoost params (optional; default off; note: your log showed it's terrible)
CAT_ITERATIONS = 12000
CAT_OD_WAIT = 400

# Sanity holdout settings
HOLDOUT_TEST_SIZE = 0.20
HOLDOUT_CV_FOLDS = 3  # run small CV on train-in only to evaluate shift

# Dev speed toggle
FAST_DEV_RUN = False
if FAST_DEV_RUN:
    N_FOLDS = 2
    HOLDOUT_CV_FOLDS = 2
    XGB_N_ESTIMATORS = 1500
    XGB_EARLY_STOPPING = 150
    TFIDF_MAX_FEATURES = 2000
    SVD_COMPONENTS = 64
    TOP_CODE_K = 60
    TOP_CODE_MIN_TOTAL_COUNT = 3
    PDF_N_JOBS = 4


# -----------------------------
# LOGGING + TIMERS
# -----------------------------
def setup_logger(log_path: Path, level=logging.INFO) -> logging.Logger:
    logger = logging.getLogger(f"ed_cost_onecell_{int(time.time())}")
    logger.setLevel(level)
    logger.handlers.clear()
    logger.propagate = False

    fmt = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    logger.addHandler(sh)

    log_path.parent.mkdir(parents=True, exist_ok=True)
    fh = logging.FileHandler(log_path, encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(fh)
    return logger


class StageTimer:
    def __init__(self, logger: logging.Logger, name: str):
        self.logger = logger
        self.name = name
        self.t0 = None

    def __enter__(self):
        self.t0 = time.perf_counter()
        self.logger.info(f"[{self.name}] START")
        return self

    def __exit__(self, exc_type, exc, tb):
        dt = time.perf_counter() - self.t0
        if exc is None:
            self.logger.info(f"[{self.name}] END ({dt:.2f}s)")
        else:
            self.logger.exception(f"[{self.name}] FAILED after {dt:.2f}s: {exc}")


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


def try_run_cmd(cmd: List[str]) -> str:
    try:
        out = subprocess.check_output(cmd, stderr=subprocess.STDOUT, text=True)
        return out.strip()
    except Exception as e:
        return f"<command failed: {e}>"


def parquet_available() -> bool:
    try:
        import pyarrow  # noqa: F401
        return True
    except Exception:
        try:
            import fastparquet  # noqa: F401
            return True
        except Exception:
            return False


# -----------------------------
# XGBoost progress callback (constructor callback)
# -----------------------------
try:
    from xgboost.callback import TrainingCallback
except Exception:
    TrainingCallback = object

class XGBProgressCallback(TrainingCallback):
    def __init__(self, logger: logging.Logger, period: int = 200, prefix: str = "XGB"):
        self.logger = logger
        self.period = int(period)
        self.prefix = prefix

    def after_iteration(self, model, epoch: int, evals_log: Dict[str, Any]) -> bool:
        if (epoch + 1) % self.period == 0:
            try:
                parts = []
                for ds, mets in (evals_log or {}).items():
                    for mname, hist in mets.items():
                        if hist:
                            parts.append(f"{ds}-{mname}={hist[-1]:.6f}")
                msg = " ".join(parts) if parts else "(no eval log)"
                self.logger.info(f"[{self.prefix}] iter={epoch+1} {msg}")
            except Exception:
                self.logger.info(f"[{self.prefix}] iter={epoch+1} (log parse fail)")
        return False


def fit_xgb_with_guard(
    params: Dict[str, Any],
    fold_seed: int,
    callbacks: List[Any],
    X_tr,
    y_tr,
    X_va,
    y_va,
    logger: logging.Logger,
) -> xgb.XGBRegressor:
    """
    DEFAULT (preserve your fixed bug pattern):
      1) early_stopping_rounds in constructor params
      2) callbacks in constructor
      3) fit(): eval_set + verbose=False only
    FALLBACK: if TypeError, try moving callbacks/early stopping to fit().
    """
    # 1) default
    try:
        model = xgb.XGBRegressor(
            **{**params, "random_state": fold_seed},
            callbacks=callbacks,
        )
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        return model
    except TypeError as e:
        logger.warning(f"[XGB guard] default constructor pattern TypeError: {e}")

    # 2) fallback A: callbacks in fit
    try:
        model = xgb.XGBRegressor(**{**params, "random_state": fold_seed})
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False, callbacks=callbacks)
        return model
    except TypeError as e:
        logger.warning(f"[XGB guard] fit(callbacks=...) TypeError: {e}")

    # 3) fallback B: move early stopping to fit
    esr = params.get("early_stopping_rounds", None)
    params2 = dict(params)
    params2.pop("early_stopping_rounds", None)
    model = xgb.XGBRegressor(**{**params2, "random_state": fold_seed})
    try:
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False,
                  early_stopping_rounds=esr, callbacks=callbacks)
        return model
    except TypeError as e:
        logger.error(f"[XGB guard] early stopping fallback failed: {e}")
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        return model


# -----------------------------
# PDF parsing (PyMuPDF, no OCR)
# -----------------------------
_MONEY_RE = re.compile(r"^\$?[\d,]+(?:\.\d{2})?$")
_CODE_RE  = re.compile(r"^[A-Z0-9]{5}$")
_LINE_PAT = re.compile(r"^([A-Z0-9]{5})\s+(.+?)\s+(\d+)\s+([\d,]+\.\d{2})\s+([\d,]+\.\d{2})$")

def _parse_money(s: str) -> float:
    s = str(s).strip().replace("$", "").replace(",", "")
    try:
        return float(s)
    except Exception:
        return float("nan")


@dataclass(frozen=True)
class PdfParsed:
    patient_id_pdf: Optional[int]
    zip3_pdf: Optional[str]
    insurance_pdf: Optional[str]
    total_pdf: float
    items: List[Tuple[str, str, int, float, float]]  # code, desc, qty, unit, line_total
    raw_text: str


def parse_receipt_pdf(pdf_path: Path) -> Optional[PdfParsed]:
    if not pdf_path.exists():
        return None
    try:
        doc = fitz.open(pdf_path)
        raw_text = "".join([p.get_text("text", sort=True) for p in doc])
        doc.close()
    except Exception:
        return None

    lines = [ln.strip() for ln in raw_text.splitlines() if ln.strip()]

    pid = None
    zip3 = None
    ins = None
    for ln in lines[:12]:
        m = re.search(r"Patient ID:\s*(\d+)", ln)
        if m: pid = int(m.group(1))
        m = re.search(r"ZIP3:\s*(\d{3})", ln)
        if m: zip3 = m.group(1)
        m = re.search(r"Insurance:\s*([A-Za-z_]+)", ln)
        if m: ins = m.group(1)

    total = float("nan")
    for i, ln in enumerate(lines):
        if ln.startswith("TOTAL"):
            parts = ln.split()
            if len(parts) >= 2 and _MONEY_RE.match(parts[1]):
                total = _parse_money(parts[1])
            elif i + 1 < len(lines) and _MONEY_RE.match(lines[i + 1]):
                total = _parse_money(lines[i + 1])
            break

    items: List[Tuple[str, str, int, float, float]] = []

    # layout A
    for ln in lines:
        m = _LINE_PAT.match(ln)
        if m:
            code, desc, qty, unit, lt = m.groups()
            items.append((code, desc, int(qty), _parse_money(unit), _parse_money(lt)))

    # layout B
    if not items:
        start = 0
        for i, ln in enumerate(lines):
            if ln.lower().startswith("line total") or ln.lower().startswith("line"):
                start = i + 1
                break

        i = start
        while i < len(lines):
            ln = lines[i]
            if ln.startswith("TOTAL"):
                break

            if _CODE_RE.match(ln):
                code = ln
                j = i + 1

                desc_parts = []
                while j < len(lines) and not re.fullmatch(r"\d+", lines[j]):
                    if lines[j].startswith("TOTAL"):
                        break
                    desc_parts.append(lines[j])
                    j += 1

                if j >= len(lines) or lines[j].startswith("TOTAL"):
                    break
                qty = int(lines[j]); j += 1
                if j + 1 >= len(lines):
                    break
                unit = _parse_money(lines[j])
                lt = _parse_money(lines[j + 1])
                desc = " ".join(desc_parts).strip()

                items.append((code, desc, qty, unit, lt))
                i = j + 2
            else:
                i += 1

    return PdfParsed(pid, zip3, ins, total, items, raw_text)


def _gini(x: np.ndarray) -> float:
    if x.size == 0:
        return 0.0
    s = np.sort(np.maximum(x, 0.0))
    tot = s.sum()
    if tot <= 0:
        return 0.0
    n = s.size
    return float((2*np.arange(1, n+1) - n - 1).dot(s) / (n * tot + 1e-12))


def compute_pdf_features(parsed: Optional[PdfParsed]) -> Dict[str, Any]:
    if parsed is None:
        return {
            "zip3_pdf": "UNK",
            "insurance_pdf": "UNK",
            "pdf_total": float("nan"),
            "pdf_n_items": 0,
            "pdf_n_unique_codes": 0,
            "pdf_sum_qty": 0.0,
            "pdf_mean_line_total": 0.0,
            "pdf_max_line_total": 0.0,
            "pdf_std_line_total": 0.0,
            "pdf_share_top1": 0.0,
            "pdf_cost_eval_mgmt": 0.0,
            "pdf_cost_lab": 0.0,
            "pdf_cost_radiology": 0.0,
            "pdf_cost_surgery": 0.0,
            "pdf_cost_other_numeric": 0.0,
            "pdf_cost_hcpcs_alpha": 0.0,
            "pdf_em_level_max": 0,
            "pdf_em_count": 0,
            "pdf_critical_count": 0,
            "pdf_code_counts_json": "{}",
            "pdf_code_costs_json": "{}",
            "pdf_cost_entropy": 0.0,
            "pdf_cost_gini": 0.0,
            "pdf_text": "",
            "pdf_missing": 1,
            "pdf_empty_items": 1,
        }

    items = parsed.items
    total = parsed.total_pdf

    codes = [it[0] for it in items]
    descs = [it[1] for it in items]
    qtys = np.array([it[2] for it in items], dtype=np.float32) if items else np.array([], dtype=np.float32)
    lts  = np.array([it[4] for it in items], dtype=np.float32) if items else np.array([], dtype=np.float32)

    n = len(items)
    n_unique = len(set(codes)) if n else 0

    mean_lt = float(lts.mean()) if n else 0.0
    max_lt  = float(lts.max()) if n else 0.0
    std_lt  = float(lts.std()) if n else 0.0
    share_top1 = float(max_lt / total) if n and (not math.isnan(total)) and total > 0 else 0.0

    eval_mgmt = lab = radiology = surgery = other_numeric = hcpcs_alpha = 0.0
    em_level_max = 0
    em_count = 0
    critical_count = 0

    code_counts = defaultdict(int)
    code_costs  = defaultdict(float)

    for code, _desc, qty, _unit, lt in items:
        if code:
            code_counts[code] += int(qty)
            code_costs[code]  += float(lt)

        if code and code[0].isdigit():
            ci = int(code)
            if 90000 <= ci <= 99999:
                eval_mgmt += lt
            elif 80000 <= ci <= 89999:
                lab += lt
            elif 70000 <= ci <= 79999:
                radiology += lt
            elif 10000 <= ci <= 69999:
                surgery += lt
            else:
                other_numeric += lt

            if 99281 <= ci <= 99285:
                em_count += 1
                em_level_max = max(em_level_max, ci - 99280)
            if ci in (99291, 99292):
                critical_count += 1
        else:
            hcpcs_alpha += lt

    cost_vals = np.array(list(code_costs.values()), dtype=np.float64)
    if cost_vals.size > 0 and cost_vals.sum() > 0:
        p = cost_vals / cost_vals.sum()
        cost_entropy = float(-(p * np.log(p + 1e-12)).sum())
        cost_gini = _gini(cost_vals)
    else:
        cost_entropy = 0.0
        cost_gini = 0.0

    pdf_text = " ".join([f"{c} {d}" for c, d in zip(codes, descs)])

    return {
        "zip3_pdf": parsed.zip3_pdf or "UNK",
        "insurance_pdf": parsed.insurance_pdf or "UNK",
        "pdf_total": total,
        "pdf_n_items": int(n),
        "pdf_n_unique_codes": int(n_unique),
        "pdf_sum_qty": float(qtys.sum()) if n else 0.0,
        "pdf_mean_line_total": mean_lt,
        "pdf_max_line_total": max_lt,
        "pdf_std_line_total": std_lt,
        "pdf_share_top1": share_top1,
        "pdf_cost_eval_mgmt": float(eval_mgmt),
        "pdf_cost_lab": float(lab),
        "pdf_cost_radiology": float(radiology),
        "pdf_cost_surgery": float(surgery),
        "pdf_cost_other_numeric": float(other_numeric),
        "pdf_cost_hcpcs_alpha": float(hcpcs_alpha),
        "pdf_em_level_max": int(em_level_max),
        "pdf_em_count": int(em_count),
        "pdf_critical_count": int(critical_count),
        "pdf_code_counts_json": json.dumps(code_counts),
        "pdf_code_costs_json": json.dumps(code_costs),
        "pdf_cost_entropy": float(cost_entropy),
        "pdf_cost_gini": float(cost_gini),
        "pdf_text": pdf_text,
        "pdf_missing": 0,
        "pdf_empty_items": 1 if n == 0 else 0,
    }


def load_or_extract_pdf_features(logger: logging.Logger, all_ids: List[int]) -> pd.DataFrame:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    use_parquet = parquet_available()
    cache_path = CACHE_DIR / (f"pdf_features_all_{PDF_CACHE_VERSION}.parquet" if use_parquet else f"pdf_features_all_{PDF_CACHE_VERSION}.pkl")

    # audit
    found = 0
    for pid in all_ids:
        if (PDF_DIR / f"receipt_{pid}.pdf").exists():
            found += 1
    logger.info(f"[PDF] Expected PDFs={len(all_ids)} | Found={found} | Missing={len(all_ids)-found}")

    required = {
        "patient_id","pdf_missing","pdf_empty_items","pdf_text",
        "pdf_code_counts_json","pdf_code_costs_json",
        "zip3_pdf","insurance_pdf",
    }

    if cache_path.exists() and not FORCE_PDF_REEXTRACT:
        with StageTimer(logger, "pdf_cache_load"):
            df = pd.read_parquet(cache_path) if cache_path.suffix == ".parquet" else pd.read_pickle(cache_path)
        missing_cols = required - set(df.columns)
        if missing_cols:
            raise RuntimeError(f"PDF cache {cache_path} missing columns {sorted(missing_cols)}. Set FORCE_PDF_REEXTRACT=True or bump PDF_CACHE_VERSION.")
        logger.info(f"[PDF] Using cached features: {cache_path}")
        return df

    logger.info(f"[PDF] Extracting PDFs via PyMuPDF -> {cache_path.name}")

    def extract_one(pid: int) -> Dict[str, Any]:
        parsed = parse_receipt_pdf(PDF_DIR / f"receipt_{pid}.pdf")
        feats = compute_pdf_features(parsed)
        feats["patient_id"] = int(pid)
        return feats

    with StageTimer(logger, "pdf_extract_total"):
        rows = Parallel(n_jobs=PDF_N_JOBS, prefer="threads")(delayed(extract_one)(pid) for pid in all_ids)

    df = pd.DataFrame(rows).sort_values("patient_id").reset_index(drop=True)
    logger.info(f"[PDF] parsed rows={len(df)} | pdf_missing={int(df['pdf_missing'].sum())} | pdf_empty_items={int(df['pdf_empty_items'].sum())}")

    with StageTimer(logger, "pdf_cache_write"):
        if cache_path.suffix == ".parquet":
            df.to_parquet(cache_path, index=False)
        else:
            df.to_pickle(cache_path)
    logger.info(f"[PDF] Cache written: {cache_path}")
    return df


# -----------------------------
# FEATURE ENGINEERING
# -----------------------------
def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # categorical normalization
    for c in ["primary_chronic", "sex", "insurance", "zip3", "zip3_pdf", "insurance_pdf"]:
        if c in out.columns:
            out[c] = out[c].fillna("UNK").astype(str)

    # numeric fill
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].fillna(0.0)

    # pdf_present (for segments + strat)
    if "pdf_missing" in out.columns:
        out["pdf_present"] = (out["pdf_missing"].astype(int) == 0).astype(int)
    else:
        out["pdf_present"] = 0

    # utilization
    out["prior_cost_log1p"] = np.log1p(out["prior_ed_cost_5y_usd"].astype(float))
    out["prior_visits_log1p"] = np.log1p(out["prior_ed_visits_5y"].astype(float))
    out["prior_cost_per_visit"] = out["prior_ed_cost_5y_usd"] / (out["prior_ed_visits_5y"] + 1.0)
    out["prior_cost_per_visit_log1p"] = np.log1p(out["prior_cost_per_visit"].astype(float))
    out["prior_cost_x_visits"] = out["prior_ed_cost_5y_usd"] * out["prior_ed_visits_5y"]

    # shares normalized by prior cost (avoid pdf_total leakage-ish duplication)
    denom = out["prior_ed_cost_5y_usd"].astype(float) + 1.0
    for c in ["pdf_cost_eval_mgmt","pdf_cost_lab","pdf_cost_radiology","pdf_cost_surgery","pdf_cost_other_numeric","pdf_cost_hcpcs_alpha"]:
        if c in out.columns:
            out[f"{c}_share"] = out[c].astype(float) / denom

    out["pdf_items_per_visit"] = out.get("pdf_n_items", 0).astype(float) / (out["prior_ed_visits_5y"] + 1.0)

    # final numeric fill
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].fillna(0.0)
    return out


def add_top_code_features(logger: logging.Logger, train_df: pd.DataFrame, test_df: pd.DataFrame,
                          top_k: int, min_total_count: int) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    def parse_dict(s: Any) -> Dict[str, Any]:
        if not isinstance(s, str) or not s:
            return {}
        try:
            return json.loads(s)
        except Exception:
            return {}

    freq: Dict[str, int] = {}
    for s in train_df["pdf_code_counts_json"].fillna("{}").values:
        d = parse_dict(s)
        for k, v in d.items():
            freq[k] = freq.get(k, 0) + int(v)

    items = [(k, v) for k, v in freq.items() if v >= min_total_count]
    items.sort(key=lambda x: x[1], reverse=True)
    top_codes = [k for k, _ in items[:top_k]]
    code_to_idx = {c: i for i, c in enumerate(top_codes)}

    logger.info(f"[TopCodes] selected={len(top_codes)} (top_k={top_k}, min_total_count={min_total_count})")
    logger.info(f"[TopCodes] examples={top_codes[:20]}")

    def build(df: pd.DataFrame) -> pd.DataFrame:
        n = len(df); K = len(top_codes)
        cnt = np.zeros((n, K), dtype=np.float32)
        shr = np.zeros((n, K), dtype=np.float32)
        denom = df["prior_ed_cost_5y_usd"].astype(np.float32).values + 1.0

        for i, s in enumerate(df["pdf_code_counts_json"].fillna("{}").values):
            d = parse_dict(s)
            for code, v in d.items():
                j = code_to_idx.get(code)
                if j is not None:
                    cnt[i, j] = float(v)

        for i, s in enumerate(df["pdf_code_costs_json"].fillna("{}").values):
            d = parse_dict(s)
            for code, v in d.items():
                j = code_to_idx.get(code)
                if j is not None:
                    shr[i, j] = float(v) / denom[i]

        cols_cnt = [f"code_cnt_{c}" for c in top_codes]
        cols_shr = [f"code_cost_share_{c}" for c in top_codes]
        return pd.DataFrame(np.hstack([cnt, shr]), columns=cols_cnt + cols_shr)

    with StageTimer(logger, "topcodes_build"):
        tr_mat = build(train_df)
        te_mat = build(test_df)

    train2 = pd.concat([train_df.reset_index(drop=True), tr_mat], axis=1)
    test2  = pd.concat([test_df.reset_index(drop=True),  te_mat], axis=1)
    logger.info(f"[TopCodes] train shape {train_df.shape} -> {train2.shape} | test {test_df.shape} -> {test2.shape}")
    return train2, test2, top_codes


# -----------------------------
# TEXT (fold-safe)
# -----------------------------
def fit_text_models(train_text: np.ndarray, seed: int) -> Tuple[Optional[TfidfVectorizer], Optional[TruncatedSVD], int]:
    for min_df in (2, 1):
        vec = TfidfVectorizer(
            lowercase=True,
            token_pattern=r"(?u)\b[\w']+\b",
            ngram_range=(1, 2),
            max_features=TFIDF_MAX_FEATURES,
            min_df=min_df,
            sublinear_tf=True,
        )
        X = vec.fit_transform(train_text)
        if X.shape[1] < 2:
            continue
        n_comp = min(SVD_COMPONENTS, X.shape[1] - 1)
        svd = TruncatedSVD(n_components=n_comp, random_state=seed)
        svd.fit(X)
        return vec, svd, n_comp
    return None, None, 0


def transform_text(vec: Optional[TfidfVectorizer], svd: Optional[TruncatedSVD], text: np.ndarray) -> np.ndarray:
    if vec is None or svd is None:
        return np.zeros((len(text), 0), dtype=np.float32)
    X = vec.transform(text)
    if X.shape[1] == 0:
        return np.zeros((len(text), 0), dtype=np.float32)
    return svd.transform(X).astype(np.float32)


def make_ohe() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True, dtype=np.float32)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True, dtype=np.float32)


# -----------------------------
# STRATIFICATION
# -----------------------------
def make_strat_labels(df: pd.DataFrame, scheme: str) -> np.ndarray:
    if scheme == "chronic_only":
        return df["primary_chronic"].astype(str).values

    if scheme == "chronic_x_costbin_x_pdf":
        # 5 bins on prior_cost_log1p
        x = df["prior_cost_log1p"].astype(float).values
        # qcut can drop bins; use rank as fallback
        try:
            bins = pd.qcut(x, 5, labels=False, duplicates="drop")
        except Exception:
            bins = pd.Series(pd.cut(pd.Series(x).rank(method="average"), bins=5, labels=False)).fillna(0).astype(int)
        bins = np.array(bins).astype(int)
        pdfp = df["pdf_present"].astype(int).values
        return (df["primary_chronic"].astype(str) + "_" + bins.astype(str) + "_" + pdfp.astype(str)).values

    raise ValueError(f"Unknown CV scheme: {scheme}")


# -----------------------------
# CROSS-FITTED SHRINKAGE CALIBRATION
# -----------------------------
def _group_key(df: pd.DataFrame, cols: List[str]) -> np.ndarray:
    return df[cols].astype(str).agg("|".join, axis=1).values

def fit_group_bias(pred: np.ndarray, y: np.ndarray, meta: pd.DataFrame, group_cols: List[str], shrink_k: float) -> Dict[str, float]:
    key = _group_key(meta, group_cols)
    resid = (pred - y).astype(np.float64)
    tmp = pd.DataFrame({"key": key, "resid": resid})
    med = tmp.groupby("key")["resid"].median()
    cnt = tmp.groupby("key")["resid"].size()
    bias = {}
    for k in med.index:
        n = float(cnt.loc[k])
        w = n / (n + float(shrink_k))
        bias[k] = float(w * med.loc[k])  # shrink toward 0
    return bias

def apply_group_bias(pred: np.ndarray, meta: pd.DataFrame, bias: Dict[str, float], group_cols: List[str]) -> np.ndarray:
    key = _group_key(meta, group_cols)
    adj = np.array([bias.get(k, 0.0) for k in key], dtype=np.float64)
    return np.clip(pred - adj, 0.0, None)

def crossfit_group_calibration(oof_pred: np.ndarray, y: np.ndarray, meta: pd.DataFrame,
                               fold_id: np.ndarray, group_cols: List[str], shrink_k: float) -> Tuple[np.ndarray, Dict[str, float]]:
    oof_cal = np.zeros_like(oof_pred, dtype=np.float64)
    for f in np.unique(fold_id):
        tr = fold_id != f
        va = fold_id == f
        b = fit_group_bias(oof_pred[tr], y[tr], meta.iloc[tr], group_cols, shrink_k=shrink_k)
        oof_cal[va] = apply_group_bias(oof_pred[va], meta.iloc[va], b, group_cols)
    bias_full = fit_group_bias(oof_pred, y, meta, group_cols, shrink_k=shrink_k)
    return oof_cal.astype(np.float32), bias_full


# -----------------------------
# METRICS LOGGING
# -----------------------------
def _mae_table_by_decile(df_meta: pd.DataFrame, y: np.ndarray, pred: np.ndarray) -> pd.DataFrame:
    tmp = df_meta.copy()
    tmp["_y"] = y
    tmp["_p"] = pred
    tmp["_ae"] = np.abs(tmp["_y"] - tmp["_p"])
    tmp["_decile"] = pd.qcut(tmp["_y"], 10, labels=False, duplicates="drop")
    g = tmp.groupby("_decile").agg(
        count=("_ae","size"),
        mae=("_ae","mean"),
        sum_ae=("_ae","sum"),
        y_mean=("_y","mean"),
        y_p90=("_y", lambda s: float(np.quantile(s, 0.9))),
    ).reset_index()
    g["share_ae"] = g["sum_ae"] / (g["sum_ae"].sum() + 1e-12)
    return g

def _mae_by_group(df_meta: pd.DataFrame, y: np.ndarray, pred: np.ndarray, col: str, topn: int = 15) -> pd.DataFrame:
    tmp = df_meta.copy()
    tmp["_y"] = y
    tmp["_p"] = pred
    tmp["_ae"] = np.abs(tmp["_y"] - tmp["_p"])
    g = tmp.groupby(col).agg(count=("_ae","size"), mae=("_ae","mean"), sum_ae=("_ae","sum")).sort_values("count", ascending=False)
    return g.head(topn)

def log_eval_report(logger: logging.Logger, title: str, df_meta: pd.DataFrame, y: np.ndarray, pred: np.ndarray) -> None:
    mae = float(mean_absolute_error(y, pred))
    tmp_ae = np.abs(y - pred)
    logger.info(f"[{title}] MAE={mae:.6f} | median_AE={float(np.median(tmp_ae)):.3f} | p90_AE={float(np.quantile(tmp_ae,0.9)):.3f}")

    dec = _mae_table_by_decile(df_meta, y, pred)
    logger.info(f"[{title}] MAE by target decile:\n" + dec.to_string(index=False))

    for col in ["primary_chronic", "insurance", "zip3", "pdf_present"]:
        if col in df_meta.columns:
            tab = _mae_by_group(df_meta, y, pred, col, topn=15)
            logger.info(f"[{title}] MAE by {col} (top 15 by count):\n" + tab.to_string())

    # interaction chronic x insurance
    if "primary_chronic" in df_meta.columns and "insurance" in df_meta.columns:
        tmp = df_meta.copy()
        tmp["_y"] = y
        tmp["_p"] = pred
        tmp["_ae"] = np.abs(tmp["_y"] - tmp["_p"])
        key = tmp["primary_chronic"].astype(str) + "|" + tmp["insurance"].astype(str)
        g = tmp.groupby(key).agg(count=("_ae","size"), mae=("_ae","mean"), sum_ae=("_ae","sum")).sort_values("count", ascending=False)
        logger.info(f"[{title}] MAE by primary_chronic x insurance:\n" + g.to_string())


# -----------------------------
# CONFIG / ABLATION TOGGLES
# -----------------------------
@dataclass
class Config:
    # Ablations (6 toggles)
    use_calib: bool = False
    use_text: bool = True
    use_topcodes: bool = True
    use_cat: bool = False          # default OFF (train_log shows ~1321 MAE)
    use_xgb_slog: bool = False     # default OFF (train_log shows ~1457 MAE)
    cv_scheme: str = "chronic_only"  # or "chronic_x_costbin_x_pdf"

    # Calibration settings
    calib_group_cols: Tuple[str, ...] = ("primary_chronic", "insurance", "pdf_present")
    calib_shrink_k: float = 50.0
    calib_apply_policy: str = "auto"  # "auto" or "always" or "never" (auto=apply only if holdout improves)

    # Mixture-of-experts by pdf_present (extra experiment toggle; optional)
    use_moe_pdf: bool = False

    # bookkeeping
    run_name: str = "baseline"


# -----------------------------
# MODEL PARAMS
# -----------------------------
def make_xgb_params(objective: str = "reg:absoluteerror") -> Dict[str, Any]:
    # Preserve your pattern: early_stopping_rounds in constructor params
    return dict(
        n_estimators=XGB_N_ESTIMATORS,
        learning_rate=0.03,
        max_depth=7,
        min_child_weight=20.0,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=2.0,
        objective=objective,
        eval_metric="mae",
        tree_method="hist",
        device="cuda:0",
        random_state=SEED,
        n_jobs=0,
        early_stopping_rounds=XGB_EARLY_STOPPING,
    )


# -----------------------------
# MATRIX BUILDERS
# -----------------------------
def build_feature_lists(df: pd.DataFrame, cfg: Config) -> Tuple[List[str], List[str]]:
    cat_cols = ["primary_chronic"]
    for c in ["sex", "insurance", "zip3", "zip3_pdf", "insurance_pdf"]:
        if c in df.columns:
            cat_cols.append(c)
    # unique order
    seen=set(); cat_cols=[c for c in cat_cols if not (c in seen or seen.add(c))]

    drop_cols = {"patient_id","ed_cost_next3y_usd","pdf_text","pdf_total","pdf_code_counts_json","pdf_code_costs_json"}
    num_cols = []
    for c in df.columns:
        if c in drop_cols or c in cat_cols:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            if (not cfg.use_topcodes) and (c.startswith("code_cnt_") or c.startswith("code_cost_share_")):
                continue
            num_cols.append(c)
    return sorted(num_cols), cat_cols


def build_xgb_matrices(tr: pd.DataFrame, va: pd.DataFrame, te: pd.DataFrame,
                       num_cols: List[str], cat_cols: List[str],
                       tr_svd: np.ndarray, va_svd: np.ndarray, te_svd: np.ndarray,
                       logger: logging.Logger, fold: int) -> Tuple[sparse.csr_matrix, sparse.csr_matrix, sparse.csr_matrix, OneHotEncoder]:
    ohe = make_ohe()
    ohe.fit(tr[cat_cols].fillna("UNK").astype(str))

    tr_cat = ohe.transform(tr[cat_cols].fillna("UNK").astype(str))
    va_cat = ohe.transform(va[cat_cols].fillna("UNK").astype(str))
    te_cat = ohe.transform(te[cat_cols].fillna("UNK").astype(str))

    tr_num = tr[num_cols].astype(np.float32).values
    va_num = va[num_cols].astype(np.float32).values
    te_num = te[num_cols].astype(np.float32).values

    tr_dense = np.hstack([tr_num, tr_svd]).astype(np.float32)
    va_dense = np.hstack([va_num, va_svd]).astype(np.float32)
    te_dense = np.hstack([te_num, te_svd]).astype(np.float32)

    X_tr = sparse.hstack([sparse.csr_matrix(tr_dense), tr_cat], format="csr")
    X_va = sparse.hstack([sparse.csr_matrix(va_dense), va_cat], format="csr")
    X_te = sparse.hstack([sparse.csr_matrix(te_dense), te_cat], format="csr")

    logger.info(f"[fold {fold}] XGB matrices: tr={X_tr.shape} va={X_va.shape} te={X_te.shape} | svd_dim={tr_svd.shape[1]}")
    return X_tr, X_va, X_te, ohe


# -----------------------------
# CV TRAINING (produces OOF + test preds)
# -----------------------------
def run_cv(train_df: pd.DataFrame, test_df: pd.DataFrame, cfg: Config, logger: logging.Logger,
           fold_count: int, return_fold_models: bool = False) -> Dict[str, Any]:
    assert fold_count == 5, "Hard requirement: CV must be exactly 5 folds."

    num_cols, cat_cols = build_feature_lists(train_df, cfg)
    logger.info(f"[Features] num_cols={len(num_cols)} cat_cols={len(cat_cols)} -> {cat_cols}")

    y = train_df["ed_cost_next3y_usd"].astype(np.float32).values
    strat = make_strat_labels(train_df, cfg.cv_scheme)
    skf = StratifiedKFold(n_splits=fold_count, shuffle=True, random_state=SEED)

    oof = np.zeros(len(train_df), dtype=np.float32)
    test_pred = np.zeros(len(test_df), dtype=np.float32)

    fold_id = np.full(len(train_df), -1, dtype=int)
    best_iters = []
    fold_maes = []

    models = [] if return_fold_models else None

    xgb_params = make_xgb_params("reg:absoluteerror")
    logger.info(f"[XGB] params={xgb_params}")

    # optional extra models (disabled by default)
    xgb_params_slog = make_xgb_params("reg:squaredlogerror") if cfg.use_xgb_slog else None

    if cfg.use_cat and (not _HAS_CATBOOST):
        logger.warning("CatBoost not importable; forcing use_cat=False")
        cfg.use_cat = False

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, strat)):
        fold_id[va_idx] = fold
        with StageTimer(logger, f"fold_{fold}"):
            tr = train_df.iloc[tr_idx].reset_index(drop=True)
            va = train_df.iloc[va_idx].reset_index(drop=True)
            y_tr = tr["ed_cost_next3y_usd"].astype(np.float32).values
            y_va = va["ed_cost_next3y_usd"].astype(np.float32).values

            # fold-safe text
            if cfg.use_text:
                with StageTimer(logger, f"fold_{fold}/text_fit"):
                    vec, svd, svd_dim = fit_text_models(tr["pdf_text"].fillna("").values, seed=SEED + fold)
                    logger.info(f"[fold {fold}] text svd_dim={svd_dim}")
                with StageTimer(logger, f"fold_{fold}/text_transform"):
                    tr_svd = transform_text(vec, svd, tr["pdf_text"].fillna("").values)
                    va_svd = transform_text(vec, svd, va["pdf_text"].fillna("").values)
                    te_svd = transform_text(vec, svd, test_df["pdf_text"].fillna("").values)
            else:
                tr_svd = np.zeros((len(tr), 0), dtype=np.float32)
                va_svd = np.zeros((len(va), 0), dtype=np.float32)
                te_svd = np.zeros((len(test_df), 0), dtype=np.float32)

            with StageTimer(logger, f"fold_{fold}/build_xgb_matrices"):
                X_tr, X_va, X_te, _ = build_xgb_matrices(tr, va, test_df, num_cols, cat_cols, tr_svd, va_svd, te_svd, logger, fold)

            # ---- XGB training (optionally mixture-of-experts by pdf_present) ----
            callbacks = [XGBProgressCallback(logger, period=200, prefix=f"XGB_f{fold}")]
            with StageTimer(logger, f"fold_{fold}/train_xgb"):
                if cfg.use_moe_pdf:
                    tr_pres = (tr["pdf_present"].values == 1)
                    va_pres = (va["pdf_present"].values == 1)
                    te_pres = (test_df["pdf_present"].values == 1)
                    logger.info(f"[fold {fold}] MOE pdf_present counts: tr1={int(tr_pres.sum())} tr0={int((~tr_pres).sum())} | va1={int(va_pres.sum())} va0={int((~va_pres).sum())}")

                    # fallback if subset too small
                    min_n = 50
                    if int(tr_pres.sum()) < min_n or int((~tr_pres).sum()) < min_n or int(va_pres.sum()) < min_n or int((~va_pres).sum()) < min_n:
                        logger.warning(f"[fold {fold}] MOE subset too small; using single model")
                        model_all = fit_xgb_with_guard(xgb_params, SEED + fold, callbacks, X_tr, y_tr, X_va, y_va, logger)
                        pred_va = model_all.predict(X_va).astype(np.float32)
                        pred_te = model_all.predict(X_te).astype(np.float32)
                        best_iters.append(getattr(model_all, "best_iteration", None))
                        if return_fold_models: models.append(("xgb_all", fold, model_all))
                    else:
                        # model for present
                        model_p = fit_xgb_with_guard(xgb_params, SEED + fold, callbacks,
                                                     X_tr[tr_pres], y_tr[tr_pres], X_va[va_pres], y_va[va_pres], logger)
                        # model for missing
                        model_m = fit_xgb_with_guard(xgb_params, SEED + fold + 1000, callbacks,
                                                     X_tr[~tr_pres], y_tr[~tr_pres], X_va[~va_pres], y_va[~va_pres], logger)

                        pred_va = np.zeros(len(y_va), dtype=np.float32)
                        pred_va[va_pres]  = model_p.predict(X_va[va_pres]).astype(np.float32)
                        pred_va[~va_pres] = model_m.predict(X_va[~va_pres]).astype(np.float32)

                        pred_te = np.zeros(len(test_df), dtype=np.float32)
                        pred_te[te_pres]  = model_p.predict(X_te[te_pres]).astype(np.float32)
                        pred_te[~te_pres] = model_m.predict(X_te[~te_pres]).astype(np.float32)

                        best_iters.append(("present", getattr(model_p, "best_iteration", None), "missing", getattr(model_m, "best_iteration", None)))
                        if return_fold_models:
                            models.append(("xgb_present", fold, model_p))
                            models.append(("xgb_missing", fold, model_m))
                else:
                    model_all = fit_xgb_with_guard(xgb_params, SEED + fold, callbacks, X_tr, y_tr, X_va, y_va, logger)
                    pred_va = model_all.predict(X_va).astype(np.float32)
                    pred_te = model_all.predict(X_te).astype(np.float32)
                    best_iters.append(getattr(model_all, "best_iteration", None))
                    if return_fold_models: models.append(("xgb_all", fold, model_all))

            # GPU verification logs
            try:
                if cfg.use_moe_pdf:
                    pass
                else:
                    logger.info(f"[fold {fold}] XGB get_params device={model_all.get_params().get('device')} tree_method={model_all.get_params().get('tree_method')}")
                    cfg_txt = model_all.get_booster().save_config().lower()
                    logger.info(f"[fold {fold}] XGB booster config contains 'cuda': {'cuda' in cfg_txt}")
            except Exception:
                pass

            # fold MAE
            mae_fold = float(mean_absolute_error(y_va, pred_va))
            fold_maes.append(mae_fold)
            logger.info(f"[fold {fold}] XGB fold MAE={mae_fold:.6f}")

            # segment fold MAE by pdf_present
            try:
                va_meta = va[["pdf_present"]].copy()
                va_meta["_y"] = y_va
                va_meta["_p"] = pred_va
                for k in [0,1]:
                    idx = (va_meta["pdf_present"].values == k)
                    if idx.any():
                        logger.info(f"[fold {fold}] XGB fold MAE pdf_present={k}: {mean_absolute_error(va_meta.loc[idx,'_y'], va_meta.loc[idx,'_p']):.6f} (n={idx.sum()})")
            except Exception:
                pass

            # store preds
            oof[va_idx] = pred_va
            test_pred += pred_te / fold_count

    # base OOF (no bias/calib)
    base_oof_mae = float(mean_absolute_error(y, oof))
    logger.info(f"[OOF] base MAE={base_oof_mae:.6f} | fold_maes={['%.3f'%m for m in fold_maes]}")

    # global median bias shift (MAE-optimal constant)
    bias = float(np.median(oof - y))
    oof_b = np.clip(oof - bias, 0.0, None).astype(np.float32)
    test_b = np.clip(test_pred - bias, 0.0, None).astype(np.float32)
    logger.info(f"[Bias] global median residual bias={bias:.6f} | OOF MAE after bias={mean_absolute_error(y, oof_b):.6f}")

    out = {
        "oof_raw": oof,
        "oof_bias": oof_b,
        "test_bias": test_b,
        "y": y,
        "fold_id": fold_id,
        "bias": bias,
        "best_iters": best_iters,
        "num_cols": num_cols,
        "cat_cols": cat_cols,
        "models": models,
    }

    # optional cross-fitted calibration (row-safe)
    if cfg.use_calib:
        group_cols = list(cfg.calib_group_cols)
        # ensure cols exist
        group_cols = [c for c in group_cols if c in train_df.columns]
        if not group_cols:
            logger.warning("[Calib] No valid group cols found; skipping calibration.")
        else:
            with StageTimer(logger, "calibration_crossfit"):
                oof_cal, bias_full = crossfit_group_calibration(oof_b, y, train_df, fold_id, group_cols, shrink_k=cfg.calib_shrink_k)
                test_cal = apply_group_bias(test_b, test_df, bias_full, group_cols).astype(np.float32)

            out["oof_cal"] = oof_cal
            out["test_cal"] = test_cal
            out["calib_group_cols"] = group_cols
            out["calib_bias_full"] = bias_full
            logger.info(f"[Calib] group_cols={group_cols} shrink_k={cfg.calib_shrink_k} | OOF MAE after calib={mean_absolute_error(y, oof_cal):.6f}")

    return out


# -----------------------------
# SANITY HOLDOUT (zip3-group shift)
# -----------------------------
def run_zip3_holdout(train_df: pd.DataFrame, cfg: Config, logger: logging.Logger) -> Dict[str, Any]:
    # group split by zip3 to mimic geo/test shift
    gss = GroupShuffleSplit(n_splits=1, test_size=HOLDOUT_TEST_SIZE, random_state=SEED)
    groups = train_df["zip3"].fillna("UNK").astype(str).values
    tr_idx, ho_idx = next(gss.split(train_df, groups=groups))

    tr = train_df.iloc[tr_idx].reset_index(drop=True)
    ho = train_df.iloc[ho_idx].reset_index(drop=True)

    logger.info(f"[Holdout] zip3-group holdout: train_in={len(tr)} holdout={len(ho)} (test_size={HOLDOUT_TEST_SIZE})")
    logger.info(f"[Holdout] unique zip3 train_in={tr['zip3'].nunique()} holdout={ho['zip3'].nunique()}")

    # We reuse CV trainer to predict holdout (treat holdout as "test_df")
    # Use smaller fold count for speed (HOLDOUT_CV_FOLDS), but keep stratification logic.
    # NOTE: This is only to evaluate generalization; final submission still uses 5-fold CV on full data.
    fold_count = int(HOLDOUT_CV_FOLDS)
    assert fold_count in (2,3,4,5)

    # Temporary CV on tr only
    num_cols, cat_cols = build_feature_lists(tr, cfg)
    y = tr["ed_cost_next3y_usd"].astype(np.float32).values
    strat = make_strat_labels(tr, cfg.cv_scheme)
    skf = StratifiedKFold(n_splits=fold_count, shuffle=True, random_state=SEED)

    ho_pred = np.zeros(len(ho), dtype=np.float32)
    oof = np.zeros(len(tr), dtype=np.float32)
    fold_id = np.full(len(tr), -1, dtype=int)

    xgb_params = make_xgb_params("reg:absoluteerror")

    for fold, (tr_i, va_i) in enumerate(skf.split(tr, strat)):
        fold_id[va_i] = fold
        with StageTimer(logger, f"holdout_cv_fold_{fold}"):
            tr_f = tr.iloc[tr_i].reset_index(drop=True)
            va_f = tr.iloc[va_i].reset_index(drop=True)
            y_tr = tr_f["ed_cost_next3y_usd"].astype(np.float32).values
            y_va = va_f["ed_cost_next3y_usd"].astype(np.float32).values

            if cfg.use_text:
                vec, svd, _ = fit_text_models(tr_f["pdf_text"].fillna("").values, seed=SEED + 999 + fold)
                tr_svd = transform_text(vec, svd, tr_f["pdf_text"].fillna("").values)
                va_svd = transform_text(vec, svd, va_f["pdf_text"].fillna("").values)
                ho_svd = transform_text(vec, svd, ho["pdf_text"].fillna("").values)
            else:
                tr_svd = np.zeros((len(tr_f), 0), dtype=np.float32)
                va_svd = np.zeros((len(va_f), 0), dtype=np.float32)
                ho_svd = np.zeros((len(ho), 0), dtype=np.float32)

            X_tr, X_va, X_ho, _ = build_xgb_matrices(tr_f, va_f, ho, num_cols, cat_cols, tr_svd, va_svd, ho_svd, logger, fold)

            callbacks = [XGBProgressCallback(logger, period=200, prefix=f"HOLD_XGB_f{fold}")]
            model = fit_xgb_with_guard(xgb_params, SEED + 999 + fold, callbacks, X_tr, y_tr, X_va, y_va, logger)

            p_va = model.predict(X_va).astype(np.float32)
            p_ho = model.predict(X_ho).astype(np.float32)

            oof[va_i] = p_va
            ho_pred += p_ho / fold_count

    # bias on train_in oof
    bias = float(np.median(oof - y))
    oof_b = np.clip(oof - bias, 0.0, None).astype(np.float32)
    ho_b = np.clip(ho_pred - bias, 0.0, None).astype(np.float32)

    ho_y = ho["ed_cost_next3y_usd"].astype(np.float32).values

    # optional cross-fit calib on tr only, then apply to holdout
    if cfg.use_calib:
        group_cols = [c for c in list(cfg.calib_group_cols) if c in tr.columns]
        if group_cols:
            oof_cal, bias_full = crossfit_group_calibration(oof_b, y, tr, fold_id, group_cols, shrink_k=cfg.calib_shrink_k)
            ho_cal = apply_group_bias(ho_b, ho, bias_full, group_cols).astype(np.float32)
        else:
            oof_cal, ho_cal = oof_b, ho_b
    else:
        oof_cal, ho_cal = oof_b, ho_b

    return {
        "train_in_meta": tr,
        "train_in_y": y,
        "train_in_pred": oof_cal,
        "holdout_meta": ho,
        "holdout_y": ho_y,
        "holdout_pred": ho_cal,
    }


# -----------------------------
# DATA LOADING + PREP
# -----------------------------
def load_and_prepare(logger: logging.Logger) -> Tuple[pd.DataFrame, pd.DataFrame]:
    with StageTimer(logger, "load_data"):
        train = pd.read_csv(TRAIN_CSV)
        test = pd.read_csv(TEST_CSV)
        logger.info(f"train shape={train.shape} test shape={test.shape}")
        test_order = test["patient_id"].astype(int).values.copy()

    if PATIENTS_CSV.exists():
        with StageTimer(logger, "join_patients"):
            patients = pd.read_csv(PATIENTS_CSV, dtype={"zip3": str})
            keep = [c for c in ["patient_id","age","sex","insurance","zip3"] if c in patients.columns]
            patients = patients[keep]
            train = train.merge(patients, on="patient_id", how="left", sort=False)
            test = test.merge(patients, on="patient_id", how="left", sort=False)
            logger.info(f"joined patients.csv cols={keep} | train now {train.shape}")
    else:
        logger.warning("patients.csv not found; continuing without it (less signal).")

    # pdf cache
    with StageTimer(logger, "pdf_features"):
        all_ids = pd.concat([train["patient_id"], test["patient_id"]]).astype(int).tolist()
        pdf = load_or_extract_pdf_features(logger, all_ids)

    with StageTimer(logger, "merge_pdf"):
        train = train.merge(pdf, on="patient_id", how="left", sort=False)
        test = test.merge(pdf, on="patient_id", how="left", sort=False)
        # preserve test order exactly
        test = test.set_index("patient_id").loc[test_order].reset_index()

    # sanity: pdf_total vs prior cost (dataset says they match)
    if "pdf_total" in train.columns:
        ok = train["pdf_total"].notna() & (train["pdf_total"] > 0)
        if ok.any():
            abs_diff = (train.loc[ok, "prior_ed_cost_5y_usd"] - train.loc[ok, "pdf_total"]).abs()
            logger.info("[PDF sanity] |prior_ed_cost_5y_usd - pdf_total| describe:\n" + str(abs_diff.describe()))

    with StageTimer(logger, "derived_features"):
        train = add_derived_features(train)
        test = add_derived_features(test)

    return train, test


# -----------------------------
# MAIN RUNNER
# -----------------------------
def run_experiment(cfg: Config, make_submission: bool = True, run_holdout: bool = True) -> None:
    run_id = f"{cfg.run_name}_{time.strftime('%Y%m%d_%H%M%S')}"
    out_dir = RUN_DIR / run_id
    out_dir.mkdir(parents=True, exist_ok=True)

    logger = setup_logger(out_dir / "train_log.txt")
    seed_everything(SEED)

    # env report
    with StageTimer(logger, "env_report"):
        logger.info(f"OS: {platform.platform()}")
        logger.info(f"Python: {platform.python_version()}")
        logger.info(f"numpy={np.__version__} pandas={pd.__version__}")
        import sklearn
        logger.info(f"sklearn={sklearn.__version__} xgboost={xgb.__version__}")
        if _HAS_CATBOOST:
            import catboost
            logger.info(f"catboost={catboost.__version__}")
        if _HAS_TORCH:
            logger.info(f"torch={torch.__version__} cuda_available={torch.cuda.is_available()}")
            if ASSERT_GPU_REQUIRED and (not torch.cuda.is_available()):
                raise RuntimeError("CUDA not available in torch; ASSERT_GPU_REQUIRED=True")
            if torch.cuda.is_available():
                logger.info(f"GPU[0]={torch.cuda.get_device_name(0)} capability={torch.cuda.get_device_capability(0)}")
                logger.info("nvidia-smi -L:\n" + try_run_cmd(["nvidia-smi", "-L"]))
        logger.info(f"Config: {asdict(cfg)}")

    # load data
    train, test = load_and_prepare(logger)

    # optional topcodes (computed once, then feature list controls inclusion)
    if cfg.use_topcodes:
        with StageTimer(logger, "topcodes"):
            train, test, top_codes = add_top_code_features(logger, train, test, TOP_CODE_K, TOP_CODE_MIN_TOTAL_COUNT)
            (out_dir / "top_codes.json").write_text(json.dumps(top_codes, indent=2))

    # CV train
    with StageTimer(logger, "cv_train"):
        cv_out = run_cv(train, test, cfg, logger, fold_count=N_FOLDS)

    y = cv_out["y"]
    base_oof = cv_out["oof_bias"]  # bias-shifted OOF

    # decide which prediction to use (calib optional)
    if cfg.use_calib and ("oof_cal" in cv_out):
        oof_final = cv_out["oof_cal"]
        test_final = cv_out["test_cal"]
    else:
        oof_final = base_oof
        test_final = cv_out["test_bias"]

    # OOF report
    with StageTimer(logger, "oof_report"):
        log_eval_report(logger, "OOF", train, y, oof_final)

    # Sanity holdout (zip3-group shift)
    holdout_ok = None
    if run_holdout:
        with StageTimer(logger, "sanity_holdout_zip3"):
            # Evaluate both: without calib and with calib (if enabled) to decide policy
            cfg_no = Config(**{**asdict(cfg), "use_calib": False, "run_name": cfg.run_name + "_holdout_noCalib"})
            cfg_yes = Config(**{**asdict(cfg), "use_calib": True,  "run_name": cfg.run_name + "_holdout_withCalib"})
            # always compute no-calib holdout
            ho_no = run_zip3_holdout(train, cfg_no, logger)
            ho_no_mae = float(mean_absolute_error(ho_no["holdout_y"], ho_no["holdout_pred"]))
            log_eval_report(logger, "HOLDOUT_NO_CALIB", ho_no["holdout_meta"], ho_no["holdout_y"], ho_no["holdout_pred"])

            ho_yes_mae = None
            if cfg.use_calib:
                ho_yes = run_zip3_holdout(train, cfg_yes, logger)
                ho_yes_mae = float(mean_absolute_error(ho_yes["holdout_y"], ho_yes["holdout_pred"]))
                log_eval_report(logger, "HOLDOUT_WITH_CALIB", ho_yes["holdout_meta"], ho_yes["holdout_y"], ho_yes["holdout_pred"])

            # Decide calibration policy (auto)
            if cfg.use_calib and cfg.calib_apply_policy == "auto" and (ho_yes_mae is not None):
                holdout_ok = (ho_yes_mae <= ho_no_mae + 1e-9)
                logger.info(f"[CalibPolicy:auto] holdout_no={ho_no_mae:.6f} holdout_yes={ho_yes_mae:.6f} => apply_to_test={holdout_ok}")
            else:
                holdout_ok = True

    # Apply calibration policy to submission
    if make_submission:
        with StageTimer(logger, "write_submission"):
            pred_to_write = test_final
            if cfg.use_calib and cfg.calib_apply_policy == "auto" and (holdout_ok is False):
                logger.warning("[Submission] Auto policy disabled calibration for test (holdout got worse). Using bias-only preds.")
                pred_to_write = cv_out["test_bias"]

            sub = pd.DataFrame({"patient_id": test["patient_id"].astype(int).values,
                                "ed_cost_next3y_usd": pred_to_write.astype(float)})
            # ensure order matches test.csv exactly
            test_order_ids = pd.read_csv(TEST_CSV, usecols=["patient_id"])["patient_id"].astype(int).values
            assert np.array_equal(sub["patient_id"].values, test_order_ids), "Test order mismatch — submission invalid."
            sub.to_csv(SUBMISSION_PATH, index=False)
            logger.info(f"Submission written: {SUBMISSION_PATH} rows={len(sub)}")
            logger.info("Submission head:\n" + str(sub.head()))

    # Save run artifacts
    with StageTimer(logger, "save_artifacts"):
        (out_dir / "config.json").write_text(json.dumps(asdict(cfg), indent=2))
        oof_df = pd.DataFrame({
            "patient_id": train["patient_id"].astype(int).values,
            "y_true": y.astype(float),
            "oof_pred": oof_final.astype(float),
            "pdf_present": train["pdf_present"].astype(int).values,
            "primary_chronic": train["primary_chronic"].astype(str).values,
            "insurance": train["insurance"].astype(str).values if "insurance" in train.columns else "UNK",
            "zip3": train["zip3"].astype(str).values if "zip3" in train.columns else "UNK",
        })
        oof_df.to_csv(out_dir / "oof_predictions.csv", index=False)

    print(f"\nDONE ✅ Run dir: {out_dir}")
    print(f"Submission: {SUBMISSION_PATH}")


# -----------------------------
# RUN SETTINGS (edit here)
# -----------------------------
# Base config to reproduce your last iteration behavior BUT with CatBoost/SLOG off by default.
CFG = Config(
    use_calib=False,             # ablation toggle 1
    use_text=True,               # ablation toggle 2
    use_topcodes=True,           # ablation toggle 3
    use_cat=False,               # ablation toggle 4 (default off)
    use_xgb_slog=False,          # ablation toggle 5 (default off)
    cv_scheme="chronic_only",    # ablation toggle 6
    use_moe_pdf=False,           # optional experiment
    run_name="ABLA_BASE_XGB",
)

# To run your key ablation quickly, change CFG fields above and re-run this cell.

RUN_ABLATION_SUITE = False  # if True, runs the 6 ablations sequentially (XGB-only)
MAKE_SUBMISSION = True
RUN_HOLDOUT = True

if RUN_ABLATION_SUITE:
    base = CFG
    suite = []

    # 1) remove calibration (already false in base); also test calib true
    suite.append(Config(**{**asdict(base), "run_name": "ABL1_calib_OFF", "use_calib": False}))
    suite.append(Config(**{**asdict(base), "run_name": "ABL1_calib_ON_crossfit", "use_calib": True, "calib_apply_policy": "auto"}))

    # 2) remove text
    suite.append(Config(**{**asdict(base), "run_name": "ABL2_text_OFF", "use_text": False}))
    # 3) remove topcodes
    suite.append(Config(**{**asdict(base), "run_name": "ABL3_topcodes_OFF", "use_topcodes": False}))
    # 4) remove catboost entirely (already off)
    suite.append(Config(**{**asdict(base), "run_name": "ABL4_cat_OFF", "use_cat": False}))
    # 5) remove xgb_slog entirely (already off)
    suite.append(Config(**{**asdict(base), "run_name": "ABL5_slog_OFF", "use_xgb_slog": False}))
    # 6) change CV scheme
    suite.append(Config(**{**asdict(base), "run_name": "ABL6_cv_chronic_cost_pdf", "cv_scheme": "chronic_x_costbin_x_pdf"}))

    for cfg in suite:
        run_experiment(cfg, make_submission=False, run_holdout=True)
else:
    run_experiment(CFG, make_submission=MAKE_SUBMISSION, run_holdout=RUN_HOLDOUT)


2026-02-13 15:47:51 | INFO | ed_cost_onecell_1771015671 | [env_report] START
2026-02-13 15:47:51 | INFO | ed_cost_onecell_1771015671 | OS: Windows-11-10.0.26100-SP0
2026-02-13 15:47:51 | INFO | ed_cost_onecell_1771015671 | Python: 3.12.0
2026-02-13 15:47:51 | INFO | ed_cost_onecell_1771015671 | numpy=2.3.4 pandas=2.3.3
2026-02-13 15:47:51 | INFO | ed_cost_onecell_1771015671 | sklearn=1.7.2 xgboost=3.0.0
2026-02-13 15:47:51 | INFO | ed_cost_onecell_1771015671 | catboost=1.2.8
2026-02-13 15:47:51 | INFO | ed_cost_onecell_1771015671 | torch=2.9.0+cu126 cuda_available=True
2026-02-13 15:47:51 | INFO | ed_cost_onecell_1771015671 | GPU[0]=NVIDIA GeForce RTX 4060 Laptop GPU capability=(8, 9)
2026-02-13 15:47:51 | INFO | ed_cost_onecell_1771015671 | nvidia-smi -L:
GPU 0: NVIDIA GeForce RTX 4060 Laptop GPU (UUID: GPU-9515bdbf-1472-4fa2-9f73-28e1d27eedbf)
2026-02-13 15:47:51 | INFO | ed_cost_onecell_1771015671 | Config: {'use_calib': False, 'use_text': True, 'use_topcodes': True, 'use_cat': Fals


DONE ✅ Run dir: D:\AgentDs\agent_ds_healthcare\runs_postmortem_onecell\ABLA_BASE_XGB_20260213_154751
Submission: D:\AgentDs\agent_ds_healthcare\submission_ICHI_V1.csv


# Iteration 4
- 489.2841

In [28]:
import os, sys, json, re, time, math, logging
from dataclasses import dataclass, asdict
from pathlib import Path
from contextlib import contextmanager
from typing import Dict, Any, List, Tuple, Optional

import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.model_selection import StratifiedKFold, GroupShuffleSplit, train_test_split
from sklearn.metrics import mean_absolute_error, roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction import FeatureHasher

import xgboost as xgb

try:
    import fitz  # PyMuPDF (only needed if you force-rebuild PDF cache)
except Exception:
    fitz = None

# =========================
# CONFIG
# =========================
@dataclass
class Config:
    # modes: "cache_only", "analyze_only", "train_only", "train_submit"
    mode: str = "train_submit"
    run_name: str = "EDCOST_V3_IMPROVED_HASH_EXT"
    seed: int = 2026

    # paths (your Windows paths)
    train_csv: str = r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"
    test_csv: str = r"D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv"
    patients_csv: str = r"D:\AgentDs\agent_ds_healthcare\patients.csv"
    pdf_dir: str = r"D:\AgentDs\agent_ds_healthcare\receipts_pdf"
    cache_dir: str = r"D:\AgentDs\agent_ds_healthcare\cache"
    out_submission: str = r"D:\AgentDs\agent_ds_healthcare\submission_ICHI_V1.csv"

    # optional extra tables (auto-skip if missing)
    admissions_train_csv: str = r"D:\AgentDs\agent_ds_healthcare\admissions_train.csv"
    admissions_test_csv: str = r"D:\AgentDs\agent_ds_healthcare\admissions_test.csv"
    stays_train_csv: str = r"D:\AgentDs\agent_ds_healthcare\stays_train.csv"
    stays_test_csv: str = r"D:\AgentDs\agent_ds_healthcare\stays_test.csv"
    vitals_json: str = r"D:\AgentDs\agent_ds_healthcare\vitals_timeseries.json"

    # feature toggles (IMPROVEMENT CHOICES)
    use_external_tables: bool = True          # big expected gain
    use_code_hash_svd: bool = True            # long-tail CPT/HCPCS signal
    code_hash_n_features: int = 2**15         # 32768 (fast, enough for 4k rows)
    code_svd_dim: int = 96                    # fold-safe compression

    # CV scheme (FIX)
    n_folds: int = 5
    cv_scheme: str = "chronic_x_costbin_x_pdf"  # balances folds better than chronic_only
    cost_bins: int = 5

    # sanity holdout (keep)
    run_sanity_holdout: bool = True
    holdout_zip3_test_size: float = 0.2
    holdout_inner_folds: int = 3

    # adversarial validation (new)
    run_adversarial: bool = True

    # XGB params (GPU + your early stopping pattern)
    xgb_n_estimators: int = 8000
    xgb_early_stopping: int = 400

    # execution controls
    force_recache_external: bool = False
    force_recache_pdf: bool = False   # only if PDF parquet missing or you want rebuild (slow)
    log_level: str = "INFO"

# =========================
# LOGGING + TIMERS
# =========================
def setup_logger(name: str, level: str = "INFO") -> logging.Logger:
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger
    logger.setLevel(getattr(logging, level.upper(), logging.INFO))
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s", "%Y-%m-%d %H:%M:%S")
    h = logging.StreamHandler(sys.stdout)
    h.setFormatter(fmt)
    logger.addHandler(h)
    logger.propagate = False
    return logger

@contextmanager
def stage(logger: logging.Logger, name: str):
    t0 = time.time()
    logger.info(f"[{name}] START")
    try:
        yield
    finally:
        logger.info(f"[{name}] END ({time.time() - t0:.2f}s)")

def seed_everything(seed: int):
    np.random.seed(seed)
    try:
        import random
        random.seed(seed)
    except Exception:
        pass

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def env_report(logger: logging.Logger):
    import platform
    logger.info("[env_report] START")
    logger.info(f"OS: {platform.platform()}")
    logger.info(f"Python: {sys.version.split()[0]}")
    try:
        import numpy, pandas, sklearn
        logger.info(f"numpy={numpy.__version__} pandas={pandas.__version__}")
        logger.info(f"sklearn={sklearn.__version__} xgboost={xgb.__version__}")
    except Exception:
        pass
    try:
        import torch
        logger.info(f"torch={torch.__version__} cuda_available={torch.cuda.is_available()}")
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                prop = torch.cuda.get_device_properties(i)
                logger.info(f"GPU[{i}]={prop.name} capability=({prop.major}, {prop.minor})")
    except Exception:
        pass
    try:
        import subprocess
        out = subprocess.check_output(["nvidia-smi", "-L"], stderr=subprocess.STDOUT, text=True)
        logger.info("nvidia-smi -L:")
        for line in out.strip().splitlines():
            logger.info(line)
    except Exception as e:
        logger.info(f"nvidia-smi not available: {e}")
    logger.info("[env_report] END")

# =========================
# PDF CACHE (LOAD)
# =========================
def load_pdf_cache(cfg: Config, logger: logging.Logger) -> pd.DataFrame:
    cache_path = Path(cfg.cache_dir) / "pdf_features_all_v2.parquet"
    if cache_path.exists() and not cfg.force_recache_pdf:
        df = pd.read_parquet(cache_path)
        if "pdf_present" in df.columns:
            found = int(df["pdf_present"].fillna(0).astype(int).sum())
            miss = int(len(df) - found)
            logger.info(f"[PDF] Cached rows={len(df)} | Found={found} | Missing={miss} | path={cache_path}")
        else:
            logger.info(f"[PDF] Cached rows={len(df)} | path={cache_path}")
        return df

    # Optional minimal rebuild (only if you really need it)
    if fitz is None:
        raise RuntimeError("PyMuPDF (fitz) not installed; cannot rebuild PDF cache.")
    pdf_dir = Path(cfg.pdf_dir)
    pdf_paths = list(pdf_dir.glob("receipt_*.pdf"))
    logger.info(f"[PDF] Rebuilding cache (minimal). Found pdf files={len(pdf_paths)} in {pdf_dir}")
    records = []
    code_pat = re.compile(r"\b([A-Z]?\d{4,5})\b")
    for p in pdf_paths:
        m = re.search(r"receipt_(\d+)\.pdf$", p.name)
        if not m:
            continue
        pid = int(m.group(1))
        text = ""
        try:
            doc = fitz.open(p)
            for page in doc:
                text += page.get_text("text") + "\n"
            doc.close()
        except Exception:
            text = ""
        codes = code_pat.findall(text)
        cnt = {}
        for c in codes:
            cnt[c] = cnt.get(c, 0) + 1
        total = np.nan
        mt = re.search(r"TOTAL\s+([0-9]{1,3}(?:,[0-9]{3})*(?:\.[0-9]{2})?)", text)
        if mt:
            try:
                total = float(mt.group(1).replace(",", ""))
            except Exception:
                total = np.nan
        records.append({
            "patient_id": pid,
            "pdf_present": 1 if text.strip() else 0,
            "pdf_text": text.strip(),
            "pdf_total": total,
            "pdf_code_counts_json": json.dumps(cnt),
            "pdf_code_costs_json": json.dumps({}),
        })
    df = pd.DataFrame(records)
    ensure_dir(Path(cfg.cache_dir))
    df.to_parquet(cache_path, index=False)
    logger.info(f"[PDF] Cache written: {cache_path} rows={len(df)}")
    return df

# =========================
# EXTERNAL TABLE FEATURES (CACHED)
# =========================
def _safe_read_csv(path: Path, logger: logging.Logger, **kwargs) -> Optional[pd.DataFrame]:
    if not path.exists():
        logger.info(f"[external] missing file: {path}")
        return None
    try:
        return pd.read_csv(path, **kwargs)
    except Exception as e:
        logger.info(f"[external] failed reading {path}: {e}")
        return None

def build_admissions_patient_features(cfg: Config, logger: logging.Logger) -> Optional[pd.DataFrame]:
    cache_path = Path(cfg.cache_dir) / "admissions_patient_features_v1.parquet"
    if cache_path.exists() and not cfg.force_recache_external:
        logger.info(f"[admissions] Using cached: {cache_path}")
        return pd.read_parquet(cache_path)

    adm_train = _safe_read_csv(Path(cfg.admissions_train_csv), logger)
    adm_test  = _safe_read_csv(Path(cfg.admissions_test_csv), logger)
    if adm_train is None and adm_test is None:
        return None

    adm = pd.concat([x for x in [adm_train, adm_test] if x is not None], ignore_index=True)
    if "readmit_30d" in adm.columns:
        adm = adm.drop(columns=["readmit_30d"])

    for col in ["los_days", "acuity_emergent", "charlson_band", "ed_visits_6m", "discharge_weekday"]:
        if col in adm.columns:
            adm[col] = pd.to_numeric(adm[col], errors="coerce")

    g = adm.groupby("patient_id", sort=False)
    feats = pd.DataFrame({"patient_id": g.size().index.astype(int), "adm_count": g.size().values.astype(np.float32)})

    def _agg(col, func, name):
        if col not in adm.columns:
            feats[name] = 0.0
            return
        feats[name] = getattr(g[col], func)().values.astype(np.float32)

    _agg("los_days", "mean", "adm_los_mean")
    _agg("los_days", "max",  "adm_los_max")
    _agg("acuity_emergent", "mean", "adm_acuity_rate")
    _agg("charlson_band", "mean", "adm_charlson_mean")
    _agg("charlson_band", "max",  "adm_charlson_max")
    _agg("ed_visits_6m",  "mean", "adm_ed6m_mean")
    _agg("ed_visits_6m",  "max",  "adm_ed6m_max")
    _agg("ed_visits_6m",  "sum",  "adm_ed6m_sum")

    if "discharge_weekday" in adm.columns:
        adm["_is_weekend"] = adm["discharge_weekday"].isin([6, 7]).astype(np.float32)
        feats["adm_weekend_discharge_rate"] = g["_is_weekend"].mean().values.astype(np.float32)
        adm = adm.drop(columns=["_is_weekend"])
    else:
        feats["adm_weekend_discharge_rate"] = 0.0

    if "primary_dx" in adm.columns:
        dx = pd.get_dummies(adm["primary_dx"], prefix="adm_dx")
        dx["patient_id"] = adm["patient_id"].values
        dx_g = dx.groupby("patient_id", sort=False).sum().reset_index()
        feats = feats.merge(dx_g, on="patient_id", how="left")
        for c in [c for c in feats.columns if c.startswith("adm_dx_")]:
            feats[c] = feats[c].fillna(0.0).astype(np.float32)
            feats[c + "_share"] = feats[c] / np.maximum(feats["adm_count"], 1.0)

    feats["adm_present"] = 1.0
    ensure_dir(Path(cfg.cache_dir))
    feats.to_parquet(cache_path, index=False)
    logger.info(f"[admissions] Cache written: {cache_path} rows={len(feats)}")
    return feats

def build_stays_patient_features(cfg: Config, logger: logging.Logger) -> Optional[pd.DataFrame]:
    cache_path = Path(cfg.cache_dir) / "stays_patient_features_v1.parquet"
    if cache_path.exists() and not cfg.force_recache_external:
        logger.info(f"[stays] Using cached: {cache_path}")
        return pd.read_parquet(cache_path)

    st_train = _safe_read_csv(Path(cfg.stays_train_csv), logger)
    st_test  = _safe_read_csv(Path(cfg.stays_test_csv), logger)
    if st_train is None and st_test is None:
        return None

    st = pd.concat([x for x in [st_train, st_test] if x is not None], ignore_index=True)
    if "discharge_ready_day11" in st.columns:
        st = st.drop(columns=["discharge_ready_day11"])

    g = st.groupby("patient_id", sort=False)
    feats = pd.DataFrame({"patient_id": g.size().index.astype(int), "stay_count": g.size().values.astype(np.float32)})

    if "unit_type" in st.columns:
        ut = pd.get_dummies(st["unit_type"], prefix="unit")
        ut["patient_id"] = st["patient_id"].values
        ut_g = ut.groupby("patient_id", sort=False).sum().reset_index()
        feats = feats.merge(ut_g, on="patient_id", how="left")
        for c in [c for c in feats.columns if c.startswith("unit_")]:
            feats[c] = feats[c].fillna(0.0).astype(np.float32)
            feats[c + "_share"] = feats[c] / np.maximum(feats["stay_count"], 1.0)

    if "admission_reason" in st.columns:
        ar = pd.get_dummies(st["admission_reason"], prefix="stay_reason")
        ar["patient_id"] = st["patient_id"].values
        ar_g = ar.groupby("patient_id", sort=False).sum().reset_index()
        feats = feats.merge(ar_g, on="patient_id", how="left")
        for c in [c for c in feats.columns if c.startswith("stay_reason_")]:
            feats[c] = feats[c].fillna(0.0).astype(np.float32)
            feats[c + "_share"] = feats[c] / np.maximum(feats["stay_count"], 1.0)

    feats["stay_present"] = 1.0
    ensure_dir(Path(cfg.cache_dir))
    feats.to_parquet(cache_path, index=False)
    logger.info(f"[stays] Cache written: {cache_path} rows={len(feats)}")
    return feats

def build_vitals_patient_features(cfg: Config, logger: logging.Logger) -> Optional[pd.DataFrame]:
    cache_path = Path(cfg.cache_dir) / "vitals_patient_features_v1.parquet"
    if cache_path.exists() and not cfg.force_recache_external:
        logger.info(f"[vitals] Using cached: {cache_path}")
        return pd.read_parquet(cache_path)

    vit_path = Path(cfg.vitals_json)
    if not vit_path.exists():
        logger.info(f"[vitals] missing file: {vit_path}")
        return None

    st_train = _safe_read_csv(Path(cfg.stays_train_csv), logger)
    st_test  = _safe_read_csv(Path(cfg.stays_test_csv), logger)
    if st_train is None and st_test is None:
        logger.info("[vitals] stays files missing; cannot map stay_id to patient_id")
        return None
    st = pd.concat([x for x in [st_train, st_test] if x is not None], ignore_index=True)
    if "discharge_ready_day11" in st.columns:
        st = st.drop(columns=["discharge_ready_day11"])
    if "stay_id" not in st.columns or "patient_id" not in st.columns:
        logger.info("[vitals] stays missing stay_id/patient_id columns")
        return None
    stay_to_pid = st.set_index("stay_id")["patient_id"].to_dict()

    try:
        with open(vit_path, "r", encoding="utf-8") as f:
            vit = json.load(f)
    except Exception as e:
        logger.info(f"[vitals] failed reading json: {e}")
        return None

    rows = []
    for obj in vit:
        sid = obj.get("stay_id")
        if sid is None:
            continue
        pid = stay_to_pid.get(sid)
        if pid is None:
            continue
        days = obj.get("days") or []
        if not days:
            continue

        def arr(key):
            return [d.get(key) for d in days if d.get(key) is not None]

        hr = arr("hr"); sbp = arr("sbp"); dbp = arr("dbp"); temp = arr("temp_c"); rr = arr("rr")
        note_len = [len(str(d.get("note", ""))) for d in days]

        def stats(v):
            a = np.asarray(v, dtype=np.float32)
            if a.size == 0:
                return (np.nan, np.nan, np.nan, np.nan, np.nan)
            return (float(np.nanmean(a)), float(np.nanmin(a)), float(np.nanmax(a)), float(np.nanstd(a)), float(a[-1] - a[0]))

        hr_m, hr_min, hr_max, hr_std, hr_d = stats(hr)
        sbp_m, sbp_min, sbp_max, sbp_std, sbp_d = stats(sbp)
        dbp_m, dbp_min, dbp_max, dbp_std, dbp_d = stats(dbp)
        t_m, t_min, t_max, t_std, t_d = stats(temp)
        rr_m, rr_min, rr_max, rr_std, rr_d = stats(rr)
        nl_m, _, nl_max, _, _ = stats(note_len)

        rows.append({
            "patient_id": int(pid),
            "vitals_stay_count": 1.0,
            "hr_mean": hr_m, "hr_min": hr_min, "hr_max": hr_max, "hr_std": hr_std, "hr_delta": hr_d,
            "sbp_mean": sbp_m, "sbp_min": sbp_min, "sbp_max": sbp_max, "sbp_std": sbp_std, "sbp_delta": sbp_d,
            "dbp_mean": dbp_m, "dbp_min": dbp_min, "dbp_max": dbp_max, "dbp_std": dbp_std, "dbp_delta": dbp_d,
            "temp_mean": t_m, "temp_min": t_min, "temp_max": t_max, "temp_std": t_std, "temp_delta": t_d,
            "rr_mean": rr_m, "rr_min": rr_min, "rr_max": rr_max, "rr_std": rr_std, "rr_delta": rr_d,
            "stay_note_len_mean": nl_m, "stay_note_len_max": nl_max,
        })

    if not rows:
        logger.info("[vitals] no rows parsed")
        return None

    stay_df = pd.DataFrame(rows)
    g = stay_df.groupby("patient_id", sort=False)
    agg = {c: "mean" for c in stay_df.columns if c != "patient_id"}
    agg["vitals_stay_count"] = "sum"
    feats = g.agg(agg).reset_index()
    feats["vitals_present"] = 1.0

    ensure_dir(Path(cfg.cache_dir))
    feats.to_parquet(cache_path, index=False)
    logger.info(f"[vitals] Cache written: {cache_path} rows={len(feats)}")
    return feats

# =========================
# DATA LOAD + MERGE
# =========================
def load_and_prepare(cfg: Config, logger: logging.Logger) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train = pd.read_csv(Path(cfg.train_csv))
    test  = pd.read_csv(Path(cfg.test_csv))
    logger.info(f"train shape={train.shape} test shape={test.shape}")

    # patients
    ppath = Path(cfg.patients_csv)
    if ppath.exists():
        patients = pd.read_csv(ppath, dtype={"zip3": str})
        keep = ["patient_id", "age", "sex", "insurance", "zip3"]
        train = train.merge(patients[keep], on="patient_id", how="left")
        test  = test.merge(patients[keep], on="patient_id", how="left")
        logger.info(f"joined patients.csv cols={keep} | train now {train.shape}")
    else:
        logger.info(f"patients.csv missing: {ppath}")

    # pdf cache
    pdf = load_pdf_cache(cfg, logger)
    if "pdf_present" not in pdf.columns:
        pdf["pdf_present"] = 0
    train = train.merge(pdf, on="patient_id", how="left")
    test  = test.merge(pdf, on="patient_id", how="left")

    for df in (train, test):
        df["pdf_present"] = df["pdf_present"].fillna(0).astype(np.int8)
        df["pdf_missing"] = (1 - df["pdf_present"]).astype(np.int8)

    # external
    if cfg.use_external_tables:
        with stage(logger, "external_features"):
            adm = build_admissions_patient_features(cfg, logger)
            st  = build_stays_patient_features(cfg, logger)
            vit = build_vitals_patient_features(cfg, logger)
            for feat in (adm, st, vit):
                if feat is None:
                    continue
                train = train.merge(feat, on="patient_id", how="left")
                test  = test.merge(feat, on="patient_id", how="left")

        for df in (train, test):
            for col in ["adm_present", "stay_present", "vitals_present"]:
                if col not in df.columns:
                    df[col] = 0.0
                df[col] = df[col].fillna(0.0).astype(np.float32)

    # categorical fill
    for df in (train, test):
        for c in ["primary_chronic", "sex", "insurance", "zip3", "zip3_pdf", "insurance_pdf"]:
            if c in df.columns:
                df[c] = df[c].fillna("UNK").astype(str)

    # derived
    age_med = pd.to_numeric(train["age"], errors="coerce").median() if "age" in train.columns else 0.0
    for df in (train, test):
        df["age"] = pd.to_numeric(df.get("age", age_med), errors="coerce").fillna(age_med).astype(np.float32)
        df["prior_ed_visits_5y"] = pd.to_numeric(df["prior_ed_visits_5y"], errors="coerce").fillna(0).astype(np.float32)
        df["prior_ed_cost_5y_usd"] = pd.to_numeric(df["prior_ed_cost_5y_usd"], errors="coerce").fillna(0).astype(np.float32)
        df["prior_cost_log1p"] = np.log1p(df["prior_ed_cost_5y_usd"]).astype(np.float32)
        df["prior_visits_log1p"] = np.log1p(df["prior_ed_visits_5y"]).astype(np.float32)
        df["cost_per_visit"] = (df["prior_ed_cost_5y_usd"] / np.maximum(df["prior_ed_visits_5y"], 1.0)).astype(np.float32)
        df["cost_per_visit_log1p"] = np.log1p(df["cost_per_visit"]).astype(np.float32)
        df["prior_cost_x_visits"] = (df["prior_cost_log1p"] * df["prior_visits_log1p"]).astype(np.float32)

    # sanity: pdf_total matches prior cost when present (dataset design)
    if "pdf_total" in train.columns:
        mask = train["pdf_present"] == 1
        if mask.any():
            diff = (train.loc[mask, "prior_ed_cost_5y_usd"].astype(np.float32) - train.loc[mask, "pdf_total"].astype(np.float32))
            logger.info(f"[PDF sanity] |prior_ed_cost_5y_usd - pdf_total| describe:\n{diff.describe()}")
    return train, test

# =========================
# CODE HASH (STRUCTURED PDF)
# =========================
def _parse_json_dict(s: Any) -> Dict[str, float]:
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return {}
    if isinstance(s, dict):
        return {str(k): float(v) for k, v in s.items()}
    if not isinstance(s, str):
        return {}
    s = s.strip()
    if not s or s.lower() in ("nan", "none"):
        return {}
    try:
        d = json.loads(s)
        if isinstance(d, dict):
            return {str(k): float(v) for k, v in d.items()}
    except Exception:
        return {}
    return {}

def build_code_feature_dicts(df: pd.DataFrame) -> List[Dict[str, float]]:
    dicts: List[Dict[str, float]] = []
    cnt_col = "pdf_code_counts_json"
    cost_col = "pdf_code_costs_json"
    if cnt_col not in df.columns:
        return [{} for _ in range(len(df))]

    for _, row in df.iterrows():
        counts = _parse_json_dict(row.get(cnt_col))
        costs  = _parse_json_dict(row.get(cost_col))
        denom = float(row.get("prior_ed_cost_5y_usd", 0.0) or 0.0)
        denom = max(denom, 1.0)

        feat: Dict[str, float] = {}
        for code, v in counts.items():
            if v <= 0:
                continue
            c = str(code)
            feat[f"cnt:{c}"] = feat.get(f"cnt:{c}", 0.0) + float(v)
            p2 = c[:2] if len(c) >= 2 else c
            feat[f"cnt_p2:{p2}"] = feat.get(f"cnt_p2:{p2}", 0.0) + float(v)

        for code, v in costs.items():
            if v == 0:
                continue
            c = str(code)
            shr = float(v) / denom
            feat[f"shr:{c}"] = feat.get(f"shr:{c}", 0.0) + shr
            p2 = c[:2] if len(c) >= 2 else c
            feat[f"shr_p2:{p2}"] = feat.get(f"shr_p2:{p2}", 0.0) + shr

        dicts.append(feat)
    return dicts

# =========================
# CV STRAT LABELS
# =========================
def make_strat_labels(train: pd.DataFrame, cfg: Config) -> np.ndarray:
    if cfg.cv_scheme == "chronic_only":
        return train["primary_chronic"].astype(str).values
    if cfg.cv_scheme == "chronic_x_costbin_x_pdf":
        x = train["prior_cost_log1p"].values
        try:
            bins = pd.qcut(x, q=cfg.cost_bins, labels=False, duplicates="drop").astype(int).astype(str)
        except Exception:
            bins = pd.Series(pd.cut(x, bins=cfg.cost_bins, labels=False)).fillna(0).astype(int).astype(str)
        return (train["primary_chronic"].astype(str) + "|" + bins + "|p" + train["pdf_present"].astype(str)).values
    return train["primary_chronic"].astype(str).values

# =========================
# FEATURES LISTS
# =========================
def build_feature_lists(train: pd.DataFrame, test: pd.DataFrame, logger: logging.Logger) -> Tuple[List[str], List[str]]:
    drop_cols = {
        "patient_id",
        "ed_cost_next3y_usd",
        "pdf_text",
        "pdf_code_counts_json",
        "pdf_code_costs_json",
        "pdf_total",  # redundant w/ prior cost (per dataset design)
    }
    cat_cols = [c for c in ["primary_chronic", "sex", "insurance", "zip3", "zip3_pdf", "insurance_pdf"] if c in train.columns]
    num_cols = [c for c in train.columns if c not in drop_cols and c not in cat_cols]

    for c in num_cols:
        train[c] = pd.to_numeric(train[c], errors="coerce")
        test[c]  = pd.to_numeric(test[c], errors="coerce")
    train[num_cols] = train[num_cols].fillna(0.0)
    test[num_cols]  = test[num_cols].fillna(0.0)

    logger.info(f"[Features] num_cols={len(num_cols)} cat_cols={len(cat_cols)} -> {cat_cols}")
    logger.info(f"[Features] train shape={train.shape} test shape={test.shape}")
    return num_cols, cat_cols

def fit_ohe(train: pd.DataFrame, test: pd.DataFrame, cat_cols: List[str]) -> Tuple[OneHotEncoder, sp.csr_matrix, sp.csr_matrix]:
    if not cat_cols:
        ohe = OneHotEncoder(handle_unknown="ignore")
        return ohe, sp.csr_matrix((len(train), 0), dtype=np.float32), sp.csr_matrix((len(test), 0), dtype=np.float32)
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True, dtype=np.float32)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=True, dtype=np.float32)
    X_tr = ohe.fit_transform(train[cat_cols].astype(str)).tocsr()
    X_te = ohe.transform(test[cat_cols].astype(str)).tocsr()
    return ohe, X_tr, X_te

# =========================
# XGB CALLBACK (YOUR PATTERN)
# =========================
class XGBProgressCallback(xgb.callback.TrainingCallback):
    def __init__(self, logger: logging.Logger, tag: str, period: int = 200):
        self.logger = logger
        self.tag = tag
        self.period = period
    def after_iteration(self, model, epoch: int, evals_log: Dict[str, Dict[str, List[float]]]):
        if epoch == 0 or (epoch + 1) % self.period == 0:
            try:
                v0 = list(evals_log.values())[0]
                metric = list(v0.keys())[0]
                val = v0[metric][-1]
                self.logger.info(f"[{self.tag}] iter={epoch+1} validation_0-{metric}={val:.6f}")
            except Exception:
                pass
        return False

# =========================
# CV TRAIN (GPU XGB)
# =========================
def build_final_matrices(
    train: pd.DataFrame,
    test: pd.DataFrame,
    num_cols: List[str],
    cat_cols: List[str],
    cfg: Config,
    logger: logging.Logger,
    folds: List[Tuple[np.ndarray, np.ndarray]],
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[Dict[str, Any]]]:

    y = train["ed_cost_next3y_usd"].astype(np.float32).values
    oof = np.zeros(len(train), dtype=np.float32)
    test_sum = np.zeros(len(test), dtype=np.float32)
    fold_info: List[Dict[str, Any]] = []

    X_num_tr = train[num_cols].astype(np.float32).values
    X_num_te = test[num_cols].astype(np.float32).values
    _, X_cat_tr, X_cat_te = fit_ohe(train, test, cat_cols)

    if cfg.use_code_hash_svd:
        with stage(logger, "code_hash_build"):
            tr_dicts = build_code_feature_dicts(train)
            te_dicts = build_code_feature_dicts(test)
            hasher = FeatureHasher(
                n_features=cfg.code_hash_n_features,
                input_type="dict",
                alternate_sign=False,
                dtype=np.float32,
            )
            X_code_tr = hasher.transform(tr_dicts).tocsr()
            X_code_te = hasher.transform(te_dicts).tocsr()
            logger.info(f"[CodeHash] csr shapes tr={X_code_tr.shape} te={X_code_te.shape} nnz_tr={X_code_tr.nnz}")

    # YOUR EARLY STOPPING PATTERN: early_stopping_rounds IN PARAMS
    xgb_params = dict(
        n_estimators=cfg.xgb_n_estimators,
        learning_rate=0.03,
        max_depth=6,
        min_child_weight=30.0,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=5.0,
        reg_alpha=0.0,
        objective="reg:absoluteerror",
        eval_metric="mae",
        tree_method="hist",
        device="cuda:0",
        random_state=cfg.seed,
        n_jobs=0,
        early_stopping_rounds=cfg.xgb_early_stopping,
    )
    logger.info(f"[XGB] params={xgb_params}")

    for fold, (tr_idx, va_idx) in enumerate(folds):
        with stage(logger, f"fold_{fold}"):
            y_tr, y_va = y[tr_idx], y[va_idx]
            X_tr_num, X_va_num = X_num_tr[tr_idx], X_num_tr[va_idx]

            if cfg.use_code_hash_svd:
                with stage(logger, f"fold_{fold}/code_svd_fit"):
                    svd = TruncatedSVD(n_components=cfg.code_svd_dim, random_state=cfg.seed + fold)
                    X_tr_code = svd.fit_transform(X_code_tr[tr_idx]).astype(np.float32)
                    X_va_code = svd.transform(X_code_tr[va_idx]).astype(np.float32)
                    X_te_code = svd.transform(X_code_te).astype(np.float32)
                    logger.info(f"[fold {fold}] code svd_dim={cfg.code_svd_dim} explained_var={svd.explained_variance_ratio_.sum():.4f}")
            else:
                X_tr_code = np.zeros((len(tr_idx), 0), dtype=np.float32)
                X_va_code = np.zeros((len(va_idx), 0), dtype=np.float32)
                X_te_code = np.zeros((len(test), 0), dtype=np.float32)

            X_tr_dense = np.hstack([X_tr_num, X_tr_code]).astype(np.float32)
            X_va_dense = np.hstack([X_va_num, X_va_code]).astype(np.float32)
            X_te_dense = np.hstack([X_num_te, X_te_code]).astype(np.float32)

            X_tr = sp.hstack([sp.csr_matrix(X_tr_dense), X_cat_tr[tr_idx]], format="csr")
            X_va = sp.hstack([sp.csr_matrix(X_va_dense), X_cat_tr[va_idx]], format="csr")
            X_te = sp.hstack([sp.csr_matrix(X_te_dense), X_cat_te], format="csr")
            logger.info(f"[fold {fold}] X shapes tr={X_tr.shape} va={X_va.shape} te={X_te.shape}")

            with stage(logger, f"fold_{fold}/train_xgb"):
                model = xgb.XGBRegressor(
                    **{**xgb_params, "random_state": cfg.seed + fold},
                    callbacks=[XGBProgressCallback(logger, f"XGB_f{fold}", period=200)],
                )
                # fit() stays clean: eval_set only
                try:
                    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
                except TypeError as e:
                    logger.info(f"[fold {fold}] XGB TypeError fallback: {e}")
                    model = xgb.XGBRegressor(**{**xgb_params, "random_state": cfg.seed + fold})
                    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

            # GPU verification logs
            try:
                params = model.get_params()
                logger.info(f"[fold {fold}] XGB get_params device={params.get('device')} tree_method={params.get('tree_method')}")
                booster_conf = model.get_booster().save_config()
                logger.info(f"[fold {fold}] XGB booster config contains 'cuda': {'cuda' in booster_conf.lower()}")
            except Exception as e:
                logger.info(f"[fold {fold}] XGB verification failed: {e}")

            pred_va = np.clip(model.predict(X_va).astype(np.float32), 0, None)
            pred_te = np.clip(model.predict(X_te).astype(np.float32), 0, None)

            oof[va_idx] = pred_va
            test_sum += pred_te

            fold_mae = float(mean_absolute_error(y_va, pred_va))
            bi = getattr(model, "best_iteration", None)
            fold_info.append({"fold": fold, "mae": fold_mae, "best_iteration": bi})
            logger.info(f"[fold {fold}] XGB fold MAE={fold_mae:.6f} best_iteration={bi}")

    test_pred = np.clip((test_sum / len(folds)).astype(np.float32), 0, None)
    return oof, test_pred, y, fold_info

# =========================
# REPORTS
# =========================
def report_oof(logger: logging.Logger, train: pd.DataFrame, oof: np.ndarray, y: np.ndarray):
    mae = float(mean_absolute_error(y, oof))
    ae = np.abs(y - oof)
    logger.info(f"[OOF] MAE={mae:.6f} | median_AE={float(np.median(ae)):.3f} | p90_AE={float(np.quantile(ae,0.9)):.3f}")

    dec = pd.qcut(y, q=10, labels=False, duplicates="drop")
    df = pd.DataFrame({"y": y, "ae": ae, "dec": dec})
    grp = df.groupby("dec").agg(
        count=("ae","size"),
        mae=("ae","mean"),
        sum_ae=("ae","sum"),
        y_mean=("y","mean"),
        y_p90=("y", lambda s: float(np.quantile(s,0.9))),
    )
    grp["share_ae"] = grp["sum_ae"] / grp["sum_ae"].sum()
    logger.info("[OOF] MAE by target decile:\n" + grp.reset_index().to_string(index=False))

    for col in ["primary_chronic", "insurance", "zip3", "pdf_present"]:
        if col in train.columns:
            g = (pd.DataFrame({col: train[col].astype(str), "ae": ae})
                 .groupby(col)
                 .agg(count=("ae","size"), mae=("ae","mean"), sum_ae=("ae","sum"))
                 .sort_values("count", ascending=False)
                 .head(15))
            logger.info(f"[OOF] MAE by {col} (top 15 by count):\n" + g.to_string())

def sanity_holdout_zip3(cfg: Config, logger: logging.Logger, train: pd.DataFrame, num_cols: List[str], cat_cols: List[str]):
    if "zip3" not in train.columns:
        logger.info("[Holdout] zip3 not present; skipping")
        return
    gss = GroupShuffleSplit(n_splits=1, test_size=cfg.holdout_zip3_test_size, random_state=cfg.seed)
    groups = train["zip3"].astype(str).values
    idx = np.arange(len(train))
    tr_in_idx, ho_idx = next(gss.split(idx, groups=groups))
    logger.info(f"[Holdout] zip3-group holdout: train_in={len(tr_in_idx)} holdout={len(ho_idx)}")
    logger.info(f"[Holdout] unique zip3 train_in={len(np.unique(groups[tr_in_idx]))} holdout={len(np.unique(groups[ho_idx]))}")

    tr_in = train.iloc[tr_in_idx].reset_index(drop=True)
    ho    = train.iloc[ho_idx].reset_index(drop=True)

    strat = make_strat_labels(tr_in, cfg)
    skf = StratifiedKFold(n_splits=cfg.holdout_inner_folds, shuffle=True, random_state=cfg.seed)
    folds = [(tr, va) for tr, va in skf.split(np.zeros(len(tr_in)), strat)]

    _, ho_pred, _, _ = build_final_matrices(tr_in, ho, num_cols, cat_cols, cfg, logger, folds)
    y_ho = ho["ed_cost_next3y_usd"].astype(np.float32).values
    mae = float(mean_absolute_error(y_ho, ho_pred))
    ae = np.abs(y_ho - ho_pred)
    logger.info(f"[HOLDOUT] MAE={mae:.6f} | median_AE={float(np.median(ae)):.3f} | p90_AE={float(np.quantile(ae,0.9)):.3f}")

# =========================
# ADVERSARIAL VALIDATION
# =========================
def adversarial_validation(cfg: Config, logger: logging.Logger, train: pd.DataFrame, test: pd.DataFrame, num_cols: List[str], cat_cols: List[str]):
    df_all = pd.concat([train.assign(_is_test=0), test.assign(_is_test=1)], ignore_index=True)
    y = df_all["_is_test"].astype(np.int32).values

    X_num = df_all[num_cols].astype(np.float32).fillna(0.0).values
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True, dtype=np.float32)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=True, dtype=np.float32)
    X_cat = ohe.fit_transform(df_all[cat_cols].astype(str)).tocsr() if cat_cols else sp.csr_matrix((len(df_all), 0), dtype=np.float32)
    X = sp.hstack([sp.csr_matrix(X_num), X_cat], format="csr")

    X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.25, random_state=cfg.seed, stratify=y)

    params = dict(
        n_estimators=2000,
        learning_rate=0.05,
        max_depth=4,
        min_child_weight=10.0,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=2.0,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        device="cuda:0",
        n_jobs=0,
        random_state=cfg.seed,
        early_stopping_rounds=100,
    )
    logger.info(f"[ADV] XGBClassifier params={params}")
    model = xgb.XGBClassifier(**params, callbacks=[XGBProgressCallback(logger, "ADV", period=200)])
    try:
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    except TypeError as e:
        logger.info(f"[ADV] TypeError fallback: {e}")
        model = xgb.XGBClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

    p = model.predict_proba(X_va)[:, 1]
    auc = float(roc_auc_score(y_va, p))
    logger.info(f"[ADV] AUC(train vs test)={auc:.6f} (0.50=no shift, >0.65=notable shift)")
    try:
        booster_conf = model.get_booster().save_config()
        logger.info(f"[ADV] booster config contains 'cuda': {'cuda' in booster_conf.lower()}")
    except Exception:
        pass

# =========================
# SUBMISSION
# =========================
def write_submission(cfg: Config, logger: logging.Logger, test: pd.DataFrame, pred: np.ndarray):
    out_path = Path(cfg.out_submission)
    sub = pd.DataFrame({"patient_id": test["patient_id"].values, "ed_cost_next3y_usd": pred.astype(np.float32)})
    sub.to_csv(out_path, index=False)
    logger.info(f"Submission written: {out_path} rows={len(sub)}")
    logger.info("Submission head:\n" + sub.head().to_string(index=False))

# =========================
# MAIN
# =========================
def main(cfg: Config):
    seed_everything(cfg.seed)
    logger = setup_logger(cfg.run_name, cfg.log_level)
    ensure_dir(Path(cfg.cache_dir))

    env_report(logger)
    logger.info(f"Config: {asdict(cfg)}")

    with stage(logger, "load_prepare"):
        train, test = load_and_prepare(cfg, logger)

    with stage(logger, "feature_lists"):
        num_cols, cat_cols = build_feature_lists(train, test, logger)

    if cfg.run_adversarial and cfg.mode in ("analyze_only", "train_only", "train_submit"):
        with stage(logger, "adversarial_validation"):
            adversarial_validation(cfg, logger, train, test, num_cols, cat_cols)

    if cfg.mode in ("cache_only", "analyze_only"):
        logger.info(f"[mode={cfg.mode}] Done.")
        return

    strat = make_strat_labels(train, cfg)
    skf = StratifiedKFold(n_splits=cfg.n_folds, shuffle=True, random_state=cfg.seed)
    folds = [(tr, va) for tr, va in skf.split(np.zeros(len(train)), strat)]
    logger.info(f"[CV] folds={len(folds)} scheme={cfg.cv_scheme}")

    with stage(logger, "cv_train"):
        oof, test_pred, y, fold_info = build_final_matrices(train, test, num_cols, cat_cols, cfg, logger, folds)

    with stage(logger, "oof_report"):
        report_oof(logger, train, oof, y)

    if cfg.run_sanity_holdout:
        with stage(logger, "sanity_holdout_zip3"):
            sanity_holdout_zip3(cfg, logger, train, num_cols, cat_cols)

    if cfg.mode == "train_submit":
        with stage(logger, "write_submission"):
            write_submission(cfg, logger, test, test_pred)

    logger.info("DONE ✅")

# =========================
# STEP-BY-STEP CONTROL
# =========================
# Recommended step sequence:
# 1) cfg = Config(mode="cache_only"); main(cfg)        # builds external caches (admissions/stays/vitals)
# 2) cfg = Config(mode="analyze_only"); main(cfg)      # adversarial AUC + feature sanity
# 3) cfg = Config(mode="train_submit"); main(cfg)      # CV + holdout + submission

cfg = Config(
    mode="train_submit",
    run_name=f"EDCOST_V3_{int(time.time())}",
    # force_recache_external=True,  # flip to rebuild external caches
    # force_recache_pdf=True,       # flip only if pdf parquet missing
)
main(cfg)


2026-02-13 17:02:22 | INFO | EDCOST_V3_1771020142 | [env_report] START
2026-02-13 17:02:22 | INFO | EDCOST_V3_1771020142 | OS: Windows-11-10.0.26100-SP0
2026-02-13 17:02:22 | INFO | EDCOST_V3_1771020142 | Python: 3.12.0
2026-02-13 17:02:22 | INFO | EDCOST_V3_1771020142 | numpy=2.3.4 pandas=2.3.3
2026-02-13 17:02:22 | INFO | EDCOST_V3_1771020142 | sklearn=1.7.2 xgboost=3.0.0
2026-02-13 17:02:22 | INFO | EDCOST_V3_1771020142 | torch=2.9.0+cu126 cuda_available=True
2026-02-13 17:02:22 | INFO | EDCOST_V3_1771020142 | GPU[0]=NVIDIA GeForce RTX 4060 Laptop GPU capability=(8, 9)
2026-02-13 17:02:22 | INFO | EDCOST_V3_1771020142 | nvidia-smi -L:
2026-02-13 17:02:22 | INFO | EDCOST_V3_1771020142 | GPU 0: NVIDIA GeForce RTX 4060 Laptop GPU (UUID: GPU-9515bdbf-1472-4fa2-9f73-28e1d27eedbf)
2026-02-13 17:02:22 | INFO | EDCOST_V3_1771020142 | [env_report] END
2026-02-13 17:02:22 | INFO | EDCOST_V3_1771020142 | Config: {'mode': 'train_submit', 'run_name': 'EDCOST_V3_1771020142', 'seed': 2026, 'train_

# Iteration 5
- 

In [31]:
# =========================
# EDCOST V4 — ONE CELL PIPELINE (GPU XGBoost, cached PDFs + clinical text)
# =========================

import os, sys, re, json, time, math, random, hashlib, platform, subprocess, logging
from dataclasses import dataclass
from pathlib import Path
from contextlib import contextmanager

import numpy as np
import pandas as pd

# Sparse / ML
from scipy import sparse
from sklearn.model_selection import StratifiedKFold, KFold, GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction import FeatureHasher
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

import xgboost as xgb

# PDFs
import fitz  # PyMuPDF

# Optional GPU probe
try:
    import torch
except Exception:
    torch = None


# -------------------------
# Logging + timing
# -------------------------
def setup_logger(run_name: str, run_dir: Path, level: str = "INFO") -> logging.Logger:
    run_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger(run_name)
    logger.setLevel(getattr(logging, level.upper(), logging.INFO))
    logger.handlers = []

    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S")

    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    logger.addHandler(sh)

    fh = logging.FileHandler(run_dir / "run.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    logger.propagate = False
    return logger


@contextmanager
def stage(logger: logging.Logger, name: str):
    t0 = time.time()
    logger.info(f"[{name}] START")
    try:
        yield
    finally:
        dt = time.time() - t0
        logger.info(f"[{name}] END ({dt:.2f}s)")


def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def env_report(logger: logging.Logger):
    with stage(logger, "env_report"):
        logger.info(f"OS: {platform.platform()}")
        logger.info(f"Python: {sys.version.split()[0]}")
        logger.info(f"numpy={np.__version__} pandas={pd.__version__}")
        try:
            import sklearn
            logger.info(f"sklearn={sklearn.__version__} xgboost={xgb.__version__}")
        except Exception as e:
            logger.info(f"sklearn/xgb version check failed: {e}")
        if torch is not None:
            try:
                logger.info(f"torch={torch.__version__} cuda_available={torch.cuda.is_available()}")
                if torch.cuda.is_available():
                    logger.info(f"GPU[0]={torch.cuda.get_device_name(0)} capability={torch.cuda.get_device_capability(0)}")
            except Exception as e:
                logger.info(f"torch gpu info failed: {e}")
        try:
            out = subprocess.check_output(["nvidia-smi", "-L"], stderr=subprocess.STDOUT, text=True)
            logger.info("nvidia-smi -L:\n" + out.strip())
        except Exception as e:
            logger.info(f"nvidia-smi not available or failed: {e}")


# -------------------------
# XGB progress callback (fold logs every N iters)
# -------------------------
class XGBProgressCallback(xgb.callback.TrainingCallback):
    def __init__(self, logger: logging.Logger, period: int = 200, tag: str = "XGB"):
        self.logger = logger
        self.period = int(period)
        self.tag = tag
        self._t0 = None

    def before_training(self, model):
        self._t0 = time.time()
        return model

    def after_iteration(self, model, epoch: int, evals_log) -> bool:
        it = epoch + 1
        if it == 1 or (self.period > 0 and it % self.period == 0):
            try:
                ds = list(evals_log.keys())[0]
                met = list(evals_log[ds].keys())[0]
                val = evals_log[ds][met][-1]
                self.logger.info(f"[{self.tag}] iter={it} {ds}-{met}={val:.6f}")
            except Exception:
                self.logger.info(f"[{self.tag}] iter={it} (eval log unavailable)")
        return False

    def after_training(self, model):
        if self._t0 is not None:
            self.logger.info(f"[{self.tag}] training_done_in={time.time()-self._t0:.2f}s")
        return model


# -------------------------
# Helpers
# -------------------------
def read_csv_safe(path: Path) -> pd.DataFrame:
    return pd.read_csv(path)

def ensure_zip3_str(df: pd.DataFrame, col: str = "zip3") -> pd.DataFrame:
    if col in df.columns:
        df[col] = df[col].astype(str).str.zfill(3)
    return df

def jloads(x):
    if x is None:
        return {}
    if isinstance(x, dict):
        return x
    if isinstance(x, float) and np.isnan(x):
        return {}
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return {}
    try:
        return json.loads(s)
    except Exception:
        return {}

def clip_preds(pred: np.ndarray, y_train: np.ndarray) -> np.ndarray:
    # keep non-negative and cap at a robust upper bound
    hi = float(np.quantile(y_train, 0.995) * 1.25)
    return np.clip(pred, 0.0, hi)


# -------------------------
# PDF parsing (robust to line-broken tables)
# -------------------------
_CODE_PAT = re.compile(r"[A-Z]?\d{4,5}")
_AMT_PAT = re.compile(r"\d+(?:,\d{3})*\.\d{2}")

def _parse_amount(s: str) -> float:
    return float(s.replace(",", ""))

def parse_receipt_text_to_items(text: str):
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    items = []
    total = None

    i = 0
    while i < len(lines):
        ln = lines[i]

        # TOTAL can be "TOTAL 7,978.28" or "TOTAL" newline "50.00"
        if ln == "TOTAL" or ln.startswith("TOTAL"):
            m = _AMT_PAT.search(ln)
            if m:
                total = _parse_amount(m.group(0))
            else:
                if i + 1 < len(lines) and _AMT_PAT.fullmatch(lines[i + 1]):
                    total = _parse_amount(lines[i + 1])
            break

        if _CODE_PAT.fullmatch(ln):
            code = ln
            i += 1
            desc_parts = []
            while i < len(lines) and (not re.fullmatch(r"\d+", lines[i])) and (not (lines[i] == "TOTAL" or lines[i].startswith("TOTAL"))) and (not _CODE_PAT.fullmatch(lines[i])):
                desc_parts.append(lines[i])
                i += 1

            qty = None
            if i < len(lines) and re.fullmatch(r"\d+", lines[i]):
                qty = int(lines[i])
                i += 1

            unit = None
            line_total = None
            if i < len(lines) and _AMT_PAT.fullmatch(lines[i]):
                unit = _parse_amount(lines[i])
                i += 1
            if i < len(lines) and _AMT_PAT.fullmatch(lines[i]):
                line_total = _parse_amount(lines[i])
                i += 1

            items.append({
                "code": code,
                "desc": " ".join(desc_parts).strip(),
                "qty": qty if qty is not None else 1,
                "unit": unit if unit is not None else 0.0,
                "line_total": line_total if line_total is not None else 0.0
            })
            continue

        i += 1

    return items, total

def extract_pdf_one(pdf_path: Path):
    # returns dict of features; if missing -> pdf_present=0
    feat = {
        "pdf_present": 0,
        "pdf_total": 0.0,
        "pdf_n_items": 0,
        "pdf_n_unique_codes": 0,
        "pdf_line_sum": 0.0,
        "pdf_zip3": "",
        "pdf_insurance": "",
        "pdf_code_counts_json": "{}",
        "pdf_code_costs_json": "{}",
        # group features
        "pdf_em_cnt": 0.0,
        "pdf_em_cost": 0.0,
        "pdf_critical_cnt": 0.0,
        "pdf_critical_cost": 0.0,
        "pdf_lab_cnt": 0.0,
        "pdf_lab_cost": 0.0,
        "pdf_imaging_cnt": 0.0,
        "pdf_imaging_cost": 0.0,
        "pdf_proc_cnt": 0.0,
        "pdf_proc_cost": 0.0,
        "pdf_obs_cnt": 0.0,
        "pdf_obs_cost": 0.0,
        "pdf_other_cnt": 0.0,
        "pdf_other_cost": 0.0,
        # severity-ish
        "pdf_has_intubation": 0.0,
        "pdf_has_cpr": 0.0,
        "pdf_has_central_line": 0.0,
        "pdf_em_level_sum": 0.0,
        "pdf_em_level_max": 0.0,
    }

    if not pdf_path.exists():
        return feat

    try:
        doc = fitz.open(pdf_path)
        text = ""
        for page in doc:
            text += page.get_text("text") + "\n"
        doc.close()
    except Exception:
        return feat

    feat["pdf_present"] = 1

    # Parse ZIP3 and insurance from header if present
    m_zip = re.search(r"ZIP3:\s*([0-9]{3})", text)
    if m_zip:
        feat["pdf_zip3"] = m_zip.group(1)
    m_ins = re.search(r"Insurance:\s*([A-Za-z_]+)", text)
    if m_ins:
        feat["pdf_insurance"] = m_ins.group(1)

    items, total = parse_receipt_text_to_items(text)
    if total is None:
        total = 0.0
    feat["pdf_total"] = float(total)

    if not items:
        return feat

    feat["pdf_n_items"] = int(len(items))
    codes = [it["code"] for it in items]
    feat["pdf_n_unique_codes"] = int(len(set(codes)))
    feat["pdf_line_sum"] = float(sum(float(it.get("line_total", 0.0) or 0.0) for it in items))

    # Aggregate per code
    code_counts = {}
    code_costs = {}
    for it in items:
        c = it["code"]
        qty = float(it.get("qty", 1) or 1)
        lt = float(it.get("line_total", 0.0) or 0.0)
        code_counts[c] = code_counts.get(c, 0.0) + qty
        code_costs[c] = code_costs.get(c, 0.0) + lt

    feat["pdf_code_counts_json"] = json.dumps(code_counts, ensure_ascii=False)
    feat["pdf_code_costs_json"] = json.dumps(code_costs, ensure_ascii=False)

    # Grouping logic (cheap but high signal)
    em_levels = {"99281":1, "99282":2, "99283":3, "99284":4, "99285":5}
    proc_codes = {"31500", "36556", "36620", "92950"}  # intubation/central line/art line/CPR
    for code, cnt in code_counts.items():
        cost = code_costs.get(code, 0.0)

        if code in em_levels:
            lvl = em_levels[code]
            feat["pdf_em_cnt"] += cnt
            feat["pdf_em_cost"] += cost
            feat["pdf_em_level_sum"] += lvl * cnt
            feat["pdf_em_level_max"] = max(feat["pdf_em_level_max"], float(lvl))
        elif code in ("99291", "99292"):
            feat["pdf_critical_cnt"] += cnt
            feat["pdf_critical_cost"] += cost
        elif code.startswith("8"):
            feat["pdf_lab_cnt"] += cnt
            feat["pdf_lab_cost"] += cost
        elif code.startswith("7"):
            feat["pdf_imaging_cnt"] += cnt
            feat["pdf_imaging_cost"] += cost
        elif code == "G0378":
            feat["pdf_obs_cnt"] += cnt
            feat["pdf_obs_cost"] += cost
        elif code in proc_codes:
            feat["pdf_proc_cnt"] += cnt
            feat["pdf_proc_cost"] += cost
        else:
            feat["pdf_other_cnt"] += cnt
            feat["pdf_other_cost"] += cost

    # Explicit severe procedure flags
    feat["pdf_has_intubation"] = 1.0 if ("31500" in code_counts) else 0.0
    feat["pdf_has_cpr"] = 1.0 if ("92950" in code_counts) else 0.0
    feat["pdf_has_central_line"] = 1.0 if ("36556" in code_counts) else 0.0

    return feat


def build_or_load_pdf_cache(logger: logging.Logger, patient_ids: np.ndarray, pdf_dir: Path, cache_path: Path, force: bool = False) -> pd.DataFrame:
    with stage(logger, "pdf_cache"):
        if cache_path.exists() and not force:
            df = pd.read_parquet(cache_path)
            logger.info(f"[PDF] Loaded cache: {cache_path} rows={len(df)} cols={list(df.columns)[:12]}...")
        else:
            pdf_dir = Path(pdf_dir)
            rows = []
            found = 0
            for pid in patient_ids:
                pdf_path = pdf_dir / f"receipt_{int(pid)}.pdf"
                feat = extract_pdf_one(pdf_path)
                feat["patient_id"] = int(pid)
                rows.append(feat)
                found += int(feat["pdf_present"])
            df = pd.DataFrame(rows)
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            df.to_parquet(cache_path, index=False)
            logger.info(f"[PDF] Cache written: {cache_path} rows={len(df)} found={found} missing={len(df)-found}")

        # Ensure required columns exist
        if "pdf_present" not in df.columns:
            df["pdf_present"] = 0
        df["pdf_present"] = df["pdf_present"].fillna(0).astype(int)

        # log found/missing explicitly
        found = int(df["pdf_present"].sum())
        logger.info(f"[PDF] Expected={len(patient_ids)} Found={found} Missing={len(patient_ids)-found}")

        return df


# -------------------------
# External features: admissions/stays/vitals + NOTES TEXT (key change)
# -------------------------
def build_or_load_external_cache(
    logger: logging.Logger,
    all_patient_ids: np.ndarray,
    cache_path: Path,
    admissions_train_csv: Path,
    admissions_test_csv: Path,
    discharge_notes_json: Path,
    stays_train_csv: Path,
    stays_test_csv: Path,
    vitals_json: Path,
    force: bool = False
) -> pd.DataFrame:
    with stage(logger, "external_cache"):
        if cache_path.exists() and not force:
            df = pd.read_parquet(cache_path)
            logger.info(f"[EXT] Loaded cache: {cache_path} rows={len(df)} cols={list(df.columns)[:12]}...")
            return df

        # Build from scratch (robust to missing files)
        out = pd.DataFrame({"patient_id": all_patient_ids.astype(int)})

        # ---- Admissions + discharge notes ----
        adm_ok = admissions_train_csv.exists() and admissions_test_csv.exists()
        notes_ok = discharge_notes_json.exists()

        if adm_ok:
            adm_tr = read_csv_safe(admissions_train_csv)
            adm_te = read_csv_safe(admissions_test_csv)
            adm = pd.concat([adm_tr, adm_te], axis=0, ignore_index=True)
            # drop labels if present
            if "readmit_30d" in adm.columns:
                adm = adm.drop(columns=["readmit_30d"])

            # numeric aggregates
            g = adm.groupby("patient_id")
            adm_feat = pd.DataFrame({
                "patient_id": g.size().index.astype(int),
                "adm_n": g.size().astype(float).values,
                "adm_los_mean": g["los_days"].mean().values if "los_days" in adm.columns else 0.0,
                "adm_los_max": g["los_days"].max().values if "los_days" in adm.columns else 0.0,
                "adm_charlson_mean": g["charlson_band"].mean().values if "charlson_band" in adm.columns else 0.0,
                "adm_charlson_max": g["charlson_band"].max().values if "charlson_band" in adm.columns else 0.0,
                "adm_edvis6m_mean": g["ed_visits_6m"].mean().values if "ed_visits_6m" in adm.columns else 0.0,
                "adm_edvis6m_max": g["ed_visits_6m"].max().values if "ed_visits_6m" in adm.columns else 0.0,
                "adm_emerg_rate": (g["acuity_emergent"].mean().values if "acuity_emergent" in adm.columns else 0.0),
            })

            # counts by primary_dx
            if "primary_dx" in adm.columns:
                dx_ct = pd.crosstab(adm["patient_id"], adm["primary_dx"])
                dx_ct = dx_ct.add_prefix("adm_dx_").reset_index().rename(columns={"patient_id":"patient_id"})
                adm_feat = adm_feat.merge(dx_ct, on="patient_id", how="left")

            # discharge note text aggregation
            discharge_text = None
            if notes_ok and ("admission_id" in adm.columns):
                try:
                    notes = json.loads(Path(discharge_notes_json).read_text(encoding="utf-8"))
                    notes_df = pd.DataFrame(notes)
                    notes_df["admission_id"] = notes_df["admission_id"].astype(int)
                    adm2 = adm.merge(notes_df, on="admission_id", how="left")
                    # aggregate to patient
                    discharge_text = adm2.groupby("patient_id")["note"].apply(
                        lambda s: " ".join([str(x) for x in s.dropna().tolist()])[:50000]
                    ).reset_index().rename(columns={"note":"discharge_text"})
                except Exception as e:
                    logger.info(f"[EXT] discharge_notes parse failed: {e}")

            if discharge_text is None:
                discharge_text = pd.DataFrame({"patient_id": adm_feat["patient_id"], "discharge_text": ""})

            # merge
            out = out.merge(adm_feat, on="patient_id", how="left")
            out = out.merge(discharge_text, on="patient_id", how="left")
            logger.info(f"[EXT] admissions features built rows={len(adm_feat)}")
        else:
            out["discharge_text"] = ""
            logger.info("[EXT] admissions files missing; skipping admissions/discharge features")

        # ---- Stays + vitals ----
        stays_ok = stays_train_csv.exists() and stays_test_csv.exists()
        vit_ok = vitals_json.exists()

        if stays_ok:
            st_tr = read_csv_safe(stays_train_csv)
            st_te = read_csv_safe(stays_test_csv)
            st = pd.concat([st_tr, st_te], axis=0, ignore_index=True)
            if "discharge_ready_day11" in st.columns:
                st = st.drop(columns=["discharge_ready_day11"])

            gs = st.groupby("patient_id")
            st_feat = pd.DataFrame({
                "patient_id": gs.size().index.astype(int),
                "stay_n": gs.size().astype(float).values,
            })

            if "unit_type" in st.columns:
                ut = pd.crosstab(st["patient_id"], st["unit_type"]).add_prefix("stay_unit_").reset_index()
                ut = ut.rename(columns={"patient_id":"patient_id"})
                st_feat = st_feat.merge(ut, on="patient_id", how="left")

            if "admission_reason" in st.columns:
                ar = pd.crosstab(st["patient_id"], st["admission_reason"]).add_prefix("stay_reason_").reset_index()
                ar = ar.rename(columns={"patient_id":"patient_id"})
                st_feat = st_feat.merge(ar, on="patient_id", how="left")

            out = out.merge(st_feat, on="patient_id", how="left")
            logger.info(f"[EXT] stays features built rows={len(st_feat)}")
        else:
            logger.info("[EXT] stays files missing; skipping stays features")

        # vitals JSON -> patient aggregates + vitals_text
        if stays_ok and vit_ok:
            try:
                vit = json.loads(Path(vitals_json).read_text(encoding="utf-8"))
                vit_df = []
                for obj in vit:
                    sid = int(obj.get("stay_id"))
                    days = obj.get("days", [])
                    if not days:
                        continue
                    # numeric stats per stay
                    def arr(k):
                        vals = [d.get(k) for d in days if d.get(k) is not None]
                        return np.array(vals, dtype=np.float32) if vals else np.array([], dtype=np.float32)

                    hr = arr("hr"); sbp = arr("sbp"); dbp = arr("dbp"); temp = arr("temp_c"); rr = arr("rr")

                    note_txt = " ".join([str(d.get("note","")) for d in days if d.get("note")])[:50000]

                    row = {
                        "stay_id": sid,
                        "vit_hr_mean": float(hr.mean()) if hr.size else 0.0,
                        "vit_hr_max": float(hr.max()) if hr.size else 0.0,
                        "vit_hr_min": float(hr.min()) if hr.size else 0.0,
                        "vit_sbp_mean": float(sbp.mean()) if sbp.size else 0.0,
                        "vit_sbp_min": float(sbp.min()) if sbp.size else 0.0,
                        "vit_temp_mean": float(temp.mean()) if temp.size else 0.0,
                        "vit_rr_mean": float(rr.mean()) if rr.size else 0.0,
                        "vitals_text": note_txt,
                    }
                    vit_df.append(row)

                vit_df = pd.DataFrame(vit_df)
                st_map = st[["stay_id","patient_id"]].copy()
                st_map["stay_id"] = st_map["stay_id"].astype(int)
                vit_df = vit_df.merge(st_map, on="stay_id", how="left")

                gp = vit_df.groupby("patient_id")
                vit_feat = gp[[
                    "vit_hr_mean","vit_hr_max","vit_hr_min","vit_sbp_mean","vit_sbp_min","vit_temp_mean","vit_rr_mean"
                ]].mean().reset_index()

                vit_text = gp["vitals_text"].apply(lambda s: " ".join([str(x) for x in s.dropna().tolist()])[:80000]).reset_index()
                out = out.merge(vit_feat, on="patient_id", how="left")
                out = out.merge(vit_text, on="patient_id", how="left")
                logger.info(f"[EXT] vitals features built rows={len(vit_feat)}")
            except Exception as e:
                logger.info(f"[EXT] vitals parse failed: {e}")
        else:
            if "vitals_text" not in out.columns:
                out["vitals_text"] = ""
            logger.info("[EXT] vitals files missing; skipping vitals features")

        # fill missing text columns
        if "discharge_text" not in out.columns:
            out["discharge_text"] = ""
        if "vitals_text" not in out.columns:
            out["vitals_text"] = ""

        # presence indicators
        out["adm_present"] = (out.get("adm_n", 0).fillna(0) > 0).astype(int)
        out["stay_present"] = (out.get("stay_n", 0).fillna(0) > 0).astype(int)

        # cache
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        out.to_parquet(cache_path, index=False)
        logger.info(f"[EXT] Cache written: {cache_path} rows={len(out)}")
        return out


# -------------------------
# CV stratification
# -------------------------
def make_strat_labels(df_train: pd.DataFrame, y: np.ndarray, cost_bins: int, scheme: str) -> np.ndarray:
    # Force pandas Series so we always have consistent methods
    bins = pd.qcut(pd.Series(y), q=cost_bins, labels=False, duplicates="drop")
    bins = bins.fillna(0).astype(int).astype(str).to_numpy()

    chronic = df_train["primary_chronic"].astype(str).to_numpy()
    pdfp = df_train.get(
        "pdf_present",
        pd.Series(np.zeros(len(df_train), dtype=int), index=df_train.index)
    ).fillna(0).astype(int).astype(str).to_numpy()

    if scheme == "chronic_x_costbin_x_pdf":
        # chronic + "_" + bins + "_" + pdfp
        return np.char.add(np.char.add(np.char.add(chronic, "_"), bins), np.char.add("_", pdfp))
    elif scheme == "chronic_x_costbin":
        return np.char.add(np.char.add(chronic, "_"), bins)
    else:
        return chronic


# -------------------------
# Diagnostics
# -------------------------
def log_mae_reports(logger: logging.Logger, df_train: pd.DataFrame, y_true: np.ndarray, oof_pred: np.ndarray, title: str):
    mae = mean_absolute_error(y_true, oof_pred)
    ae = np.abs(y_true - oof_pred)
    logger.info(f"[{title}] MAE={mae:.6f} | median_AE={float(np.median(ae)):.3f} | p90_AE={float(np.quantile(ae,0.9)):.3f}")

    # deciles of y_true
    dec = pd.qcut(y_true, q=10, labels=False, duplicates="drop")
    rep = []
    for d in sorted(dec.unique()):
        idx = np.where(dec == d)[0]
        rep.append({
            "dec": int(d),
            "count": int(len(idx)),
            "mae": float(np.mean(ae[idx])),
            "sum_ae": float(np.sum(ae[idx])),
            "y_mean": float(np.mean(y_true[idx])),
            "y_p90": float(np.quantile(y_true[idx], 0.9)) if len(idx) else 0.0,
        })
    rep_df = pd.DataFrame(rep)
    rep_df["share_ae"] = rep_df["sum_ae"] / rep_df["sum_ae"].sum()
    logger.info(f"[{title}] MAE by target decile:\n{rep_df.to_string(index=False)}")

    # segments
    for col in ["primary_chronic", "insurance", "zip3", "pdf_present"]:
        if col in df_train.columns:
            seg = df_train.copy()
            seg["_ae"] = ae
            g = seg.groupby(col)["_ae"].agg(["count", "mean", "sum"]).sort_values("count", ascending=False).head(15)
            g = g.rename(columns={"mean":"mae","sum":"sum_ae"})
            logger.info(f"[{title}] MAE by {col} (top 15 by count):\n{g.to_string()}")


# -------------------------
# Adversarial validation
# -------------------------
def adversarial_validation(logger: logging.Logger, X_train_sparse, X_test_sparse, seed: int):
    with stage(logger, "adversarial_validation"):
        X_adv = sparse.vstack([X_train_sparse, X_test_sparse], format="csr")
        y_adv = np.array([0]*X_train_sparse.shape[0] + [1]*X_test_sparse.shape[0], dtype=np.int32)

        # small classifier
        params = dict(
            n_estimators=2000,
            learning_rate=0.05,
            max_depth=4,
            min_child_weight=10.0,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=2.0,
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            device="cuda:0",
            n_jobs=0,
            random_state=seed,
            early_stopping_rounds=100,
        )
        logger.info(f"[ADV] XGBClassifier params={params}")

        # simple split
        n = X_train_sparse.shape[0]
        idx = np.arange(X_adv.shape[0])
        rng = np.random.default_rng(seed)
        rng.shuffle(idx)
        cut = int(0.8 * len(idx))
        tr_i, va_i = idx[:cut], idx[cut:]

        clf = xgb.XGBClassifier(
            **params,
            callbacks=[XGBProgressCallback(logger, period=250, tag="ADV")]
        )
        try:
            clf.fit(X_adv[tr_i], y_adv[tr_i], eval_set=[(X_adv[va_i], y_adv[va_i])], verbose=False)
        except TypeError:
            # fallback if callback/early stop signature mismatch
            clf = xgb.XGBClassifier(**{k:v for k,v in params.items() if k != "early_stopping_rounds"})
            clf.fit(X_adv[tr_i], y_adv[tr_i], eval_set=[(X_adv[va_i], y_adv[va_i])],
                    early_stopping_rounds=params["early_stopping_rounds"], verbose=False)

        p = clf.predict_proba(X_adv[va_i])[:,1]
        auc = roc_auc_score(y_adv[va_i], p)
        logger.info(f"[ADV] AUC(train vs test)={auc:.6f} (0.50=no shift, >0.65=notable shift)")
        try:
            cfg = clf.get_booster().save_config()
            logger.info(f"[ADV] booster config contains 'cuda': {'cuda' in cfg.lower()}")
        except Exception:
            pass


# -------------------------
# Main training (GPU XGB MAE) with NOTE TEXT + CODE HASH SPARSE
# -------------------------
def train_xgb_cv(
    logger: logging.Logger,
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    y: np.ndarray,
    base_num_sparse_train,
    base_num_sparse_test,
    cat_sparse_train,
    cat_sparse_test,
    code_sparse_train,
    code_sparse_test,
    notes_text_train: np.ndarray,
    notes_text_test: np.ndarray,
    seed: int,
    n_folds: int,
    cv_scheme: str,
    cost_bins: int,
    xgb_params: dict,
    text_svd_dim: int,
    text_max_features: int,
    text_ngram_range=(3,5),
    text_analyzer="char_wb",
):
    with stage(logger, "cv_train"):
        # Strat labels
        strat = make_strat_labels(df_train, y, cost_bins=cost_bins, scheme=cv_scheme)

        try:
            skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
            splits = list(skf.split(df_train, strat))
        except Exception as e:
            logger.info(f"[CV] StratifiedKFold failed ({e}); falling back to KFold.")
            kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
            splits = list(kf.split(df_train))

        logger.info(f"[CV] folds={len(splits)} scheme={cv_scheme}")

        oof = np.zeros(len(df_train), dtype=np.float32)
        pred_test = np.zeros(len(df_test), dtype=np.float32)
        fold_maes = []

        # Pre-stack sparse parts that don't depend on text fitting
        base_train = sparse.hstack([base_num_sparse_train, cat_sparse_train, code_sparse_train], format="csr")
        base_test  = sparse.hstack([base_num_sparse_test,  cat_sparse_test,  code_sparse_test ], format="csr")
        logger.info(f"[X] base sparse shapes: train={base_train.shape} test={base_test.shape}")

        logger.info(f"[XGB] params={xgb_params}")

        for fold, (tr_idx, va_idx) in enumerate(splits):
            with stage(logger, f"fold_{fold}"):
                # --- Fold-safe text -> TFIDF -> SVD ---
                with stage(logger, f"fold_{fold}/text_fit"):
                    vec = TfidfVectorizer(
                        analyzer=text_analyzer,
                        ngram_range=text_ngram_range,
                        min_df=2,
                        max_df=0.95,
                        max_features=int(text_max_features),
                    )
                    Xtr_txt = vec.fit_transform(notes_text_train[tr_idx])
                    Xva_txt = vec.transform(notes_text_train[va_idx])
                    Xte_txt = vec.transform(notes_text_test)

                    # Choose SVD dim safely
                    n_comp = int(text_svd_dim)
                    # n_comp must be <= min(n_samples-1, n_features-1)
                    max_comp = max(2, min(Xtr_txt.shape[0]-1, Xtr_txt.shape[1]-1))
                    n_comp = min(n_comp, max_comp)

                    svd = TruncatedSVD(n_components=n_comp, random_state=seed + fold)
                    Xtr_emb = svd.fit_transform(Xtr_txt).astype(np.float32)
                    Xva_emb = svd.transform(Xva_txt).astype(np.float32)
                    Xte_emb = svd.transform(Xte_txt).astype(np.float32)

                    logger.info(f"[fold {fold}] text svd_dim={n_comp} | tfidf_features={Xtr_txt.shape[1]}")

                # stack full features
                X_tr = sparse.hstack([base_train[tr_idx], sparse.csr_matrix(Xtr_emb)], format="csr")
                X_va = sparse.hstack([base_train[va_idx], sparse.csr_matrix(Xva_emb)], format="csr")
                X_te = sparse.hstack([base_test,          sparse.csr_matrix(Xte_emb)], format="csr")

                logger.info(f"[fold {fold}] X shapes tr={X_tr.shape} va={X_va.shape} te={X_te.shape}")

                # train model with REQUIRED early stopping pattern
                with stage(logger, f"fold_{fold}/train_xgb"):
                    params_fold = dict(xgb_params)
                    params_fold["random_state"] = seed + fold

                    try:
                        model = xgb.XGBRegressor(
                            **params_fold,
                            callbacks=[XGBProgressCallback(logger, period=200, tag=f"XGB_f{fold}")]
                        )
                        model.fit(X_tr, y[tr_idx], eval_set=[(X_va, y[va_idx])], verbose=False)
                    except TypeError as e:
                        # fallback guard requested
                        logger.info(f"[fold {fold}] XGB fit TypeError fallback: {e}")
                        # move early_stopping_rounds to fit if needed
                        esr = params_fold.pop("early_stopping_rounds", None)
                        model = xgb.XGBRegressor(**params_fold)
                        model.fit(
                            X_tr, y[tr_idx],
                            eval_set=[(X_va, y[va_idx])],
                            early_stopping_rounds=esr if esr is not None else 200,
                            verbose=False
                        )

                # GPU verification logs
                try:
                    gp = model.get_params()
                    logger.info(f"[fold {fold}] XGB get_params device={gp.get('device')} tree_method={gp.get('tree_method')}")
                    cfg = model.get_booster().save_config()
                    logger.info(f"[fold {fold}] XGB booster config contains 'cuda': {'cuda' in cfg.lower()}")
                except Exception:
                    pass

                p_va = model.predict(X_va).astype(np.float32)
                p_te = model.predict(X_te).astype(np.float32)

                # clip
                p_va = clip_preds(p_va, y)
                p_te = clip_preds(p_te, y)

                oof[va_idx] = p_va
                pred_test += p_te / len(splits)

                fold_mae = mean_absolute_error(y[va_idx], p_va)
                fold_maes.append(fold_mae)
                best_it = getattr(model, "best_iteration", None)
                logger.info(f"[fold {fold}] MAE={fold_mae:.6f} best_iteration={best_it}")

        logger.info(f"[OOF] base MAE={mean_absolute_error(y, oof):.6f} | fold_maes={[f'{m:.3f}' for m in fold_maes]}")
        return oof, pred_test


# -------------------------
# Build feature matrices (numeric + categorical + code-hash)
# -------------------------
def build_base_matrices(logger, df_all: pd.DataFrame, n_train: int):
    with stage(logger, "build_base_matrices"):
        df = df_all.copy()

        # Derived numeric features (cheap + important)
        if "prior_ed_cost_5y_usd" in df.columns:
            df["prior_cost_log1p"] = np.log1p(df["prior_ed_cost_5y_usd"].astype(float).fillna(0.0))
        if "prior_ed_visits_5y" in df.columns:
            df["prior_visits_log1p"] = np.log1p(df["prior_ed_visits_5y"].astype(float).fillna(0.0))
        if "prior_ed_cost_5y_usd" in df.columns and "prior_ed_visits_5y" in df.columns:
            df["prior_cost_per_visit"] = df["prior_ed_cost_5y_usd"].astype(float).fillna(0.0) / (df["prior_ed_visits_5y"].astype(float).fillna(0.0) + 1.0)

        # pdf interaction helpers
        if "pdf_present" in df.columns:
            df["pdf_present"] = df["pdf_present"].fillna(0).astype(int)
            df["prior_cost_x_pdf"] = df["prior_ed_cost_5y_usd"].astype(float).fillna(0.0) * df["pdf_present"]

        # Identify text/json columns we don't want in numeric/cat
        text_cols = [c for c in ["discharge_text", "vitals_text"] if c in df.columns]
        json_cols = [c for c in df.columns if c.endswith("_json") or c.endswith("_jsons") or "code_counts_json" in c or "code_costs_json" in c]

        # categorical
        cat_cols = []
        for c in ["primary_chronic", "sex", "insurance", "zip3", "pdf_zip3", "pdf_insurance"]:
            if c in df.columns:
                cat_cols.append(c)

        # numeric columns: all numeric excluding ids/target
        drop_cols = set(["patient_id", "ed_cost_next3y_usd"]) | set(text_cols) | set(json_cols)
        num_cols = [c for c in df.columns if c not in drop_cols and pd.api.types.is_numeric_dtype(df[c])]

        logger.info(f"[Features] num_cols={len(num_cols)} cat_cols={len(cat_cols)} -> {cat_cols}")

        # OneHot categorical (fit on ALL to avoid unknown categories; no target used)
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
        X_cat = ohe.fit_transform(df[cat_cols].fillna("NA")) if cat_cols else sparse.csr_matrix((len(df), 0))
        logger.info(f"[OHE] shape={X_cat.shape}")

        # Numeric sparse
        X_num = sparse.csr_matrix(df[num_cols].fillna(0.0).astype(np.float32).values)
        logger.info(f"[NUM] shape={X_num.shape}")

        # Split
        X_num_tr, X_num_te = X_num[:n_train], X_num[n_train:]
        X_cat_tr, X_cat_te = X_cat[:n_train], X_cat[n_train:]

        return (X_num_tr, X_num_te, X_cat_tr, X_cat_te, num_cols, cat_cols)


def build_code_hash_matrices(logger, df_all: pd.DataFrame, n_train: int, n_features: int = 32768):
    with stage(logger, "code_hash_build"):
        # detect expected json columns
        counts_col = "pdf_code_counts_json" if "pdf_code_counts_json" in df_all.columns else None
        costs_col  = "pdf_code_costs_json"  if "pdf_code_costs_json"  in df_all.columns else None

        if counts_col is None:
            logger.info("[CodeHash] Missing code counts json column; returning empty.")
            X = sparse.csr_matrix((len(df_all), 0))
            return X[:n_train], X[n_train:]

        dicts = []
        for i in range(len(df_all)):
            cnts = jloads(df_all.iloc[i][counts_col])
            costs = jloads(df_all.iloc[i][costs_col]) if costs_col is not None else {}
            total_cost = float(df_all.iloc[i].get("pdf_total", 0.0) or 0.0)
            # feature dict for hashing
            d = {}
            # counts
            for code, v in cnts.items():
                try:
                    vv = float(v)
                except Exception:
                    continue
                d[f"cnt_{code}"] = vv
                d[f"cnt_p2_{code[:2]}"] = d.get(f"cnt_p2_{code[:2]}", 0.0) + vv
                d[f"cnt_p3_{code[:3]}"] = d.get(f"cnt_p3_{code[:3]}", 0.0) + vv

            # costs + shares (key improvement)
            for code, v in costs.items():
                try:
                    vv = float(v)
                except Exception:
                    continue
                d[f"cost_{code}"] = vv
                d[f"cost_p2_{code[:2]}"] = d.get(f"cost_p2_{code[:2]}", 0.0) + vv
                if total_cost > 0:
                    d[f"share_{code}"] = vv / total_cost

            dicts.append(d)

        hasher = FeatureHasher(n_features=int(n_features), input_type="dict", alternate_sign=False)
        X = hasher.transform(dicts).tocsr().astype(np.float32)
        nnz = int(X.nnz)
        logger.info(f"[CodeHash] csr shapes all={X.shape} nnz={nnz}")
        return X[:n_train], X[n_train:]


# -------------------------
# Sanity holdout by zip3 groups
# -------------------------
def sanity_holdout_zip3(
    logger: logging.Logger,
    df_train: pd.DataFrame,
    y: np.ndarray,
    base_num_tr, base_cat_tr, code_tr,
    notes_text_train: np.ndarray,
    seed: int,
    test_size: float = 0.2,
    inner_folds: int = 3,
    xgb_params: dict = None,
    text_svd_dim: int = 64,
    text_max_features: int = 50000,
):
    with stage(logger, "sanity_holdout_zip3"):
        if "zip3" not in df_train.columns:
            logger.info("[Holdout] zip3 not found; skipping.")
            return

        gss = GroupShuffleSplit(n_splits=1, test_size=float(test_size), random_state=seed)
        groups = df_train["zip3"].astype(str).values
        tr_idx, ho_idx = next(gss.split(df_train, y, groups=groups))
        logger.info(f"[Holdout] zip3-group holdout: train_in={len(tr_idx)} holdout={len(ho_idx)}")
        logger.info(f"[Holdout] unique zip3 train_in={len(set(groups[tr_idx]))} holdout={len(set(groups[ho_idx]))}")

        # stack base sparse
        base_tr = sparse.hstack([base_num_tr, base_cat_tr, code_tr], format="csr")

        # inner CV on train_in to build holdout predictions
        kf = KFold(n_splits=int(inner_folds), shuffle=True, random_state=seed)
        ho_pred = np.zeros(len(ho_idx), dtype=np.float32)

        for f, (a, b) in enumerate(kf.split(tr_idx)):
            with stage(logger, f"holdout_cv_fold_{f}"):
                tr2 = tr_idx[a]
                va2 = tr_idx[b]

                # fit text on tr2
                vec = TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=(3,5),
                    min_df=2,
                    max_df=0.95,
                    max_features=int(text_max_features),
                )
                Xtr_txt = vec.fit_transform(notes_text_train[tr2])
                Xva_txt = vec.transform(notes_text_train[va2])
                Xho_txt = vec.transform(notes_text_train[ho_idx])

                max_comp = max(2, min(Xtr_txt.shape[0]-1, Xtr_txt.shape[1]-1))
                n_comp = min(int(text_svd_dim), max_comp)
                svd = TruncatedSVD(n_components=n_comp, random_state=seed + 100 + f)

                Xtr_emb = svd.fit_transform(Xtr_txt).astype(np.float32)
                Xva_emb = svd.transform(Xva_txt).astype(np.float32)
                Xho_emb = svd.transform(Xho_txt).astype(np.float32)

                X_tr = sparse.hstack([base_tr[tr2], sparse.csr_matrix(Xtr_emb)], format="csr")
                X_va = sparse.hstack([base_tr[va2], sparse.csr_matrix(Xva_emb)], format="csr")
                X_ho = sparse.hstack([base_tr[ho_idx], sparse.csr_matrix(Xho_emb)], format="csr")

                params_fold = dict(xgb_params)
                params_fold["random_state"] = seed + 1000 + f

                model = xgb.XGBRegressor(
                    **params_fold,
                    callbacks=[XGBProgressCallback(logger, period=400, tag=f"HOLD_XGB_f{f}")]
                )
                model.fit(X_tr, y[tr2], eval_set=[(X_va, y[va2])], verbose=False)

                p = model.predict(X_ho).astype(np.float32)
                p = clip_preds(p, y)
                ho_pred += p / inner_folds

        mae = mean_absolute_error(y[ho_idx], ho_pred)
        ae = np.abs(y[ho_idx] - ho_pred)
        logger.info(f"[HOLDOUT] MAE={mae:.6f} | median_AE={float(np.median(ae)):.3f} | p90_AE={float(np.quantile(ae,0.9)):.3f}")


# -------------------------
# MAIN
# -------------------------
def main(CONFIG: dict):
    # run dir
    ts = time.strftime("%Y%m%d_%H%M%S")
    run_name = CONFIG.get("run_name", f"EDCOST_V4_{ts}")
    run_dir = Path(CONFIG["cache_dir"]) / "runs_v4_onecell" / f"{run_name}_{ts}"
    logger = setup_logger(run_name, run_dir, CONFIG.get("log_level", "INFO"))
    set_all_seeds(int(CONFIG.get("seed", 2026)))

    env_report(logger)
    logger.info(f"Config: {CONFIG}")

    # paths
    train_csv = Path(CONFIG["train_csv"])
    test_csv  = Path(CONFIG["test_csv"])
    patients_csv = Path(CONFIG["patients_csv"])
    pdf_dir = Path(CONFIG["pdf_dir"])

    # external paths
    admissions_train_csv = Path(CONFIG.get("admissions_train_csv", ""))
    admissions_test_csv  = Path(CONFIG.get("admissions_test_csv", ""))
    discharge_notes_json = Path(CONFIG.get("discharge_notes_json", ""))
    stays_train_csv = Path(CONFIG.get("stays_train_csv", ""))
    stays_test_csv  = Path(CONFIG.get("stays_test_csv", ""))
    vitals_json = Path(CONFIG.get("vitals_json", ""))

    # caches
    cache_dir = Path(CONFIG["cache_dir"])
    pdf_cache_path = cache_dir / "pdf_features_v4.parquet"
    ext_cache_path = cache_dir / "external_patient_features_v4.parquet"

    mode = CONFIG.get("mode", "train_submit")

    # Load core
    with stage(logger, "load_prepare"):
        train = read_csv_safe(train_csv)
        test  = read_csv_safe(test_csv)
        logger.info(f"train shape={train.shape} test shape={test.shape}")

        patients = read_csv_safe(patients_csv)
        patients = ensure_zip3_str(patients, "zip3")

        train = train.merge(patients, on="patient_id", how="left")
        test  = test.merge(patients, on="patient_id", how="left")

        # ensure zip3 string
        train = ensure_zip3_str(train, "zip3")
        test  = ensure_zip3_str(test, "zip3")

    # Combine ids
    all_ids = pd.concat([train["patient_id"], test["patient_id"]], axis=0).astype(int).values
    n_train = len(train)

    # PDF cache v4
    pdf_df = build_or_load_pdf_cache(
        logger=logger,
        patient_ids=all_ids,
        pdf_dir=pdf_dir,
        cache_path=pdf_cache_path,
        force=bool(CONFIG.get("force_recache_pdf", False))
    )

    # External cache v4 (includes discharge_text + vitals_text)
    if CONFIG.get("use_external_tables", True):
        ext_df = build_or_load_external_cache(
            logger=logger,
            all_patient_ids=all_ids,
            cache_path=ext_cache_path,
            admissions_train_csv=admissions_train_csv,
            admissions_test_csv=admissions_test_csv,
            discharge_notes_json=discharge_notes_json,
            stays_train_csv=stays_train_csv,
            stays_test_csv=stays_test_csv,
            vitals_json=vitals_json,
            force=bool(CONFIG.get("force_recache_external", False))
        )
    else:
        ext_df = pd.DataFrame({"patient_id": all_ids.astype(int), "discharge_text":"", "vitals_text":""})

    # Merge all features
    with stage(logger, "merge_all_features"):
        all_df = pd.concat([train, test], axis=0, ignore_index=True)
        all_df = all_df.merge(pdf_df, on="patient_id", how="left")
        all_df = all_df.merge(ext_df, on="patient_id", how="left")

        # fix pdf_present if missing
        if "pdf_present" not in all_df.columns:
            all_df["pdf_present"] = 0
        all_df["pdf_present"] = all_df["pdf_present"].fillna(0).astype(int)

        # Optional sanity check: pdf_total should match prior cost when pdf is present (dataset design)
        if "pdf_total" in all_df.columns and "prior_ed_cost_5y_usd" in all_df.columns:
            diff = (all_df.loc[:n_train-1, "prior_ed_cost_5y_usd"].astype(float) - all_df.loc[:n_train-1, "pdf_total"].astype(float)).abs()
            diff = diff[all_df.loc[:n_train-1, "pdf_present"] == 1]
            if len(diff) > 0:
                logger.info(f"[PDF sanity] |prior_ed_cost_5y_usd - pdf_total| (pdf_present=1) describe:\n{diff.describe()}")

    # If cache_only mode, stop here
    if mode == "cache_only":
        logger.info("MODE=cache_only done ✅")
        return

    # Build base matrices
    X_num_tr, X_num_te, X_cat_tr, X_cat_te, num_cols, cat_cols = build_base_matrices(logger, all_df, n_train=n_train)

    # Build code hash matrices (sparse)
    X_code_tr, X_code_te = build_code_hash_matrices(
        logger, all_df, n_train=n_train, n_features=int(CONFIG.get("code_hash_n_features", 32768))
    )

    # Notes text (NEW major signal): patient-level combined text
    with stage(logger, "build_notes_text"):
        # combine discharge + vitals notes
        t_dis = all_df.get("discharge_text", pd.Series([""]*len(all_df))).fillna("").astype(str)
        t_vit = all_df.get("vitals_text", pd.Series([""]*len(all_df))).fillna("").astype(str)
        notes_all = (t_dis + " " + t_vit).str.slice(0, 120000).values
        notes_tr = notes_all[:n_train]
        notes_te = notes_all[n_train:]
        logger.info(f"[NOTES] train nonempty={(np.array([len(s.strip()) for s in notes_tr])>0).sum()} / {len(notes_tr)}")
        logger.info(f"[NOTES] test  nonempty={(np.array([len(s.strip()) for s in notes_te])>0).sum()} / {len(notes_te)}")

    # Train target
    y = train["ed_cost_next3y_usd"].astype(float).values

    # Optional adversarial on base sparse only (fast)
    if CONFIG.get("run_adversarial", True):
        X_adv_tr = sparse.hstack([X_num_tr, X_cat_tr, X_code_tr], format="csr")
        X_adv_te = sparse.hstack([X_num_te, X_cat_te, X_code_te], format="csr")
        adversarial_validation(logger, X_adv_tr, X_adv_te, seed=int(CONFIG.get("seed", 2026)))

    # XGB params (your required early stopping pattern)
    SEED = int(CONFIG.get("seed", 2026))
    XGB_N_ESTIMATORS = int(CONFIG.get("xgb_n_estimators", 8000))
    XGB_EARLY_STOPPING = int(CONFIG.get("xgb_early_stopping", 400))

    xgb_params = dict(
        n_estimators=XGB_N_ESTIMATORS,
        learning_rate=0.03,
        max_depth=7,
        min_child_weight=20.0,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=2.0,
        reg_alpha=0.0,
        objective="reg:absoluteerror",
        eval_metric="mae",
        tree_method="hist",
        device="cuda:0",
        random_state=SEED,
        n_jobs=0,
        early_stopping_rounds=XGB_EARLY_STOPPING,
    )

    # CV train + predict
    oof, pred_test = train_xgb_cv(
        logger=logger,
        df_train=all_df.iloc[:n_train].copy(),
        df_test=all_df.iloc[n_train:].copy(),
        y=y,
        base_num_sparse_train=X_num_tr,
        base_num_sparse_test=X_num_te,
        cat_sparse_train=X_cat_tr,
        cat_sparse_test=X_cat_te,
        code_sparse_train=X_code_tr,
        code_sparse_test=X_code_te,
        notes_text_train=notes_tr,
        notes_text_test=notes_te,
        seed=SEED,
        n_folds=int(CONFIG.get("n_folds", 5)),
        cv_scheme=str(CONFIG.get("cv_scheme", "chronic_x_costbin_x_pdf")),
        cost_bins=int(CONFIG.get("cost_bins", 5)),
        xgb_params=xgb_params,
        text_svd_dim=int(CONFIG.get("note_svd_dim", 64)),
        text_max_features=int(CONFIG.get("note_max_features", 50000)),
    )

    # Reports
    with stage(logger, "oof_report"):
        df_tr = all_df.iloc[:n_train].copy()
        log_mae_reports(logger, df_tr, y, oof, title="OOF")

    # Sanity holdout
    if CONFIG.get("run_sanity_holdout", True):
        sanity_holdout_zip3(
            logger=logger,
            df_train=all_df.iloc[:n_train].copy(),
            y=y,
            base_num_tr=X_num_tr,
            base_cat_tr=X_cat_tr,
            code_tr=X_code_tr,
            notes_text_train=notes_tr,
            seed=SEED,
            test_size=float(CONFIG.get("holdout_zip3_test_size", 0.2)),
            inner_folds=int(CONFIG.get("holdout_inner_folds", 3)),
            xgb_params=xgb_params,
            text_svd_dim=int(CONFIG.get("note_svd_dim", 64)),
            text_max_features=int(CONFIG.get("note_max_features", 50000)),
        )

    # Write submission (match test order EXACTLY)
    out_path = Path(CONFIG["out_submission"])
    with stage(logger, "write_submission"):
        sub = pd.DataFrame({
            "patient_id": test["patient_id"].astype(int).values,
            "ed_cost_next3y_usd": pred_test.astype(float)
        })
        out_path.parent.mkdir(parents=True, exist_ok=True)
        sub.to_csv(out_path, index=False)
        logger.info(f"Submission written: {out_path} rows={len(sub)}")
        logger.info("Submission head:\n" + sub.head().to_string(index=False))

    # Save artifacts
    with stage(logger, "save_artifacts"):
        pd.DataFrame({"patient_id": train["patient_id"].astype(int).values, "y": y, "oof": oof}).to_csv(run_dir / "oof.csv", index=False)
        pd.DataFrame({"patient_id": test["patient_id"].astype(int).values, "pred": pred_test}).to_csv(run_dir / "test_pred.csv", index=False)
        (run_dir / "config.json").write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")

    logger.info("DONE ✅")
    logger.info(f"Run dir: {run_dir}")
    logger.info(f"Submission: {out_path}")


# =========================
# CONFIG (EDIT THESE PATHS)
# =========================
CONFIG = dict(
    mode="train_submit",  # "cache_only" then "train_submit" if you want a 2-step run
    run_name=f"EDCOST_V4",
    seed=2026,

    train_csv=r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv",
    test_csv=r"D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv",
    patients_csv=r"D:\AgentDs\agent_ds_healthcare\patients.csv",
    pdf_dir=r"D:\AgentDs\agent_ds_healthcare\receipts_pdf",
    cache_dir=r"D:\AgentDs\agent_ds_healthcare\cache",
    out_submission=r"D:\AgentDs\agent_ds_healthcare\submission_ICHI_V1.csv",

    # external tables (major new signal is TEXT in notes)
    use_external_tables=True,
    admissions_train_csv=r"D:\AgentDs\agent_ds_healthcare\admissions_train.csv",
    admissions_test_csv=r"D:\AgentDs\agent_ds_healthcare\admissions_test.csv",
    discharge_notes_json=r"D:\AgentDs\agent_ds_healthcare\discharge_notes.json",
    stays_train_csv=r"D:\AgentDs\agent_ds_healthcare\stays_train.csv",
    stays_test_csv=r"D:\AgentDs\agent_ds_healthcare\stays_test.csv",
    vitals_json=r"D:\AgentDs\agent_ds_healthcare\vitals_timeseries.json",

    # caching controls
    force_recache_pdf=False,
    force_recache_external=False,

    # features
    code_hash_n_features=32768,  # sparse + cheap
    note_svd_dim=64,
    note_max_features=50000,

    # CV
    n_folds=5,
    cv_scheme="chronic_x_costbin_x_pdf",
    cost_bins=5,

    # extra validation
    run_sanity_holdout=True,
    holdout_zip3_test_size=0.2,
    holdout_inner_folds=3,
    run_adversarial=True,

    # XGB runtime control
    xgb_n_estimators=8000,
    xgb_early_stopping=400,

    log_level="INFO",
)

main(CONFIG)


2026-02-13 18:29:36 | INFO | EDCOST_V4 | [env_report] START
2026-02-13 18:29:36 | INFO | EDCOST_V4 | OS: Windows-11-10.0.26100-SP0
2026-02-13 18:29:36 | INFO | EDCOST_V4 | Python: 3.12.0
2026-02-13 18:29:36 | INFO | EDCOST_V4 | numpy=2.3.4 pandas=2.3.3
2026-02-13 18:29:36 | INFO | EDCOST_V4 | sklearn=1.7.2 xgboost=3.0.0
2026-02-13 18:29:36 | INFO | EDCOST_V4 | torch=2.9.0+cu126 cuda_available=True
2026-02-13 18:29:36 | INFO | EDCOST_V4 | GPU[0]=NVIDIA GeForce RTX 4060 Laptop GPU capability=(8, 9)
2026-02-13 18:29:36 | INFO | EDCOST_V4 | nvidia-smi -L:
GPU 0: NVIDIA GeForce RTX 4060 Laptop GPU (UUID: GPU-9515bdbf-1472-4fa2-9f73-28e1d27eedbf)
2026-02-13 18:29:36 | INFO | EDCOST_V4 | [env_report] END (0.05s)
2026-02-13 18:29:36 | INFO | EDCOST_V4 | Config: {'mode': 'train_submit', 'run_name': 'EDCOST_V4', 'seed': 2026, 'train_csv': 'D:\\AgentDs\\agent_ds_healthcare\\ed_cost_train.csv', 'test_csv': 'D:\\AgentDs\\agent_ds_healthcare\\ed_cost_test.csv', 'patients_csv': 'D:\\AgentDs\\agent_ds

AttributeError: 'numpy.ndarray' object has no attribute 'unique'

# New Iter

In [32]:
import os, re, sys, subprocess
from pathlib import Path

# ---- EDIT THIS if your notebook is not launched from the project folder ----
PROJECT_DIR = r"D:\AgentDs\agent_ds_healthcare"   # <- based on your traceback
os.chdir(PROJECT_DIR)

pyfile = Path("challenge2_multimodal_ensemble.py")
assert pyfile.exists(), f"Can't find {pyfile} in {os.getcwd()}"

txt = pyfile.read_text(encoding="utf-8")

# -------------------------------------------------------------------------
# PATCH 1: fix entropy + normalized entropy to never produce +/-inf
# -------------------------------------------------------------------------
old_entropy_block = """def entropy_cost(series: pd.Series) -> float:
        total = series.sum()
        if total <= 0:
            return 0.0
        p = series / total
        return float(-(p * np.log(p + 1e-12)).sum())

    ent = cost_by_code.groupby(level=0).apply(entropy_cost)
    feat["pdf_code_cost_entropy"] = ent
    feat["pdf_code_cost_entropy_norm"] = feat["pdf_code_cost_entropy"] / np.log(
        feat["pdf_n_unique_codes"].replace(0, np.nan)
    ).fillna(1)
"""

new_entropy_block = """def entropy_cost(series: pd.Series) -> float:
        \"\"\"Shannon entropy of the *code-level* cost distribution.

        - Defined as 0 when only one code is present.
        - Uses p*log(p) (no +eps inside the log) so p=1 -> entropy=0 exactly.
        \"\"\"
        arr = series.to_numpy(dtype=float)
        total = float(np.sum(arr))
        if total <= 0:
            return 0.0
        p = arr / total
        p = p[p > 0]
        return float(-(p * np.log(p)).sum())

    ent = cost_by_code.groupby(level=0).apply(entropy_cost)
    feat["pdf_code_cost_entropy"] = ent

    # Normalized entropy: H / log(K). For K<=1, define as 0 to avoid division-by-zero -> inf.
    k = feat["pdf_n_unique_codes"].astype(float).clip(lower=1)
    denom = np.log(k).replace(0, np.nan)  # log(1)=0 -> NaN
    feat["pdf_code_cost_entropy_norm"] = (feat["pdf_code_cost_entropy"] / denom).replace([np.inf, -np.inf], np.nan).fillna(0.0)
"""

if old_entropy_block in txt:
    txt = txt.replace(old_entropy_block, new_entropy_block)
else:
    # More robust fallback if whitespace differs
    pattern = r"""    def entropy_cost\(series: pd\.Series\) -> float:[\s\S]*?feat\[\s*"pdf_code_cost_entropy_norm"\s*\][\s\S]*?\)\.fillna\(1\)"""
    txt, n = re.subn(pattern, "    " + new_entropy_block.replace("\n", "\n    ").strip(), txt, count=1)
    if n != 1:
        raise RuntimeError("Could not find the entropy block to patch. Your file differs from the expected version.")

# -------------------------------------------------------------------------
# PATCH 2: add sanitize_inf helper + call it before modeling
# -------------------------------------------------------------------------
if "def sanitize_inf" not in txt:
    helper = r'''
# =========================
# Generic safety helpers
# =========================

def sanitize_inf(df: pd.DataFrame) -> pd.DataFrame:
    """Replace +/-inf with NaN for numeric columns (sklearn imputers can't handle infinities)."""
    if df is None or df.empty:
        return df
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols):
        df.loc[:, num_cols] = df.loc[:, num_cols].replace([np.inf, -np.inf], np.nan)
    return df
'''
    txt = txt.replace('warnings.filterwarnings("ignore")', 'warnings.filterwarnings("ignore")\n' + helper)

# Call it right before building X/X_test (safe + simple insertion point)
if "df_tr = sanitize_inf(df_tr)" not in txt:
    txt = txt.replace(
        "    # Build modeling matrices\n",
        "    # Final safety: remove +/-inf values before modeling\n"
        "    df_tr = sanitize_inf(df_tr)\n"
        "    df_te = sanitize_inf(df_te)\n\n"
        "    # Build modeling matrices\n"
    )

pyfile.write_text(txt, encoding="utf-8")
print(f"✅ Patched: {pyfile.resolve()}")

# -------------------------------------------------------------------------
# Run the pipeline (still 1 cell)
# -------------------------------------------------------------------------
cmd = [sys.executable, "-m", "challenge2_multimodal_ensemble", "--data_dir", ".", "--out", "submission.csv"]
print("▶ Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("✅ Done. Wrote submission.csv")


✅ Patched: D:\AgentDs\agent_ds_healthcare\challenge2_multimodal_ensemble.py
▶ Running: c:\Users\shend\AppData\Local\Programs\Python\Python312\python.exe -m challenge2_multimodal_ensemble --data_dir . --out submission.csv


CalledProcessError: Command '['c:\\Users\\shend\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'challenge2_multimodal_ensemble', '--data_dir', '.', '--out', 'submission.csv']' returned non-zero exit status 1.

# Submission

In [29]:
# 3. Submit Predictions

# Submit predictions to the competition
print("🚀 Submitting predictions...")

try:
    result = client.submit_prediction("Healthcare", 2, "D:/AgentDs/agent_ds_healthcare/submission_ICHI_V1.csv")
    
    if result['success']:
        print("✅ Submission successful!")
        print(f"   📊 Score: {result['score']:.4f}")
        print(f"   📏 Metric: {result['metric_name']}")
        print(f"   ✔️  Validation: {'Passed' if result['validation_passed'] else 'Failed'}")
    else:
        print("❌ Submission failed!")
        print(f"   Error details: {result.get('details', {}).get('validation_errors', 'Unknown error')}")
        
except Exception as e:
    print(f"💥 Submission error: {e}")
    print("🔧 Check your API key and team name are correct!")

print("\n🎯 Next steps:")
print("   1. Try incorporating relevant information outside this table!")
print("   2. Move on to Healthcare Challenge 3!")


🚀 Submitting predictions...
✅ Prediction submitted successfully!
📊 Score: 489.2841 (MAE)
✅ Validation passed
✅ Submission successful!
   📊 Score: 489.2841
   📏 Metric: MAE
   ✔️  Validation: Passed

🎯 Next steps:
   1. Try incorporating relevant information outside this table!
   2. Move on to Healthcare Challenge 3!
